In [ ]:
# ============================================================
# PHASE 6: DATA CLEANING & TRANSFORMATION
# Meridian Retail Group Customer Churn Project
# ============================================================

# PURPOSE:
# This phase prepares the raw Meridian Retail Group datasets
# for Exploratory Data Analysis (EDA), feature engineering,
# and Machine Learning.
#
# IMPORTANT:
# The raw datasets will NOT be modified.
# We will create separate cleaned copies.
#
# CLEANING AREAS:
# 1. Missing values
# 2. Duplicate records
# 3. Invalid numerical values
# 4. Inconsistent categorical values
# 5. Incorrect data types
# 6. Invalid dates and prices
# 7. Referential integrity
# 8. Outlier investigation
# 9. Feature preparation and scaling considerations
#
# PRINCIPLE:
# Every cleaning decision should be evidence-based,
# reproducible, and defensible during the project viva.
#
# DATA FLOW:
#
# Raw CSV files
#      ↓
# Data-quality audit
#      ↓
# Cleaning decisions
#      ↓
# Cleaned datasets
#      ↓
# Validation
#      ↓
# EDA / Feature Engineering / Machine Learning

print("Phase 6: Data Cleaning & Transformation")
print("Meridian Retail Group Customer Churn Project")
print("-" * 60)
print("Objective: Prepare reliable, validated datasets for")
print("EDA, feature engineering, and machine learning.")

Phase 6: Data Cleaning & Transformation
Meridian Retail Group Customer Churn Project
------------------------------------------------------------
Objective: Prepare reliable, validated datasets for
EDA, feature engineering, and machine learning.


In [ ]:
# ============================================================
# CELL 2: IMPORT LIBRARIES
# ============================================================

# pandas:
# Used for loading, cleaning, transforming, and analysing
# tabular datasets.
import pandas as pd

# numpy:
# Used for numerical calculations and statistical operations.
import numpy as np

# os:
# Used for working with Google Drive folders and file paths.
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# ============================================================
# CELL 3: GOOGLE DRIVE AND PROJECT PATHS
# ============================================================

from google.colab import drive

# Mount Google Drive so that the project files can be accessed.
drive.mount('/content/drive')

# Main project folder.
PROJECT_PATH = "/content/drive/MyDrive/PORA Capstone"

# Folder containing the original raw CSV files.
RAW_PATH = os.path.join(PROJECT_PATH, "data", "raw")

# Folder where our cleaned datasets will eventually be saved.
PROCESSED_PATH = os.path.join(PROJECT_PATH, "data", "processed")

print("Project path:", PROJECT_PATH)
print("Raw data path:", RAW_PATH)
print("Processed data path:", PROCESSED_PATH)

print("\nRaw data folder exists:", os.path.exists(RAW_PATH))
print("Processed data folder exists:", os.path.exists(PROCESSED_PATH))

Mounted at /content/drive
Project path: /content/drive/MyDrive/PORA Capstone
Raw data path: /content/drive/MyDrive/PORA Capstone/data/raw
Processed data path: /content/drive/MyDrive/PORA Capstone/data/processed

Raw data folder exists: True
Processed data folder exists: False


In [ ]:
# ============================================================
# CELL 4: LOAD THE RAW DATASETS
# ============================================================

# We are loading the original CSV files from data/raw.
# IMPORTANT:
# We will NOT modify these raw datasets directly.
# All cleaning will be performed on separate copies.

customers = pd.read_csv(
    os.path.join(RAW_PATH, "customers.csv")
)

products = pd.read_csv(
    os.path.join(RAW_PATH, "products.csv")
)

orders = pd.read_csv(
    os.path.join(RAW_PATH, "orders.csv")
)

order_items = pd.read_csv(
    os.path.join(RAW_PATH, "order_items.csv")
)

support_tickets = pd.read_csv(
    os.path.join(RAW_PATH, "support_tickets.csv")
)


# Store the datasets in a dictionary.
# This will make it easier to perform repeated audits later.
raw_datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "support_tickets": support_tickets
}


print("RAW DATASETS LOADED SUCCESSFULLY")
print("=" * 60)

for name, df in raw_datasets.items():
    print(f"{name:18} : {df.shape[0]:,} rows × {df.shape[1]} columns")

RAW DATASETS LOADED SUCCESSFULLY
customers          : 6,090 rows × 14 columns
products           : 220 rows × 9 columns
orders             : 28,297 rows × 9 columns
order_items        : 73,505 rows × 5 columns
support_tickets    : 5,200 rows × 8 columns


In [ ]:
# ============================================================
# CELL 5: BASELINE DATA-QUALITY AUDIT
# ============================================================

# PURPOSE:
# Before cleaning any dataset, we need to understand the
# quality problems present in the raw data.
#
# This audit checks:
# 1. Missing values
# 2. Duplicate rows
# 3. Data types
# 4. Unique values in categorical columns
#
# IMPORTANT:
# We are only AUDITING at this stage.
# No data will be changed or deleted.


def quality_audit(df, dataset_name):
    """
    Perform a basic data-quality audit for one dataset.

    Parameters:
        df : pandas DataFrame
            Dataset being audited.

        dataset_name : str
            Name used to identify the dataset in the output.
    """

    print("=" * 70)
    print(f"DATA-QUALITY AUDIT: {dataset_name.upper()}")
    print("=" * 70)

    # --------------------------------------------------------
    # 1. BASIC STRUCTURE
    # --------------------------------------------------------

    print("\n1. DATASET STRUCTURE")
    print("-" * 70)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    # --------------------------------------------------------
    # 2. MISSING VALUES
    # --------------------------------------------------------

    print("\n2. MISSING VALUES")
    print("-" * 70)

    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)

    missing_summary = pd.DataFrame({
        "Missing_Count": missing,
        "Missing_Percentage": missing_pct
    })

    # Display only columns that contain missing values.
    missing_summary = missing_summary[
        missing_summary["Missing_Count"] > 0
    ]

    if len(missing_summary) > 0:
        print(missing_summary)
    else:
        print("No missing values detected.")

    # --------------------------------------------------------
    # 3. DUPLICATE ROWS
    # --------------------------------------------------------

    print("\n3. DUPLICATE ROWS")
    print("-" * 70)

    duplicate_count = df.duplicated().sum()

    print("Exact duplicate rows:", duplicate_count)

    # --------------------------------------------------------
    # 4. DATA TYPES
    # --------------------------------------------------------

    print("\n4. DATA TYPES")
    print("-" * 70)

    print(df.dtypes)

    # --------------------------------------------------------
    # 5. CATEGORICAL / OBJECT COLUMNS
    # --------------------------------------------------------

    print("\n5. CATEGORICAL COLUMN SUMMARY")
    print("-" * 70)

    categorical_columns = df.select_dtypes(
        include=["object"]
    ).columns

    for column in categorical_columns:

        unique_count = df[column].nunique(dropna=True)

        print(
            f"{column}: "
            f"{unique_count:,} unique non-null values"
        )


# ------------------------------------------------------------
# RUN THE AUDIT FOR ALL FIVE RAW DATASETS
# ------------------------------------------------------------

for dataset_name, dataframe in raw_datasets.items():

    quality_audit(
        dataframe,
        dataset_name
    )

DATA-QUALITY AUDIT: CUSTOMERS

1. DATASET STRUCTURE
----------------------------------------------------------------------
Rows: 6090
Columns: 14

2. MISSING VALUES
----------------------------------------------------------------------
                  Missing_Count  Missing_Percentage
email                       242                3.97
phone                       423                6.95
date_of_birth               181                2.97
city                        121                1.99
marketing_opt_in           1191               19.56
referral_source            1705               28.00

3. DUPLICATE ROWS
----------------------------------------------------------------------
Exact duplicate rows: 0

4. DATA TYPES
----------------------------------------------------------------------
customer_id          object
first_name           object
last_name            object
email                object
phone                object
gender               object
date_of_birth        object
sign

In [ ]:
# ============================================================
# CELL 6: CREATE WORKING COPIES FOR CLEANING
# ============================================================

# IMPORTANT:
# We never clean the original raw DataFrames directly.
#
# The raw datasets remain unchanged and will serve as our
# reference point throughout Phase 6.
#
# We create separate working copies for cleaning.

customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
support_tickets_clean = support_tickets.copy()


print("Working copies created successfully.")
print("=" * 60)

print("Customers:", customers_clean.shape)
print("Products:", products_clean.shape)
print("Orders:", orders_clean.shape)
print("Order Items:", order_items_clean.shape)
print("Support Tickets:", support_tickets_clean.shape)

print("\nRaw datasets remain unchanged.")

Working copies created successfully.
Customers: (6090, 14)
Products: (220, 9)
Orders: (28297, 9)
Order Items: (73505, 5)
Support Tickets: (5200, 8)

Raw datasets remain unchanged.


In [ ]:
# ============================================================
# CELL 7: INVESTIGATE CUSTOMER CATEGORICAL VALUES
# ============================================================

# PURPOSE:
# Before standardizing categorical variables, we need to see
# exactly what values exist in the raw data.
#
# We will inspect:
# 1. Membership tier
# 2. Marketing opt-in
# 3. Gender
# 4. Preferred channel
# 5. Referral source
#
# We will NOT modify anything in this cell.


categorical_columns = [
    "membership_tier",
    "marketing_opt_in",
    "gender",
    "preferred_channel",
    "referral_source"
]


for column in categorical_columns:

    print("\n" + "=" * 70)
    print(f"{column.upper()}")
    print("=" * 70)

    # value_counts(dropna=False) allows us to see missing values
    # alongside the actual categories.
    value_counts = customers_clean[column].value_counts(
        dropna=False
    )

    print(value_counts)


MEMBERSHIP_TIER
membership_tier
bronze      944
BRONZE      939
Bronze      902
silver      606
SILVER      596
Silver      566
gold        394
GOLD        385
Gold        347
platinum    142
PLATINUM    139
Platinum    130
Name: count, dtype: int64

MARKETING_OPT_IN
marketing_opt_in
Y      1249
N      1208
NaN    1191
0       623
no      615
yes     612
1       592
Name: count, dtype: int64

GENDER
gender
M        2530
F        2363
Other    1197
Name: count, dtype: int64

PREFERRED_CHANNEL
preferred_channel
Mobile App    2053
In-Store      2041
Online        1996
Name: count, dtype: int64

REFERRAL_SOURCE
referral_source
NaN                1705
Social Media        650
Walk-in             649
TV Ad               635
Influencer          629
Friend Referral     614
Email Campaign      604
Search Engine       604
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 8: STANDARDIZE CUSTOMER CATEGORICAL VARIABLES
# ============================================================

# PURPOSE:
# Standardize inconsistent categorical encodings while
# preserving genuine missing values.
#
# Decisions:
#
# 1. membership_tier:
#    Different capitalization represents the same tier.
#    Example: "gold", "GOLD", "Gold" -> "Gold"
#
# 2. marketing_opt_in:
#    Y, yes and 1 -> Yes
#    N, no and 0 -> No
#    Missing values remain missing because we cannot know
#    the customer's preference.
#
# 3. gender:
#    Already standardized; no change required.
#
# 4. preferred_channel:
#    Already standardized; no change required.
#
# 5. referral_source:
#    Existing categories are consistent.
#    Missing values remain missing.


# ------------------------------------------------------------
# 1. STANDARDIZE MEMBERSHIP TIER
# ------------------------------------------------------------

customers_clean["membership_tier"] = (
    customers_clean["membership_tier"]
    .astype("string")
    .str.strip()
    .str.title()
)


# ------------------------------------------------------------
# 2. STANDARDIZE MARKETING OPT-IN
# ------------------------------------------------------------

marketing_mapping = {
    "Y": "Yes",
    "yes": "Yes",
    "1": "Yes",
    "N": "No",
    "no": "No",
    "0": "No"
}

customers_clean["marketing_opt_in"] = (
    customers_clean["marketing_opt_in"]
    .map(marketing_mapping)
)


# ------------------------------------------------------------
# 3. DISPLAY THE CLEANED CATEGORIES
# ------------------------------------------------------------

print("STANDARDIZED CUSTOMER CATEGORIES")
print("=" * 70)

print("\nMembership Tier:")
print(
    customers_clean["membership_tier"]
    .value_counts(dropna=False)
)

print("\nMarketing Opt-In:")
print(
    customers_clean["marketing_opt_in"]
    .value_counts(dropna=False)
)

print("\nGender:")
print(
    customers_clean["gender"]
    .value_counts(dropna=False)
)

print("\nPreferred Channel:")
print(
    customers_clean["preferred_channel"]
    .value_counts(dropna=False)
)

print("\nReferral Source:")
print(
    customers_clean["referral_source"]
    .value_counts(dropna=False)
)

STANDARDIZED CUSTOMER CATEGORIES

Membership Tier:
membership_tier
Bronze      2785
Silver      1768
Gold        1126
Platinum     411
Name: count, dtype: Int64

Marketing Opt-In:
marketing_opt_in
Yes    2453
No     2446
NaN    1191
Name: count, dtype: int64

Gender:
gender
M        2530
F        2363
Other    1197
Name: count, dtype: int64

Preferred Channel:
preferred_channel
Mobile App    2053
In-Store      2041
Online        1996
Name: count, dtype: int64

Referral Source:
referral_source
NaN                1705
Social Media        650
Walk-in             649
TV Ad               635
Influencer          629
Friend Referral     614
Email Campaign      604
Search Engine       604
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 9: INVESTIGATE CUSTOMER STATE VALUES
# ============================================================

# PURPOSE:
# The raw customer dataset contains 80 unique state values.
#
# Since the project is based on Nigerian customers, this may
# indicate inconsistent representations such as:
#
#   Lagos
#   lagos
#   LAGOS
#
# or abbreviations / spelling variations.
#
# We will inspect the actual values before deciding how to
# standardize them.
#
# IMPORTANT:
# No values will be changed in this cell.


state_counts = (
    customers_clean["state"]
    .value_counts(dropna=False)
)

print("CUSTOMER STATE VALUES")
print("=" * 70)

print(state_counts.to_string())

print("\n" + "=" * 70)
print("TOTAL UNIQUE NON-MISSING STATES:",
      customers_clean["state"].nunique(dropna=True))

print("MISSING STATE VALUES:",
      customers_clean["state"].isna().sum())

CUSTOMER STATE VALUES
state
NY                143
Illinois          141
OH                139
Missouri          139
TX                138
Washington        135
California        135
GA                133
MA                132
Wisconsin         132
Tennessee         131
Georgia           130
Florida           127
AZ                125
Arizona           125
New York          124
Michigan          124
Maryland          124
Indiana           123
PA                123
FL                121
Pennsylvania      120
New Jersey        120
WA                119
IL                119
VA                118
Massachusetts     118
MI                116
IN                116
CA                116
MO                116
NJ                112
North Carolina    111
WI                108
TN                108
Virginia          106
Ohio              106
MD                106
Texas             106
NC                 92
az                 40
PENNSYLVANIA       40
VIRGINIA           38
va                 38
nc  

In [ ]:
# ============================================================
# CELL 10: STANDARDIZE CUSTOMER STATE VALUES
# ============================================================

# PURPOSE:
# The raw state column contains multiple representations of
# the same U.S. states.
#
# Examples:
#   NY, ny, New York, NEW YORK -> New York
#   CA, ca, California, CALIFORNIA -> California
#
# We standardize these representations to full canonical
# state names.
#
# IMPORTANT:
# We are only standardizing values that have an explicit
# mapping. We are not guessing unknown states.


state_mapping = {
    "AL": "Alabama",
    "AK": "Alaska",
    "AZ": "Arizona",
    "AR": "Arkansas",
    "CA": "California",
    "CO": "Colorado",
    "CT": "Connecticut",
    "DE": "Delaware",
    "FL": "Florida",
    "GA": "Georgia",
    "HI": "Hawaii",
    "ID": "Idaho",
    "IL": "Illinois",
    "IN": "Indiana",
    "IA": "Iowa",
    "KS": "Kansas",
    "KY": "Kentucky",
    "LA": "Louisiana",
    "ME": "Maine",
    "MD": "Maryland",
    "MA": "Massachusetts",
    "MI": "Michigan",
    "MN": "Minnesota",
    "MS": "Mississippi",
    "MO": "Missouri",
    "MT": "Montana",
    "NE": "Nebraska",
    "NV": "Nevada",
    "NH": "New Hampshire",
    "NJ": "New Jersey",
    "NM": "New Mexico",
    "NY": "New York",
    "NC": "North Carolina",
    "ND": "North Dakota",
    "OH": "Ohio",
    "OK": "Oklahoma",
    "OR": "Oregon",
    "PA": "Pennsylvania",
    "RI": "Rhode Island",
    "SC": "South Carolina",
    "SD": "South Dakota",
    "TN": "Tennessee",
    "TX": "Texas",
    "UT": "Utah",
    "VT": "Vermont",
    "VA": "Virginia",
    "WA": "Washington",
    "WV": "West Virginia",
    "WI": "Wisconsin",
    "WY": "Wyoming"
}


# ------------------------------------------------------------
# CREATE A NORMALIZED VERSION OF EACH STATE VALUE
# ------------------------------------------------------------

# Strip surrounding spaces and convert to uppercase.
# This allows us to match abbreviations and capitalization
# variants consistently.

normalized_state = (
    customers_clean["state"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# ------------------------------------------------------------
# CONVERT ABBREVIATIONS AND FULL NAMES
# ------------------------------------------------------------

# First check whether the value is a two-letter abbreviation.
# If it is, replace it using state_mapping.
#
# Otherwise, retain the original normalized value for the
# full state name and convert it to title case.

customers_clean["state"] = normalized_state.map(
    state_mapping
).fillna(
    normalized_state.str.title()
)


# ------------------------------------------------------------
# DISPLAY THE RESULT
# ------------------------------------------------------------

print("STANDARDIZED STATE VALUES")
print("=" * 70)

print(
    customers_clean["state"]
    .value_counts()
    .to_string()
)

print("\n" + "=" * 70)

print(
    "Unique states after standardization:",
    customers_clean["state"].nunique()
)

print(
    "Missing states after standardization:",
    customers_clean["state"].isna().sum()
)

STANDARDIZED STATE VALUES
state
Missouri          326
New York          325
Georgia           324
Arizona           321
California        316
Illinois          314
Pennsylvania      309
Washington        308
Massachusetts     307
Wisconsin         306
Florida           305
Tennessee         302
Virginia          300
Texas             298
Maryland          296
Michigan          294
Ohio              292
New Jersey        292
Indiana           289
North Carolina    266

Unique states after standardization: 20
Missing states after standardization: 0


In [ ]:
# ============================================================
# CELL 11: VALIDATE STATE STANDARDIZATION
# ============================================================

# PURPOSE:
# Confirm that every state value now belongs to the set of
# canonical state names represented in the dataset.
#
# This is a validation step only.
# No data will be modified.


valid_states = set(state_mapping.values())

invalid_states = sorted(
    set(customers_clean["state"].dropna()) - valid_states
)


print("STATE STANDARDIZATION VALIDATION")
print("=" * 70)

print("Unique standardized states:",
      customers_clean["state"].nunique())

print("Missing states:",
      customers_clean["state"].isna().sum())

print("Unexpected state values:",
      len(invalid_states))


if len(invalid_states) == 0:
    print("\nPASS: All customer state values are valid")
    print("canonical state names.")
else:
    print("\nCHECK: Unexpected state values found:")
    print(invalid_states)

STATE STANDARDIZATION VALIDATION
Unique standardized states: 20
Missing states: 0
Unexpected state values: 0

PASS: All customer state values are valid
canonical state names.


In [ ]:
# ============================================================
# CELL 12: INVESTIGATE CUSTOMER DATE FIELDS
# ============================================================

# PURPOSE:
# Inspect the raw date values before converting them.
#
# We are particularly interested in:
# 1. Date-of-birth formats
# 2. Signup-date formats
# 3. Missing values
# 4. Potentially unusual date values
#
# IMPORTANT:
# No dates will be changed in this cell.


date_columns = [
    "date_of_birth",
    "signup_date"
]


for column in date_columns:

    print("\n" + "=" * 70)
    print(f"DATE FIELD: {column.upper()}")
    print("=" * 70)

    print("\nData type:")
    print(customers_clean[column].dtype)

    print("\nMissing values:")
    print(customers_clean[column].isna().sum())

    print("\nSample values:")
    print(
        customers_clean[column]
        .dropna()
        .head(20)
        .to_string(index=False)
    )

    print("\nNumber of unique non-missing values:")
    print(customers_clean[column].nunique(dropna=True))


DATE FIELD: DATE_OF_BIRTH

Data type:
object

Missing values:
181

Sample values:
1957-07-03
1966-12-07
1998-06-14
1975-08-30
1999-09-15
2006-03-07
1945-11-07
1953-11-29
1985-10-25
1999-01-25
1942-04-05
1975-01-06
1997-03-05
1986-07-06
1958-04-29
1942-03-24
1971-04-16
1993-01-28
1989-04-23
1966-08-13

Number of unique non-missing values:
5152

DATE FIELD: SIGNUP_DATE

Data type:
object

Missing values:
0

Sample values:
2026-05-28
2022-12-23
2022-07-10
11/03/2022
2023-07-05
08/11/2022
2023-09-04
2021-12-07
11/12/2021
2024-09-15
2021-05-22
2025-04-01
2025-06-18
2021-04-16
2021-12-21
2022-08-27
2022-05-08
2021-12-28
2024-08-22
2022-09-19

Number of unique non-missing values:
2394


In [ ]:
# ============================================================
# CELL 13: CHECK DATE OF BIRTH VALIDITY
# ============================================================

# PURPOSE:
# Convert date_of_birth to a temporary datetime representation
# so that we can investigate whether any dates are invalid or
# suspicious.
#
# We will NOT modify the original date_of_birth column yet.
#
# We will check for:
# 1. Values that cannot be parsed as dates
# 2. Dates in the future
# 3. Dates that would imply an unusually young or old customer
#
# The age checks are only for investigation at this stage.


# ------------------------------------------------------------
# 1. TEMPORARY DATE CONVERSION
# ------------------------------------------------------------

dob_temp = pd.to_datetime(
    customers_clean["date_of_birth"],
    errors="coerce"
)


# ------------------------------------------------------------
# 2. INVALID / UNPARSED VALUES
# ------------------------------------------------------------

invalid_dob_mask = (
    customers_clean["date_of_birth"].notna()
    & dob_temp.isna()
)

invalid_dob_count = invalid_dob_mask.sum()


# ------------------------------------------------------------
# 3. DEFINE A REFERENCE DATE
# ------------------------------------------------------------

# We use the project's churn snapshot date as a consistent
# analytical reference point.
reference_date = pd.Timestamp("2026-07-01")


# ------------------------------------------------------------
# 4. FUTURE DOB VALUES
# ------------------------------------------------------------

future_dob_mask = (
    dob_temp.notna()
    & (dob_temp > reference_date)
)

future_dob_count = future_dob_mask.sum()


# ------------------------------------------------------------
# 5. CALCULATE APPROXIMATE AGE
# ------------------------------------------------------------

age_years = (
    (reference_date - dob_temp).dt.days / 365.25
)


# ------------------------------------------------------------
# 6. INVESTIGATE EXTREME AGES
# ------------------------------------------------------------

# These thresholds are deliberately used for investigation,
# not automatic deletion.
#
# Below 18:
# Potentially valid, but worth checking because this is a
# customer dataset and the business context may matter.
#
# Above 100:
# Potentially valid, but worth investigating for data errors.

under_18_mask = (
    age_years.notna()
    & (age_years < 18)
)

over_100_mask = (
    age_years.notna()
    & (age_years > 100)
)


# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("DATE OF BIRTH VALIDITY AUDIT")
print("=" * 70)

print("Missing DOB values:",
      customers_clean["date_of_birth"].isna().sum())

print("Invalid/unparsed DOB values:",
      invalid_dob_count)

print("Future DOB values:",
      future_dob_count)

print("Customers apparently under 18:",
      under_18_mask.sum())

print("Customers apparently over 100:",
      over_100_mask.sum())


# ------------------------------------------------------------
# 8. SHOW SUSPICIOUS RECORDS
# ------------------------------------------------------------

if invalid_dob_count > 0:

    print("\nInvalid/unparsed DOB examples:")
    print(
        customers_clean.loc[
            invalid_dob_mask,
            ["customer_id", "date_of_birth"]
        ].head(20).to_string(index=False)
    )


if future_dob_count > 0:

    print("\nFuture DOB examples:")
    print(
        customers_clean.loc[
            future_dob_mask,
            ["customer_id", "date_of_birth"]
        ].head(20).to_string(index=False)
    )


if under_18_mask.sum() > 0:

    print("\nUnder-18 DOB examples:")
    print(
        customers_clean.loc[
            under_18_mask,
            ["customer_id", "date_of_birth"]
        ].head(20).to_string(index=False)
    )


if over_100_mask.sum() > 0:

    print("\nOver-100 DOB examples:")
    print(
        customers_clean.loc[
            over_100_mask,
            ["customer_id", "date_of_birth"]
        ].head(20).to_string(index=False)
    )

DATE OF BIRTH VALIDITY AUDIT
Missing DOB values: 181
Invalid/unparsed DOB values: 0
Future DOB values: 7
Customers apparently under 18: 185
Customers apparently over 100: 8

Future DOB examples:
customer_id date_of_birth
 CUST002417    2031-05-04
 CUST004179    2031-05-04
 CUST004771    2031-05-04
 CUST002679    2031-05-04
 CUST000885    2031-05-04
 CUST003311    2031-05-04
 CUST005029    2031-05-04

Under-18 DOB examples:
customer_id date_of_birth
 CUST003555    2010-08-15
 CUST002483    2009-10-05
 CUST000982    2009-01-01
 CUST003025    2009-09-21
 CUST000694    2008-11-21
 CUST002417    2031-05-04
 CUST005660    2010-01-01
 CUST001523    2009-01-28
 CUST004977    2009-03-14
 CUST003286    2008-08-17
 CUST000995    2008-12-05
 CUST003655    2009-08-13
 CUST000577    2009-07-02
 CUST000296    2010-06-30
 CUST000003    2008-10-16
 CUST000155    2010-07-09
 CUST001875    2008-07-14
 CUST001266    2010-07-23
 CUST005657    2009-07-08
 CUST004381    2008-07-19

Over-100 DOB examples:
cus

In [ ]:
# ============================================================
# CELL 14: INVESTIGATE SUSPICIOUS DATE-OF-BIRTH RECORDS
# ============================================================

# PURPOSE:
# Before changing any DOB values, we investigate the records
# identified in Cell 13.
#
# We will specifically examine:
# 1. Future DOBs
# 2. Under-18 customers
# 3. Over-100 customers
#
# The goal is to understand whether these are genuine values,
# obvious data-entry errors, or placeholder values.
#
# IMPORTANT:
# We are investigating only.
# We are NOT modifying the dataset in this cell.


# ------------------------------------------------------------
# 1. CREATE AN INVESTIGATION TABLE
# ------------------------------------------------------------

dob_investigation = customers_clean[
    [
        "customer_id",
        "date_of_birth",
        "signup_date",
        "membership_tier",
        "state",
        "city"
    ]
].copy()


# ------------------------------------------------------------
# 2. ADD THE TEMPORARY AGE CALCULATION
# ------------------------------------------------------------

dob_investigation["dob_temp"] = pd.to_datetime(
    dob_investigation["date_of_birth"],
    errors="coerce"
)

dob_investigation["age_at_snapshot"] = (
    (reference_date - dob_investigation["dob_temp"]).dt.days / 365.25
)


# ------------------------------------------------------------
# 3. INVESTIGATE FUTURE DOBs
# ------------------------------------------------------------

future_records = dob_investigation[
    dob_investigation["dob_temp"] > reference_date
].copy()

print("FUTURE DOB RECORDS")
print("=" * 70)

print("Number of future DOB records:",
      len(future_records))

if len(future_records) > 0:
    print(
        future_records[
            [
                "customer_id",
                "date_of_birth",
                "signup_date",
                "membership_tier",
                "state"
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# 4. INVESTIGATE UNDER-18 CUSTOMERS
# ------------------------------------------------------------

under_18_records = dob_investigation[
    (dob_investigation["age_at_snapshot"] < 18)
    & (dob_investigation["age_at_snapshot"] >= 0)
].copy()

print("\n\nUNDER-18 CUSTOMER RECORDS")
print("=" * 70)

print("Number of customers apparently under 18:",
      len(under_18_records))

print("\nAge distribution:")
print(
    under_18_records["age_at_snapshot"]
    .round(1)
    .describe()
)


# Show a sample rather than every record.
print("\nSample under-18 records:")

print(
    under_18_records[
        [
            "customer_id",
            "date_of_birth",
            "age_at_snapshot",
            "signup_date",
            "membership_tier",
            "state"
        ]
    ]
    .sort_values("age_at_snapshot")
    .head(30)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. INVESTIGATE OVER-100 CUSTOMERS
# ------------------------------------------------------------

over_100_records = dob_investigation[
    dob_investigation["age_at_snapshot"] > 100
].copy()

print("\n\nOVER-100 CUSTOMER RECORDS")
print("=" * 70)

print("Number of customers apparently over 100:",
      len(over_100_records))

if len(over_100_records) > 0:

    print(
        over_100_records[
            [
                "customer_id",
                "date_of_birth",
                "age_at_snapshot",
                "signup_date",
                "membership_tier",
                "state"
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# 6. CHECK WHETHER 1899-01-01 IS A REPEATED VALUE
# ------------------------------------------------------------

print("\n\nREPEATED SUSPICIOUS DOB VALUES")
print("=" * 70)

suspicious_dob_counts = (
    dob_investigation["date_of_birth"]
    .value_counts(dropna=False)
    .loc[lambda x: x.index.astype(str).str.contains("1899-01-01", na=False)]
)

print(suspicious_dob_counts)

FUTURE DOB RECORDS
Number of future DOB records: 7
customer_id date_of_birth signup_date membership_tier         state
 CUST002417    2031-05-04  2021-11-18          Silver         Texas
 CUST004179    2031-05-04  2024-12-29          Bronze      Missouri
 CUST004771    2031-05-04  2025-08-22          Bronze      Virginia
 CUST002679    2031-05-04  2022-04-22          Bronze       Arizona
 CUST000885    2031-05-04  2026-04-14          Bronze Massachusetts
 CUST003311    2031-05-04  2024-02-16          Bronze          Ohio
 CUST005029    2031-05-04  2026-04-22          Silver    Washington


UNDER-18 CUSTOMER RECORDS
Number of customers apparently under 18: 178

Age distribution:
count    178.000000
mean      16.982022
std        0.618009
min       15.900000
25%       16.500000
50%       17.000000
75%       17.600000
max       18.000000
Name: age_at_snapshot, dtype: float64

Sample under-18 records:
customer_id date_of_birth  age_at_snapshot signup_date membership_tier          state
 CU

In [ ]:
# ============================================================
# CELL 15: CLEAN DATE OF BIRTH
# ============================================================

# PURPOSE:
# Apply the DOB cleaning decisions made after our investigation.
#
# Cleaning rules:
# 1. Convert DOB values to proper datetime format.
# 2. Future DOBs are invalid -> set to missing.
# 3. The repeated 1899-01-01 value is treated as an invalid
#    placeholder -> set to missing.
# 4. Plausible under-18 DOBs are retained.
# 5. Existing missing DOBs remain missing.
#
# IMPORTANT:
# We are modifying customers_clean, NOT the raw customers data.


# ------------------------------------------------------------
# 1. CONVERT DOB TO DATETIME
# ------------------------------------------------------------

customers_clean["date_of_birth"] = pd.to_datetime(
    customers_clean["date_of_birth"],
    errors="coerce"
)


# ------------------------------------------------------------
# 2. IDENTIFY CLEARLY INVALID DOB VALUES
# ------------------------------------------------------------

invalid_future_dob = (
    customers_clean["date_of_birth"] > reference_date
)

invalid_placeholder_dob = (
    customers_clean["date_of_birth"] == pd.Timestamp("1899-01-01")
)


# ------------------------------------------------------------
# 3. COUNT INVALID VALUES BEFORE REPLACEMENT
# ------------------------------------------------------------

future_count_before = invalid_future_dob.sum()
placeholder_count_before = invalid_placeholder_dob.sum()


# ------------------------------------------------------------
# 4. REPLACE INVALID DOB VALUES WITH MISSING
# ------------------------------------------------------------

customers_clean.loc[
    invalid_future_dob | invalid_placeholder_dob,
    "date_of_birth"
] = pd.NaT


# ------------------------------------------------------------
# 5. CHECK THE RESULT
# ------------------------------------------------------------

print("DATE OF BIRTH CLEANING RESULT")
print("=" * 70)

print("Future DOB values replaced with missing:",
      future_count_before)

print("1899-01-01 placeholder values replaced with missing:",
      placeholder_count_before)

print("Total missing DOB values after cleaning:",
      customers_clean["date_of_birth"].isna().sum())

print("Valid DOB values remaining:",
      customers_clean["date_of_birth"].notna().sum())

print("DOB data type:",
      customers_clean["date_of_birth"].dtype)

DATE OF BIRTH CLEANING RESULT
Future DOB values replaced with missing: 7
1899-01-01 placeholder values replaced with missing: 8
Total missing DOB values after cleaning: 196
Valid DOB values remaining: 5894
DOB data type: datetime64[ns]


In [ ]:
# ============================================================
# CELL 16: AUDIT CUSTOMER SIGNUP DATE
# ============================================================

# PURPOSE:
# Investigate the signup_date column before final cleaning.
#
# We already know that the raw data contains mixed date formats.
# Here we will:
#
# 1. Check missing values
# 2. Identify the different date-format patterns
# 3. Safely parse both YYYY-MM-DD and MM/DD/YYYY formats
# 4. Check for dates that cannot be parsed
# 5. Check for signup dates in the future
#
# IMPORTANT:
# We are investigating first.
# We will modify the actual column in the next cell.


# ------------------------------------------------------------
# 1. CHECK BASIC INFORMATION
# ------------------------------------------------------------

print("SIGNUP DATE AUDIT")
print("=" * 70)

print("Data type before cleaning:",
      customers_clean["signup_date"].dtype)

print("Missing signup dates:",
      customers_clean["signup_date"].isna().sum())

print("Unique signup date values:",
      customers_clean["signup_date"].nunique())


# ------------------------------------------------------------
# 2. DEFINE A SAFE MIXED-DATE PARSER
# ------------------------------------------------------------

def parse_signup_date(value):
    """
    Safely parse the two expected signup-date formats:

    YYYY-MM-DD
    MM/DD/YYYY

    Invalid or missing values return NaT.
    """

    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    if value == "":
        return pd.NaT

    try:

        # Format 1: YYYY-MM-DD
        if "-" in value:
            return pd.to_datetime(
                value,
                format="%Y-%m-%d"
            )

        # Format 2: MM/DD/YYYY
        elif "/" in value:
            return pd.to_datetime(
                value,
                format="%m/%d/%Y"
            )

        # Any other format is treated as invalid
        else:
            return pd.NaT

    except (ValueError, TypeError):
        return pd.NaT


# ------------------------------------------------------------
# 3. APPLY THE SAFE PARSER TEMPORARILY
# ------------------------------------------------------------

signup_date_parsed = (
    customers_clean["signup_date"]
    .apply(parse_signup_date)
)


# ------------------------------------------------------------
# 4. CHECK UNPARSED VALUES
# ------------------------------------------------------------

unparsed_signup_mask = (
    customers_clean["signup_date"].notna()
    & signup_date_parsed.isna()
)

unparsed_signup_count = unparsed_signup_mask.sum()


# ------------------------------------------------------------
# 5. CHECK FUTURE SIGNUP DATES
# ------------------------------------------------------------

future_signup_mask = (
    signup_date_parsed > reference_date
)

future_signup_count = future_signup_mask.sum()


# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nParsed signup dates:",
      signup_date_parsed.notna().sum())

print("Unparsed signup dates:",
      unparsed_signup_count)

print("Future signup dates:",
      future_signup_count)


# ------------------------------------------------------------
# 7. SHOW UNPARSED VALUES IF ANY
# ------------------------------------------------------------

if unparsed_signup_count > 0:

    print("\nUnparsed signup date examples:")

    print(
        customers_clean.loc[
            unparsed_signup_mask,
            ["customer_id", "signup_date"]
        ]
        .head(20)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 8. SHOW FUTURE SIGNUP DATES IF ANY
# ------------------------------------------------------------

if future_signup_count > 0:

    print("\nFuture signup date examples:")

    print(
        customers_clean.loc[
            future_signup_mask,
            ["customer_id", "signup_date"]
        ]
        .head(20)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 9. DISPLAY EXAMPLES OF SUCCESSFUL CONVERSION
# ------------------------------------------------------------

print("\nExamples of parsed signup dates:")

signup_examples = pd.DataFrame({
    "original": customers_clean["signup_date"],
    "parsed": signup_date_parsed
})

print(
    signup_examples
    .drop_duplicates()
    .head(15)
    .to_string(index=False)
)

SIGNUP DATE AUDIT
Data type before cleaning: object
Missing signup dates: 0
Unique signup date values: 2394

Parsed signup dates: 6090
Unparsed signup dates: 0
Future signup dates: 0

Examples of parsed signup dates:
  original     parsed
2026-05-28 2026-05-28
2022-12-23 2022-12-23
2022-07-10 2022-07-10
11/03/2022 2022-11-03
2023-07-05 2023-07-05
08/11/2022 2022-08-11
2023-09-04 2023-09-04
2021-12-07 2021-12-07
11/12/2021 2021-11-12
2024-09-15 2024-09-15
2021-05-22 2021-05-22
2025-04-01 2025-04-01
2025-06-18 2025-06-18
2021-04-16 2021-04-16
2021-12-21 2021-12-21


In [ ]:
# ============================================================
# CELL 17: STANDARDIZE CUSTOMER SIGNUP DATE
# ============================================================

# PURPOSE:
# Apply the validated mixed-format parser to the actual
# signup_date column.
#
# After this step:
# - YYYY-MM-DD dates will be standardized
# - MM/DD/YYYY dates will be standardized
# - The column will use a proper datetime data type
# - No valid customer records will be removed


# ------------------------------------------------------------
# 1. APPLY THE VALIDATED PARSER
# ------------------------------------------------------------

customers_clean["signup_date"] = (
    customers_clean["signup_date"]
    .apply(parse_signup_date)
)


# ------------------------------------------------------------
# 2. VALIDATE THE CLEANED COLUMN
# ------------------------------------------------------------

print("SIGNUP DATE CLEANING RESULT")
print("=" * 70)

print("Data type after cleaning:",
      customers_clean["signup_date"].dtype)

print("Missing signup dates:",
      customers_clean["signup_date"].isna().sum())

print("Valid signup dates:",
      customers_clean["signup_date"].notna().sum())

print("Future signup dates:",
      (
          customers_clean["signup_date"] > reference_date
      ).sum())


# ------------------------------------------------------------
# 3. DISPLAY A FEW CLEANED VALUES
# ------------------------------------------------------------

print("\nSample standardized signup dates:")

print(
    customers_clean[
        ["customer_id", "signup_date"]
    ]
    .head(15)
    .to_string(index=False)
)

SIGNUP DATE CLEANING RESULT
Data type after cleaning: datetime64[ns]
Missing signup dates: 0
Valid signup dates: 6090
Future signup dates: 0

Sample standardized signup dates:
customer_id signup_date
 CUST002565  2026-05-28
 CUST004275  2022-12-23
 CUST003045  2022-07-10
 CUST005168  2022-11-03
 CUST003986  2023-07-05
 CUST004228  2022-08-11
 CUST002495  2023-09-04
 CUST004379  2021-12-07
 CUST004964  2021-11-12
 CUST004930  2024-09-15
 CUST000542  2021-05-22
 CUST003259  2025-04-01
 CUST000335  2025-06-18
 CUST004110  2021-04-16
 CUST005839  2021-12-21


In [ ]:
# ============================================================
# CELL 18: AUDIT CUSTOMER CONTACT INFORMATION
# ============================================================

# PURPOSE:
# Investigate the quality of customer email and phone fields
# before deciding whether any standardization is necessary.
#
# We will check:
# 1. Missing email and phone values
# 2. Duplicate email addresses
# 3. Duplicate phone numbers
# 4. Leading/trailing whitespace
# 5. Basic email-format problems
#
# IMPORTANT:
# This is an investigation cell.
# We are NOT deleting customer records here.


# ------------------------------------------------------------
# 1. CREATE TEMPORARY CLEAN VERSIONS
# ------------------------------------------------------------

email_temp = (
    customers_clean["email"]
    .astype("string")
    .str.strip()
)

phone_temp = (
    customers_clean["phone"]
    .astype("string")
    .str.strip()
)


# ------------------------------------------------------------
# 2. MISSING VALUES
# ------------------------------------------------------------

missing_email = email_temp.isna().sum()
missing_phone = phone_temp.isna().sum()


# ------------------------------------------------------------
# 3. DUPLICATE EMAIL ADDRESSES
# ------------------------------------------------------------

duplicate_email_mask = (
    email_temp.notna()
    & email_temp.duplicated(keep=False)
)

duplicate_email_count = duplicate_email_mask.sum()

duplicate_email_values = (
    email_temp[duplicate_email_mask]
    .value_counts()
)


# ------------------------------------------------------------
# 4. DUPLICATE PHONE NUMBERS
# ------------------------------------------------------------

duplicate_phone_mask = (
    phone_temp.notna()
    & phone_temp.duplicated(keep=False)
)

duplicate_phone_count = duplicate_phone_mask.sum()

duplicate_phone_values = (
    phone_temp[duplicate_phone_mask]
    .value_counts()
)


# ------------------------------------------------------------
# 5. CHECK LEADING/TRAILING WHITESPACE
# ------------------------------------------------------------

email_whitespace_count = (
    customers_clean["email"].notna()
    & (
        customers_clean["email"].astype(str)
        != customers_clean["email"].astype(str).str.strip()
    )
).sum()

phone_whitespace_count = (
    customers_clean["phone"].notna()
    & (
        customers_clean["phone"].astype(str)
        != customers_clean["phone"].astype(str).str.strip()
    )
).sum()


# ------------------------------------------------------------
# 6. BASIC EMAIL FORMAT CHECK
# ------------------------------------------------------------

# This is NOT intended to be a complete email validator.
# It simply identifies values that do not contain the basic
# structure of an email address.

email_format_mask = (
    email_temp.notna()
    & ~email_temp.str.contains(
        r"^[^@\s]+@[^@\s]+\.[^@\s]+$",
        regex=True,
        na=False
    )
)

invalid_email_format_count = email_format_mask.sum()


# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("CUSTOMER CONTACT INFORMATION AUDIT")
print("=" * 70)

print("Missing email values:",
      missing_email)

print("Missing phone values:",
      missing_phone)

print("Customer records involved in duplicate email values:",
      duplicate_email_count)

print("Customer records involved in duplicate phone values:",
      duplicate_phone_count)

print("Email values with leading/trailing whitespace:",
      email_whitespace_count)

print("Phone values with leading/trailing whitespace:",
      phone_whitespace_count)

print("Email values failing basic format check:",
      invalid_email_format_count)


# ------------------------------------------------------------
# 8. SHOW DUPLICATE EMAIL EXAMPLES
# ------------------------------------------------------------

if duplicate_email_count > 0:

    print("\nDuplicate email examples:")

    print(
        customers_clean.loc[
            duplicate_email_mask,
            ["customer_id", "email", "first_name", "last_name"]
        ]
        .sort_values("email")
        .head(20)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 9. SHOW DUPLICATE PHONE EXAMPLES
# ------------------------------------------------------------

if duplicate_phone_count > 0:

    print("\nDuplicate phone examples:")

    print(
        customers_clean.loc[
            duplicate_phone_mask,
            ["customer_id", "phone", "first_name", "last_name"]
        ]
        .sort_values("phone")
        .head(20)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 10. SHOW INVALID EMAIL EXAMPLES
# ------------------------------------------------------------

if invalid_email_format_count > 0:

    print("\nInvalid email-format examples:")

    print(
        customers_clean.loc[
            email_format_mask,
            ["customer_id", "email"]
        ]
        .head(20)
        .to_string(index=False)
    )

CUSTOMER CONTACT INFORMATION AUDIT
Missing email values: 242
Missing phone values: 423
Customer records involved in duplicate email values: 0
Customer records involved in duplicate phone values: 174
Email values with leading/trailing whitespace: 0
Phone values with leading/trailing whitespace: 0
Email values failing basic format check: 0

Duplicate phone examples:
customer_id               phone first_name last_name
 CUST004178 (334)485-2852x95803       Paul     Short
 CUST006043 (334)485-2852x95803       Paul     Short
 CUST006038 (371)662-9968x17548      Jamie    Chavez
 CUST003856 (371)662-9968x17548      Jamie    Chavez
 CUST006070 (403)232-7399x74159      James    Carter
 CUST001707 (403)232-7399x74159      James    Carter
 CUST006006  (431)587-9897x6330     Nathan    Fisher
 CUST002184  (431)587-9897x6330     Nathan    Fisher
 CUST006020   (433)804-3494x815      Maria   Houston
 CUST003056   (433)804-3494x815      Maria   Houston
 CUST006030       (492)432-4687     Robert   Ferre

In [ ]:
# ============================================================
# CELL 19: INVESTIGATE POSSIBLE NEAR-DUPLICATE CUSTOMERS
# ============================================================

# PURPOSE:
# The audit found 174 customer records sharing phone numbers.
#
# However, a shared phone number does not automatically mean
# that two customer records are duplicates.
#
# We will investigate combinations of:
# - Phone number
# - First name
# - Last name
# - Email
# - Signup date
# - Customer ID
#
# This will help us distinguish:
# 1. Likely duplicate customer records
# 2. Legitimate shared contact information
#
# IMPORTANT:
# No records will be deleted in this cell.


# ------------------------------------------------------------
# 1. CREATE TEMPORARY STANDARDIZED CONTACT FIELDS
# ------------------------------------------------------------

investigation = customers_clean[
    [
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone",
        "signup_date",
        "membership_tier",
        "state",
        "city"
    ]
].copy()


investigation["phone_temp"] = (
    investigation["phone"]
    .astype("string")
    .str.strip()
)

investigation["first_name_temp"] = (
    investigation["first_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

investigation["last_name_temp"] = (
    investigation["last_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# 2. IDENTIFY RECORDS SHARING BOTH PHONE AND NAME
# ------------------------------------------------------------

same_phone_name_mask = (
    investigation["phone_temp"].notna()
    & investigation["first_name_temp"].notna()
    & investigation["last_name_temp"].notna()
)

phone_name_groups = (
    investigation[same_phone_name_mask]
    .groupby(
        [
            "phone_temp",
            "first_name_temp",
            "last_name_temp"
        ]
    )
    .size()
    .reset_index(name="record_count")
)

duplicate_phone_name_groups = phone_name_groups[
    phone_name_groups["record_count"] > 1
]


# ------------------------------------------------------------
# 3. COUNT RECORDS INVOLVED
# ------------------------------------------------------------

duplicate_phone_name_records = (
    investigation.merge(
        duplicate_phone_name_groups[
            [
                "phone_temp",
                "first_name_temp",
                "last_name_temp"
            ]
        ],
        on=[
            "phone_temp",
            "first_name_temp",
            "last_name_temp"
        ],
        how="inner"
    )
)


# ------------------------------------------------------------
# 4. DISPLAY RESULTS
# ------------------------------------------------------------

print("NEAR-DUPLICATE CUSTOMER INVESTIGATION")
print("=" * 70)

print(
    "Duplicate phone + name groups:",
    len(duplicate_phone_name_groups)
)

print(
    "Customer records involved in phone + name duplication:",
    len(duplicate_phone_name_records)
)


# ------------------------------------------------------------
# 5. SHOW EXAMPLES
# ------------------------------------------------------------

if len(duplicate_phone_name_records) > 0:

    print("\nExamples of possible near-duplicate customers:")

    print(
        duplicate_phone_name_records[
            [
                "customer_id",
                "first_name",
                "last_name",
                "email",
                "phone",
                "signup_date",
                "membership_tier",
                "state",
                "city"
            ]
        ]
        .sort_values(
            [
                "phone",
                "last_name",
                "first_name"
            ]
        )
        .head(30)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 6. CHECK WHETHER DUPLICATE PAIRS HAVE DIFFERENT EMAILS
# ------------------------------------------------------------

email_variation = (
    duplicate_phone_name_records
    .groupby(
        [
            "phone_temp",
            "first_name_temp",
            "last_name_temp"
        ]
    )["email"]
    .nunique(dropna=True)
)

groups_with_multiple_emails = (
    email_variation > 1
).sum()

print(
    "\nPhone + name groups with multiple email addresses:",
    groups_with_multiple_emails
)


# ------------------------------------------------------------
# 7. CHECK WHETHER DUPLICATE PAIRS HAVE DIFFERENT SIGNUP DATES
# ------------------------------------------------------------

signup_variation = (
    duplicate_phone_name_records
    .groupby(
        [
            "phone_temp",
            "first_name_temp",
            "last_name_temp"
        ]
    )["signup_date"]
    .nunique(dropna=True)
)

groups_with_multiple_signup_dates = (
    signup_variation > 1
).sum()

print(
    "Phone + name groups with multiple signup dates:",
    groups_with_multiple_signup_dates
)

NEAR-DUPLICATE CUSTOMER INVESTIGATION
Duplicate phone + name groups: 87
Customer records involved in phone + name duplication: 174

Examples of possible near-duplicate customers:
customer_id first_name last_name                         email               phone signup_date membership_tier          state              city
 CUST004178       Paul     Short     paul.short672@outlook.com (334)485-2852x95803  2021-03-27          Silver        Florida       South Debra
 CUST006043       Paul     Short     PAUL.SHORT672@OUTLOOK.COM (334)485-2852x95803  2021-03-27          Silver        Florida       South Debra
 CUST006038      Jamie    Chavez   JAMIE.CHAVEZ422@OUTLOOK.COM (371)662-9968x17548  2023-08-18          Bronze       Illinois         East Lori
 CUST003856      Jamie    Chavez   jamie.chavez422@outlook.com (371)662-9968x17548  2023-08-18          Bronze       Illinois         East Lori
 CUST001707      James    Carter     james.carter163@yahoo.com (403)232-7399x74159  2022-04-23       

In [ ]:
# ============================================================
# CELL 20: CHECK NEAR-DUPLICATE CUSTOMERS AGAINST TRANSACTIONS
# ============================================================

# PURPOSE:
# Before removing near-duplicate customer records, we need to
# determine whether their customer IDs are referenced by:
#
# 1. Orders
# 2. Support tickets
#
# This is important because deleting a customer record without
# considering related transactions could cause loss of useful
# business history.
#
# We will NOT delete anything in this cell.


# ------------------------------------------------------------
# 1. IDENTIFY THE NEAR-DUPLICATE CUSTOMER IDS
# ------------------------------------------------------------

duplicate_customer_ids = (
    duplicate_phone_name_records["customer_id"]
    .drop_duplicates()
    .tolist()
)


# ------------------------------------------------------------
# 2. CHECK ORDERS ASSOCIATED WITH THESE CUSTOMERS
# ------------------------------------------------------------

duplicate_customer_orders = orders[
    orders["customer_id"].isin(duplicate_customer_ids)
].copy()


# ------------------------------------------------------------
# 3. CHECK SUPPORT TICKETS ASSOCIATED WITH THESE CUSTOMERS
# ------------------------------------------------------------

duplicate_customer_tickets = support_tickets[
    support_tickets["customer_id"].isin(duplicate_customer_ids)
].copy()


# ------------------------------------------------------------
# 4. COUNT RECORDS
# ------------------------------------------------------------

print("NEAR-DUPLICATE TRANSACTION IMPACT AUDIT")
print("=" * 70)

print(
    "Near-duplicate customer records:",
    len(duplicate_customer_ids)
)

print(
    "Orders linked to near-duplicate customers:",
    len(duplicate_customer_orders)
)

print(
    "Support tickets linked to near-duplicate customers:",
    len(duplicate_customer_tickets)
)


# ------------------------------------------------------------
# 5. COUNT DISTINCT CUSTOMERS WITH ORDERS
# ------------------------------------------------------------

customers_with_orders = (
    duplicate_customer_orders["customer_id"]
    .nunique()
)

customers_with_tickets = (
    duplicate_customer_tickets["customer_id"]
    .nunique()
)

print(
    "Near-duplicate customer IDs with orders:",
    customers_with_orders
)

print(
    "Near-duplicate customer IDs with support tickets:",
    customers_with_tickets
)


# ------------------------------------------------------------
# 6. SHOW ORDER COUNTS BY CUSTOMER
# ------------------------------------------------------------

order_counts_by_duplicate_customer = (
    duplicate_customer_orders
    .groupby("customer_id")
    .size()
    .reset_index(name="order_count")
    .sort_values("order_count", ascending=False)
)

print("\nOrder counts for near-duplicate customers:")

print(
    order_counts_by_duplicate_customer
    .head(30)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 7. SHOW SUPPORT TICKET COUNTS BY CUSTOMER
# ------------------------------------------------------------

ticket_counts_by_duplicate_customer = (
    duplicate_customer_tickets
    .groupby("customer_id")
    .size()
    .reset_index(name="ticket_count")
    .sort_values("ticket_count", ascending=False)
)

print("\nSupport-ticket counts for near-duplicate customers:")

print(
    ticket_counts_by_duplicate_customer
    .head(30)
    .to_string(index=False)
)

NEAR-DUPLICATE TRANSACTION IMPACT AUDIT
Near-duplicate customer records: 174
Orders linked to near-duplicate customers: 777
Support tickets linked to near-duplicate customers: 162
Near-duplicate customer IDs with orders: 165
Near-duplicate customer IDs with support tickets: 113

Order counts for near-duplicate customers:
customer_id  order_count
 CUST000122           11
 CUST005710           10
 CUST006081           10
 CUST004892           10
 CUST002388            9
 CUST006051            9
 CUST005202            8
 CUST006061            8
 CUST006009            8
 CUST003074            8
 CUST006073            8
 CUST006090            8
 CUST006047            8
 CUST002570            7
 CUST003846            7
 CUST004015            7
 CUST006001            7
 CUST006028            7
 CUST006087            7
 CUST006063            7
 CUST006029            7
 CUST003179            7
 CUST000802            7
 CUST000890            7
 CUST006075            7
 CUST006036            7
 C

In [ ]:
# ============================================================
# CELL 21: STANDARDIZE CUSTOMER CONTACT FIELDS
# ============================================================

# PURPOSE:
# Apply safe formatting corrections to customer contact fields.
#
# Email:
# - Remove leading/trailing whitespace
# - Convert to lowercase
#
# Phone:
# - Remove leading/trailing whitespace only
# - Preserve the original phone/extension information
#
# Missing values will remain missing.
#
# We will NOT attempt to invent missing contact information.


# ------------------------------------------------------------
# 1. STANDARDIZE EMAIL
# ------------------------------------------------------------

customers_clean["email"] = (
    customers_clean["email"]
    .astype("string")
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# 2. STANDARDIZE PHONE
# ------------------------------------------------------------

customers_clean["phone"] = (
    customers_clean["phone"]
    .astype("string")
    .str.strip()
)


# ------------------------------------------------------------
# 3. VALIDATE THE RESULT
# ------------------------------------------------------------

print("CUSTOMER CONTACT CLEANING RESULT")
print("=" * 70)

print("Missing email values:",
      customers_clean["email"].isna().sum())

print("Missing phone values:",
      customers_clean["phone"].isna().sum())

print("Unique non-missing email values:",
      customers_clean["email"].nunique())

print("Unique non-missing phone values:",
      customers_clean["phone"].nunique())


# ------------------------------------------------------------
# 4. DISPLAY SAMPLE EMAILS
# ------------------------------------------------------------

print("\nSample standardized email values:")

print(
    customers_clean[
        ["customer_id", "email"]
    ]
    .dropna(subset=["email"])
    .head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. DISPLAY SAMPLE PHONES
# ------------------------------------------------------------

print("\nSample standardized phone values:")

print(
    customers_clean[
        ["customer_id", "phone"]
    ]
    .dropna(subset=["phone"])
    .head(10)
    .to_string(index=False)
)

CUSTOMER CONTACT CLEANING RESULT
Missing email values: 242
Missing phone values: 423
Unique non-missing email values: 5760
Unique non-missing phone values: 5580

Sample standardized email values:
customer_id                         email
 CUST002565     karen.castro168@gmail.com
 CUST004275    tonya.webster780@gmail.com
 CUST003045    alexis.thomas522@gmail.com
 CUST005168 danielle.goodman202@gmail.com
 CUST003986    daniel.bowman295@yahoo.com
 CUST002495  david.holland932@outlook.com
 CUST004379 kimberly.bentley747@gmail.com
 CUST004964    zachary.brown622@gmail.com
 CUST004930    courtney.duke523@yahoo.com
 CUST000542  bethany.miles195@hotmail.com

Sample standardized phone values:
customer_id               phone
 CUST002565  784-499-9916x95359
 CUST004275          3985078721
 CUST003045          5624741570
 CUST005168  505.988.6915x03091
 CUST003986 (988)305-0745x11524
 CUST002495        562.945.8846
 CUST004379        888.928.5578
 CUST004964       (829)259-3564
 CUST004930       (

In [ ]:
# ============================================================
# CELL 22: FINAL CUSTOMER CATEGORICAL AUDIT
# ============================================================

# PURPOSE:
# Perform a final quality check on the customer categorical
# fields after our earlier standardization.
#
# Fields checked:
# - first_name
# - last_name
# - gender
# - membership_tier
# - preferred_channel
# - marketing_opt_in
# - referral_source
# - state
#
# We will also check whether any required name fields are missing.


# ------------------------------------------------------------
# 1. CHECK REQUIRED NAME FIELDS
# ------------------------------------------------------------

missing_first_name = (
    customers_clean["first_name"]
    .isna()
    .sum()
)

missing_last_name = (
    customers_clean["last_name"]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# 2. DISPLAY BASIC RESULTS
# ------------------------------------------------------------

print("FINAL CUSTOMER CATEGORICAL AUDIT")
print("=" * 70)

print("Missing first names:",
      missing_first_name)

print("Missing last names:",
      missing_last_name)


# ------------------------------------------------------------
# 3. GENDER VALUES
# ------------------------------------------------------------

print("\nGender values:")
print(
    customers_clean["gender"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 4. MEMBERSHIP-TIER VALUES
# ------------------------------------------------------------

print("\nMembership-tier values:")
print(
    customers_clean["membership_tier"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 5. PREFERRED-CHANNEL VALUES
# ------------------------------------------------------------

print("\nPreferred-channel values:")
print(
    customers_clean["preferred_channel"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 6. MARKETING OPT-IN VALUES
# ------------------------------------------------------------

print("\nMarketing-opt-in values:")
print(
    customers_clean["marketing_opt_in"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 7. REFERRAL-SOURCE VALUES
# ------------------------------------------------------------

print("\nReferral-source values:")
print(
    customers_clean["referral_source"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 8. STATE VALUES
# ------------------------------------------------------------

print("\nStandardized state values:")
print(
    customers_clean["state"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 9. CHECK EXPECTED CATEGORICAL VALUES
# ------------------------------------------------------------

expected_gender = {
    "F",
    "M",
    "Other"
}

expected_membership = {
    "Bronze",
    "Silver",
    "Gold",
    "Platinum"
}

expected_channel = {
    "Online",
    "Mobile App",
    "In-Store"
}

expected_marketing = {
    "Yes",
    "No"
}


actual_gender = set(
    customers_clean["gender"]
    .dropna()
    .unique()
)

actual_membership = set(
    customers_clean["membership_tier"]
    .dropna()
    .unique()
)

actual_channel = set(
    customers_clean["preferred_channel"]
    .dropna()
    .unique()
)

actual_marketing = set(
    customers_clean["marketing_opt_in"]
    .dropna()
    .unique()
)


print("\nCATEGORY VALIDATION")
print("=" * 70)

print(
    "Gender unexpected values:",
    actual_gender - expected_gender
)

print(
    "Membership-tier unexpected values:",
    actual_membership - expected_membership
)

print(
    "Preferred-channel unexpected values:",
    actual_channel - expected_channel
)

print(
    "Marketing-opt-in unexpected values:",
    actual_marketing - expected_marketing
)

FINAL CUSTOMER CATEGORICAL AUDIT
Missing first names: 0
Missing last names: 0

Gender values:
gender
M        2530
F        2363
Other    1197
Name: count, dtype: int64

Membership-tier values:
membership_tier
Bronze      2785
Silver      1768
Gold        1126
Platinum     411
Name: count, dtype: Int64

Preferred-channel values:
preferred_channel
Mobile App    2053
In-Store      2041
Online        1996
Name: count, dtype: int64

Marketing-opt-in values:
marketing_opt_in
Yes    2453
No     2446
NaN    1191
Name: count, dtype: int64

Referral-source values:
referral_source
NaN                1705
Social Media        650
Walk-in             649
TV Ad               635
Influencer          629
Friend Referral     614
Email Campaign      604
Search Engine       604
Name: count, dtype: int64

Standardized state values:
state
Missouri          326
New York          325
Georgia           324
Arizona           321
California        316
Illinois          314
Pennsylvania      309
Washington      

In [ ]:
# ============================================================
# CELL 23: PRODUCTS - INITIAL DATA-QUALITY AUDIT
# ============================================================

print("PRODUCTS - INITIAL DATA-QUALITY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. BASIC INFORMATION
# ------------------------------------------------------------

print("Shape:", products_clean.shape)

print("\nData types:")
print(products_clean.dtypes)

# ------------------------------------------------------------
# 2. MISSING VALUES
# ------------------------------------------------------------

print("\nMissing values by column:")
print(products_clean.isna().sum())

# ------------------------------------------------------------
# 3. DUPLICATE RECORDS
# ------------------------------------------------------------

print("\nDuplicate rows:")
print("Exact duplicate rows:", products_clean.duplicated().sum())

print("Duplicate product IDs:",
      products_clean["product_id"].duplicated().sum())

# ------------------------------------------------------------
# 4. PRODUCT ID CHECK
# ------------------------------------------------------------

print("\nProduct ID audit:")
print("Missing product IDs:", products_clean["product_id"].isna().sum())
print("Unique product IDs:", products_clean["product_id"].nunique())

# ------------------------------------------------------------
# 5. CATEGORICAL VALUES
# ------------------------------------------------------------

print("\nCategory values:")
print(products_clean["category"].value_counts(dropna=False))

print("\nSub-category values:")
print("Unique sub-categories:",
      products_clean["sub_category"].nunique(dropna=True))

print(products_clean["sub_category"].value_counts(dropna=False).head(20))

print("\nBrand values:")
print("Unique brands:",
      products_clean["brand"].nunique(dropna=True))

# ------------------------------------------------------------
# 6. PRODUCT NAME CHECK
# ------------------------------------------------------------

print("\nProduct name audit:")
print("Unique product names:",
      products_clean["product_name"].nunique())

print("Duplicate product names:",
      products_clean["product_name"].duplicated().sum())

# ------------------------------------------------------------
# 7. ACTIVE STATUS
# ------------------------------------------------------------

print("\nActive-status values:")
print(products_clean["is_active"].value_counts(dropna=False))

# ------------------------------------------------------------
# 8. PRICE AND COST AUDIT
# ------------------------------------------------------------

print("\nUnit cost audit:")
print("Missing unit cost:",
      products_clean["unit_cost"].isna().sum())

print("Negative unit cost:",
      (products_clean["unit_cost"] < 0).sum())

print("\nUnit price audit:")
print("Missing unit price:",
      products_clean["unit_price"].isna().sum())

print("Negative unit price:",
      (products_clean["unit_price"] < 0).sum())

# ------------------------------------------------------------
# 9. COST GREATER THAN PRICE
# ------------------------------------------------------------

cost_greater_than_price = (
    products_clean["unit_cost"] > products_clean["unit_price"]
).sum()

print("\nProducts where unit cost > unit price:",
      cost_greater_than_price)

# ------------------------------------------------------------
# 10. DATE AUDIT
# ------------------------------------------------------------

print("\nLaunch-date audit:")
print("Missing launch dates:",
      products_clean["launch_date"].isna().sum())

print("Launch-date dtype:",
      products_clean["launch_date"].dtype)

# ------------------------------------------------------------
# 11. SUMMARY
# ------------------------------------------------------------

print("\nAUDIT COMPLETE")
print("=" * 70)

PRODUCTS - INITIAL DATA-QUALITY AUDIT
Shape: (220, 9)

Data types:
product_id       object
product_name     object
category         object
sub_category     object
brand            object
unit_cost       float64
unit_price       object
launch_date      object
is_active        object
dtype: object

Missing values by column:
product_id       0
product_name     0
category         0
sub_category    11
brand            0
unit_cost        0
unit_price       0
launch_date      0
is_active        0
dtype: int64

Duplicate rows:
Exact duplicate rows: 0
Duplicate product IDs: 0

Product ID audit:
Missing product IDs: 0
Unique product IDs: 220

Category values:
category
Electronics               48
Beauty & Personal Care    45
Home & Kitchen            45
Sports & Outdoors         44
Apparel                   38
Name: count, dtype: int64

Sub-category values:
Unique sub-categories: 23
sub_category
Small Appliances    16
Cycling             16
Camping             13
Wellness            12
Smart Hom

TypeError: '<' not supported between instances of 'str' and 'int'

In [ ]:
# ============================================================
# CELL 24: INVESTIGATE PRODUCT UNIT PRICE FORMAT
# ============================================================

print("PRODUCT UNIT PRICE FORMAT INVESTIGATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. CHECK THE DATA TYPE
# ------------------------------------------------------------

print("Current dtype:")
print(products_clean["unit_price"].dtype)

# ------------------------------------------------------------
# 2. SHOW SAMPLE VALUES
# ------------------------------------------------------------

print("\nSample unit_price values:")
print(products_clean["unit_price"].head(20).to_string(index=False))

# ------------------------------------------------------------
# 3. SHOW PYTHON DATA TYPES INSIDE THE COLUMN
# ------------------------------------------------------------

print("\nPython data types found inside unit_price:")

print(
    products_clean["unit_price"]
    .map(type)
    .value_counts()
)

# ------------------------------------------------------------
# 4. CHECK WHETHER VALUES CONTAIN A DOLLAR SIGN
# ------------------------------------------------------------

unit_price_text = products_clean["unit_price"].astype("string")

currency_values = unit_price_text.str.contains(
    r"\$",
    regex=True,
    na=False
)

print("\nValues containing '$':")
print(currency_values.sum())

# ------------------------------------------------------------
# 5. CHECK FOR NON-NUMERIC VALUES
# ------------------------------------------------------------

numeric_test = pd.to_numeric(
    unit_price_text.str.replace("$", "", regex=False).str.strip(),
    errors="coerce"
)

non_numeric_count = numeric_test.isna().sum()

print("\nValues that cannot be converted to numeric:")
print(non_numeric_count)

if non_numeric_count > 0:
    print("\nExamples of non-numeric values:")
    print(
        products_clean.loc[
            numeric_test.isna(),
            "unit_price"
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 6. SHOW BASIC NUMERIC SUMMARY
# ------------------------------------------------------------

print("\nNumeric unit_price summary:")
print(numeric_test.describe())

print("\nInvestigation complete.")
print("=" * 70)

PRODUCT UNIT PRICE FORMAT INVESTIGATION
Current dtype:
object

Sample unit_price values:
 415.24
 645.92
$100.91
  70.18
 571.72
  33.47
 540.02
 119.71
 259.81
 304.03
 371.71
$221.37
  457.4
 172.07
  326.8
 380.53
  81.05
 970.84
 180.28
 538.98

Python data types found inside unit_price:
unit_price
<class 'str'>    220
Name: count, dtype: int64

Values containing '$':
22

Values that cannot be converted to numeric:
0

Numeric unit_price summary:
count         220.0
mean        406.414
std      269.160514
min            9.25
25%        180.9625
50%         370.005
75%          644.54
max         1027.96
Name: unit_price, dtype: Float64

Investigation complete.


In [ ]:
# ============================================================
# CELL 25: CLEAN PRODUCT UNIT PRICE
# ============================================================

print("CLEANING PRODUCT UNIT PRICE")
print("=" * 70)

# ------------------------------------------------------------
# 1. COUNT CURRENCY-FORMATTED VALUES BEFORE CLEANING
# ------------------------------------------------------------

before_currency_count = (
    products_clean["unit_price"]
    .astype("string")
    .str.contains(r"\$", regex=True, na=False)
    .sum()
)

print("Currency-formatted values before cleaning:",
      before_currency_count)

# ------------------------------------------------------------
# 2. REMOVE '$' AND CONVERT TO NUMERIC
# ------------------------------------------------------------

products_clean["unit_price"] = (
    products_clean["unit_price"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.strip()
)

products_clean["unit_price"] = pd.to_numeric(
    products_clean["unit_price"],
    errors="coerce"
)

# ------------------------------------------------------------
# 3. VALIDATE THE CLEANED COLUMN
# ------------------------------------------------------------

print("\nData type after cleaning:")
print(products_clean["unit_price"].dtype)

print("\nMissing unit prices after cleaning:")
print(products_clean["unit_price"].isna().sum())

print("\nNegative unit prices after cleaning:")
print((products_clean["unit_price"] < 0).sum())

print("\nUnit price summary after cleaning:")
print(products_clean["unit_price"].describe())

print("\nCleaning complete.")
print("=" * 70)

CLEANING PRODUCT UNIT PRICE
Currency-formatted values before cleaning: 22

Data type after cleaning:
Float64

Missing unit prices after cleaning:
0

Negative unit prices after cleaning:
0

Unit price summary after cleaning:
count         220.0
mean        406.414
std      269.160514
min            9.25
25%        180.9625
50%         370.005
75%          644.54
max         1027.96
Name: unit_price, dtype: Float64

Cleaning complete.


In [ ]:
# ============================================================
# CELL 26: PRODUCT COST VS SELLING PRICE INVESTIGATION
# ============================================================

print("PRODUCT COST VS SELLING PRICE INVESTIGATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. CALCULATE PRODUCTS WHERE COST EXCEEDS SELLING PRICE
# ------------------------------------------------------------

cost_greater_than_price = (
    products_clean["unit_cost"] > products_clean["unit_price"]
)

print("Products where unit cost > unit price:",
      cost_greater_than_price.sum())

# ------------------------------------------------------------
# 2. CALCULATE PRODUCTS WHERE COST EQUALS PRICE
# ------------------------------------------------------------

cost_equal_price = (
    products_clean["unit_cost"] == products_clean["unit_price"]
)

print("Products where unit cost = unit price:",
      cost_equal_price.sum())

# ------------------------------------------------------------
# 3. CALCULATE PRODUCTS WHERE PRICE EXCEEDS COST
# ------------------------------------------------------------

price_greater_than_cost = (
    products_clean["unit_price"] > products_clean["unit_cost"]
)

print("Products where unit price > unit cost:",
      price_greater_than_cost.sum())

# ------------------------------------------------------------
# 4. CALCULATE GROSS MARGIN
# ------------------------------------------------------------

products_clean["gross_margin"] = (
    products_clean["unit_price"] - products_clean["unit_cost"]
)

products_clean["gross_margin_pct"] = np.where(
    products_clean["unit_price"] > 0,
    (products_clean["gross_margin"] /
     products_clean["unit_price"]) * 100,
    np.nan
)

# ------------------------------------------------------------
# 5. DISPLAY PRODUCTS WITH COST GREATER THAN PRICE
# ------------------------------------------------------------

if cost_greater_than_price.sum() > 0:

    print("\nProducts where cost exceeds selling price:")
    print(
        products_clean.loc[
            cost_greater_than_price,
            [
                "product_id",
                "product_name",
                "category",
                "unit_cost",
                "unit_price",
                "gross_margin",
                "gross_margin_pct"
            ]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 6. MARGIN SUMMARY
# ------------------------------------------------------------

print("\nGross-margin summary:")
print(products_clean["gross_margin"].describe())

print("\nGross-margin percentage summary:")
print(products_clean["gross_margin_pct"].describe())

print("\nInvestigation complete.")
print("=" * 70)

PRODUCT COST VS SELLING PRICE INVESTIGATION
Products where unit cost > unit price: 6
Products where unit cost = unit price: 0
Products where unit price > unit cost: 214

Products where cost exceeds selling price:
product_id                   product_name          category  unit_cost  unit_price  gross_margin  gross_margin_pct
  PROD1027 Everwood Fitness Equipment Max Sports & Outdoors    968.910      645.94       -322.97             -50.0
  PROD1031       Lumen Cookware Essential    Home & Kitchen    914.385      609.59      -304.795             -50.0
  PROD1037      Pinecrest Smart Home Plus       Electronics   1019.355      679.57      -339.785             -50.0
  PROD1039  Meridian Basics Smartphones X       Electronics    322.650       215.1       -107.55             -50.0
  PROD1121              Vertex Team Sport Sports & Outdoors   1019.340      679.56       -339.78             -50.0
  PROD1170        Cadence Accessories Max           Apparel    599.085      399.39      -199.695 

In [ ]:
# ============================================================
# CELL 27: INVESTIGATE NEGATIVE-MARGIN PATTERN
# ============================================================

print("NEGATIVE-MARGIN PATTERN INVESTIGATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. SELECT PRODUCTS WITH NEGATIVE GROSS MARGIN
# ------------------------------------------------------------

negative_margin = products_clean["gross_margin"] < 0

negative_margin_products = products_clean.loc[
    negative_margin,
    [
        "product_id",
        "product_name",
        "category",
        "unit_cost",
        "unit_price",
        "gross_margin",
        "gross_margin_pct"
    ]
].copy()

print("Number of negative-margin products:",
      len(negative_margin_products))

# ------------------------------------------------------------
# 2. CALCULATE COST-TO-PRICE RATIO
# ------------------------------------------------------------

negative_margin_products["cost_price_ratio"] = (
    negative_margin_products["unit_cost"] /
    negative_margin_products["unit_price"]
)

print("\nNegative-margin products with cost/price ratio:")
print(
    negative_margin_products.to_string(index=False)
)

# ------------------------------------------------------------
# 3. CHECK WHETHER ALL HAVE THE SAME RATIO
# ------------------------------------------------------------

print("\nCost-to-price ratio summary:")
print(
    negative_margin_products["cost_price_ratio"].describe()
)

# ------------------------------------------------------------
# 4. CHECK WHETHER COST IS EXACTLY 1.5 × PRICE
# ------------------------------------------------------------

expected_cost = (
    negative_margin_products["unit_price"] * 1.5
)

negative_margin_products["expected_cost_1_5x"] = expected_cost

negative_margin_products["difference"] = (
    negative_margin_products["unit_cost"] -
    negative_margin_products["expected_cost_1_5x"]
)

print("\nComparison with cost = 1.5 × unit price:")
print(
    negative_margin_products[
        [
            "product_id",
            "unit_price",
            "unit_cost",
            "expected_cost_1_5x",
            "difference"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 5. CHECK THE ORIGINAL PRICE/COST RELATIONSHIP
# ------------------------------------------------------------

print("\nOverall cost-to-price ratio summary:")
print(
    (
        products_clean["unit_cost"] /
        products_clean["unit_price"]
    ).describe()
)

print("\nInvestigation complete.")
print("=" * 70)

NEGATIVE-MARGIN PATTERN INVESTIGATION
Number of negative-margin products: 6

Negative-margin products with cost/price ratio:
product_id                   product_name          category  unit_cost  unit_price  gross_margin  gross_margin_pct  cost_price_ratio
  PROD1027 Everwood Fitness Equipment Max Sports & Outdoors    968.910      645.94       -322.97             -50.0               1.5
  PROD1031       Lumen Cookware Essential    Home & Kitchen    914.385      609.59      -304.795             -50.0               1.5
  PROD1037      Pinecrest Smart Home Plus       Electronics   1019.355      679.57      -339.785             -50.0               1.5
  PROD1039  Meridian Basics Smartphones X       Electronics    322.650       215.1       -107.55             -50.0               1.5
  PROD1121              Vertex Team Sport Sports & Outdoors   1019.340      679.56       -339.78             -50.0               1.5
  PROD1170        Cadence Accessories Max           Apparel    599.085      3

In [ ]:
# ============================================================
# CELL 28: PRODUCTS - CATEGORY, STATUS AND DATE AUDIT
# ============================================================

print("PRODUCTS - CATEGORY, STATUS AND DATE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. CATEGORY STANDARDIZATION CHECK
# ------------------------------------------------------------

print("Category values:")
print(products_clean["category"].value_counts(dropna=False))

# ------------------------------------------------------------
# 2. SUB-CATEGORY MISSINGNESS
# ------------------------------------------------------------

print("\nSub-category missing values:")
print(products_clean["sub_category"].isna().sum())

# ------------------------------------------------------------
# 3. BRAND AUDIT
# ------------------------------------------------------------

print("\nBrand missing values:")
print(products_clean["brand"].isna().sum())

print("\nNumber of unique brands:")
print(products_clean["brand"].nunique(dropna=True))

# ------------------------------------------------------------
# 4. ACTIVE STATUS AUDIT
# ------------------------------------------------------------

print("\nActive-status values:")
print(products_clean["is_active"].value_counts(dropna=False))

# ------------------------------------------------------------
# 5. CHECK FOR UNEXPECTED ACTIVE-STATUS VALUES
# ------------------------------------------------------------

expected_active = {"Yes", "No"}

actual_active = set(
    products_clean["is_active"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("\nUnexpected active-status values:")
print(actual_active - expected_active)

# ------------------------------------------------------------
# 6. LAUNCH DATE AUDIT
# ------------------------------------------------------------

print("\nLaunch-date audit:")
print("Current dtype:",
      products_clean["launch_date"].dtype)

print("Missing launch dates:",
      products_clean["launch_date"].isna().sum())

# ------------------------------------------------------------
# 7. TEST DATE CONVERSION
# ------------------------------------------------------------

launch_dates_test = pd.to_datetime(
    products_clean["launch_date"],
    errors="coerce"
)

print("\nUnparseable launch dates:",
      launch_dates_test.isna().sum())

print("Earliest launch date:",
      launch_dates_test.min())

print("Latest launch date:",
      launch_dates_test.max())

# ------------------------------------------------------------
# 8. FUTURE DATE CHECK
# ------------------------------------------------------------

today = pd.Timestamp.today().normalize()

future_launch_dates = (
    launch_dates_test > today
).sum()

print("Future launch dates:",
      future_launch_dates)

# ------------------------------------------------------------
# 9. PRODUCT NAME DUPLICATE INVESTIGATION
# ------------------------------------------------------------

duplicate_names = products_clean[
    products_clean["product_name"].duplicated(keep=False)
].sort_values("product_name")

print("\nDuplicate product names:")
if len(duplicate_names) > 0:
    print(
        duplicate_names[
            [
                "product_id",
                "product_name",
                "category",
                "sub_category",
                "brand",
                "unit_cost",
                "unit_price",
                "is_active"
            ]
        ].to_string(index=False)
    )
else:
    print("No duplicate product names found.")

print("\nAudit complete.")
print("=" * 70)

PRODUCTS - CATEGORY, STATUS AND DATE AUDIT
Category values:
category
Electronics               48
Beauty & Personal Care    45
Home & Kitchen            45
Sports & Outdoors         44
Apparel                   38
Name: count, dtype: int64

Sub-category missing values:
11

Brand missing values:
0

Number of unique brands:
15

Active-status values:
is_active
Yes    194
No      26
Name: count, dtype: int64

Unexpected active-status values:
set()

Launch-date audit:
Current dtype: object
Missing launch dates: 0

Unparseable launch dates: 0
Earliest launch date: 2021-09-14 00:00:00
Latest launch date: 2026-05-05 00:00:00
Future launch dates: 0

Duplicate product names:
product_id               product_name               category sub_category     brand  unit_cost  unit_price is_active
  PROD1024 Pinecrest Skincare Classic Beauty & Personal Care     Skincare Pinecrest     220.50      347.79       Yes
  PROD1150 Pinecrest Skincare Classic Beauty & Personal Care     Skincare Pinecrest      24.

In [ ]:
# ============================================================
# CELL 29: INVESTIGATE DUPLICATE PRODUCT NAME
# ============================================================

print("DUPLICATE PRODUCT NAME INVESTIGATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. IDENTIFY PRODUCT NAMES THAT APPEAR MORE THAN ONCE
# ------------------------------------------------------------

duplicate_name_counts = (
    products_clean["product_name"]
    .value_counts()
)

duplicate_names_list = duplicate_name_counts[
    duplicate_name_counts > 1
].index.tolist()

print("Product names appearing more than once:")
print(duplicate_names_list)

# ------------------------------------------------------------
# 2. DISPLAY ALL RECORDS FOR THE DUPLICATE NAME(S)
# ------------------------------------------------------------

duplicate_name_records = products_clean[
    products_clean["product_name"].isin(duplicate_names_list)
].copy()

print("\nDuplicate-name records:")
print(
    duplicate_name_records[
        [
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "brand",
            "unit_cost",
            "unit_price",
            "is_active",
            "launch_date"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 3. CHECK WHETHER THE RECORDS ARE IDENTICAL
# ------------------------------------------------------------

comparison_columns = [
    "category",
    "sub_category",
    "brand",
    "unit_cost",
    "unit_price",
    "is_active",
    "launch_date"
]

print("\nAre the non-ID attributes identical?")

for product_name in duplicate_names_list:

    records = products_clean[
        products_clean["product_name"] == product_name
    ][comparison_columns]

    print(
        f"{product_name}:",
        records.duplicated().any()
    )

# ------------------------------------------------------------
# 4. CHECK WHETHER PRODUCT IDs ARE DIFFERENT
# ------------------------------------------------------------

print("\nNumber of unique product IDs among duplicate-name records:")
print(
    duplicate_name_records["product_id"].nunique()
)

# ------------------------------------------------------------
# 5. FINAL INTERPRETATION
# ------------------------------------------------------------

print("\nInvestigation complete.")
print("=" * 70)

DUPLICATE PRODUCT NAME INVESTIGATION
Product names appearing more than once:
['Pinecrest Skincare Classic']

Duplicate-name records:
product_id               product_name               category sub_category     brand  unit_cost  unit_price is_active launch_date
  PROD1024 Pinecrest Skincare Classic Beauty & Personal Care     Skincare Pinecrest     220.50      347.79       Yes  2022-09-04
  PROD1150 Pinecrest Skincare Classic Beauty & Personal Care     Skincare Pinecrest      24.47       51.31       Yes  2022-05-22

Are the non-ID attributes identical?
Pinecrest Skincare Classic: False

Number of unique product IDs among duplicate-name records:
2

Investigation complete.


In [ ]:
# ============================================================
# CELL 30: CLEAN PRODUCT LAUNCH DATE
# ============================================================

print("CLEANING PRODUCT LAUNCH DATE")
print("=" * 70)

# ------------------------------------------------------------
# 1. STORE ORIGINAL MISSING COUNT
# ------------------------------------------------------------

missing_before = products_clean["launch_date"].isna().sum()

print("Missing launch dates before cleaning:",
      missing_before)

print("Original dtype:",
      products_clean["launch_date"].dtype)

# ------------------------------------------------------------
# 2. CONVERT TO DATETIME
# ------------------------------------------------------------

products_clean["launch_date"] = pd.to_datetime(
    products_clean["launch_date"],
    errors="coerce"
)

# ------------------------------------------------------------
# 3. VALIDATE THE CONVERSION
# ------------------------------------------------------------

print("\nDtype after cleaning:")
print(products_clean["launch_date"].dtype)

print("\nMissing launch dates after cleaning:")
print(products_clean["launch_date"].isna().sum())

print("\nUnparseable dates created:")
print(
    products_clean["launch_date"].isna().sum()
    - missing_before
)

print("\nEarliest launch date:")
print(products_clean["launch_date"].min())

print("\nLatest launch date:")
print(products_clean["launch_date"].max())

print("\nCleaning complete.")
print("=" * 70)

CLEANING PRODUCT LAUNCH DATE
Missing launch dates before cleaning: 0
Original dtype: object

Dtype after cleaning:
datetime64[ns]

Missing launch dates after cleaning:
0

Unparseable dates created:
0

Earliest launch date:
2021-09-14 00:00:00

Latest launch date:
2026-05-05 00:00:00

Cleaning complete.


In [ ]:
# ============================================================
# CELL 31: FINAL PRODUCTS VALIDATION
# ============================================================

print("FINAL PRODUCTS VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. ROW AND COLUMN COUNT
# ------------------------------------------------------------

print("Shape:", products_clean.shape)

# ------------------------------------------------------------
# 2. PRIMARY KEY VALIDATION
# ------------------------------------------------------------

print("\nPRIMARY KEY CHECK")
print("Missing product IDs:",
      products_clean["product_id"].isna().sum())

print("Duplicate product IDs:",
      products_clean["product_id"].duplicated().sum())

print("Unique product IDs:",
      products_clean["product_id"].nunique())

# ------------------------------------------------------------
# 3. REQUIRED FIELD VALIDATION
# ------------------------------------------------------------

print("\nREQUIRED FIELD CHECK")

for column in [
    "product_name",
    "category",
    "unit_cost",
    "unit_price"
]:
    print(
        f"{column} missing:",
        products_clean[column].isna().sum()
    )

# ------------------------------------------------------------
# 4. PRICE AND COST VALIDATION
# ------------------------------------------------------------

print("\nPRICE / COST CHECK")

print("Negative unit costs:",
      (products_clean["unit_cost"] < 0).sum())

print("Negative unit prices:",
      (products_clean["unit_price"] < 0).sum())

print("Products where cost > price:",
      (products_clean["unit_cost"] >
       products_clean["unit_price"]).sum())

# ------------------------------------------------------------
# 5. CATEGORY VALIDATION
# ------------------------------------------------------------

expected_categories = {
    "Electronics",
    "Beauty & Personal Care",
    "Home & Kitchen",
    "Sports & Outdoors",
    "Apparel"
}

actual_categories = set(
    products_clean["category"].dropna().unique()
)

print("\nCATEGORY CHECK")
print("Unexpected categories:",
      actual_categories - expected_categories)

# ------------------------------------------------------------
# 6. ACTIVE STATUS VALIDATION
# ------------------------------------------------------------

expected_active = {"Yes", "No"}

actual_active = set(
    products_clean["is_active"].dropna().unique()
)

print("\nACTIVE STATUS CHECK")
print("Unexpected values:",
      actual_active - expected_active)

# ------------------------------------------------------------
# 7. DATE VALIDATION
# ------------------------------------------------------------

print("\nDATE CHECK")
print("Launch-date dtype:",
      products_clean["launch_date"].dtype)

print("Missing launch dates:",
      products_clean["launch_date"].isna().sum())

# ------------------------------------------------------------
# 8. TEMPORARY INVESTIGATION COLUMNS
# ------------------------------------------------------------

print("\nTEMPORARY COLUMN CHECK")

temporary_columns = [
    "gross_margin",
    "gross_margin_pct"
]

for column in temporary_columns:
    print(
        f"{column} exists:",
        column in products_clean.columns
    )

print("\nProduct validation complete.")
print("=" * 70)

FINAL PRODUCTS VALIDATION
Shape: (220, 11)

PRIMARY KEY CHECK
Missing product IDs: 0
Duplicate product IDs: 0
Unique product IDs: 220

REQUIRED FIELD CHECK
product_name missing: 0
category missing: 0
unit_cost missing: 0
unit_price missing: 0

PRICE / COST CHECK
Negative unit costs: 0
Negative unit prices: 0
Products where cost > price: 6

CATEGORY CHECK
Unexpected categories: set()

ACTIVE STATUS CHECK
Unexpected values: set()

DATE CHECK
Launch-date dtype: datetime64[ns]
Missing launch dates: 0

TEMPORARY COLUMN CHECK
gross_margin exists: True
gross_margin_pct exists: True

Product validation complete.


In [ ]:
# ============================================================
# CELL 32: REMOVE TEMPORARY PRODUCT INVESTIGATION COLUMNS
# ============================================================

print("REMOVING TEMPORARY PRODUCT INVESTIGATION COLUMNS")
print("=" * 70)

# ------------------------------------------------------------
# 1. REMOVE COLUMNS CREATED ONLY FOR INVESTIGATION
# ------------------------------------------------------------

temporary_columns = [
    "gross_margin",
    "gross_margin_pct"
]

products_clean = products_clean.drop(
    columns=temporary_columns,
    errors="ignore"
)

# ------------------------------------------------------------
# 2. VERIFY FINAL PRODUCT STRUCTURE
# ------------------------------------------------------------

print("Final products shape:",
      products_clean.shape)

print("\nFinal products columns:")
print(products_clean.columns.tolist())

print("\nFinal products data types:")
print(products_clean.dtypes)

# ------------------------------------------------------------
# 3. CONFIRM TEMPORARY COLUMNS ARE GONE
# ------------------------------------------------------------

remaining_temporary = [
    column
    for column in temporary_columns
    if column in products_clean.columns
]

print("\nTemporary columns remaining:",
      remaining_temporary)

print("\nProducts section cleanup complete.")
print("=" * 70)

REMOVING TEMPORARY PRODUCT INVESTIGATION COLUMNS
Final products shape: (220, 9)

Final products columns:
['product_id', 'product_name', 'category', 'sub_category', 'brand', 'unit_cost', 'unit_price', 'launch_date', 'is_active']

Final products data types:
product_id              object
product_name            object
category                object
sub_category            object
brand                   object
unit_cost              float64
unit_price             Float64
launch_date     datetime64[ns]
is_active               object
dtype: object

Temporary columns remaining: []

Products section cleanup complete.


In [ ]:
# ============================================================
# CELL 33: ORDERS - INITIAL DATA-QUALITY AUDIT
# ============================================================

print("ORDERS - INITIAL DATA-QUALITY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. BASIC INFORMATION
# ------------------------------------------------------------

print("Shape:", orders_clean.shape)

print("\nData types:")
print(orders_clean.dtypes)

# ------------------------------------------------------------
# 2. MISSING VALUES
# ------------------------------------------------------------

print("\nMissing values by column:")
print(orders_clean.isna().sum())

# ------------------------------------------------------------
# 3. EXACT DUPLICATE ROWS
# ------------------------------------------------------------

print("\nDuplicate audit:")
print("Exact duplicate rows:",
      orders_clean.duplicated().sum())

# ------------------------------------------------------------
# 4. ORDER ID DUPLICATES
# ------------------------------------------------------------

print("Duplicate order IDs:",
      orders_clean["order_id"].duplicated().sum())

print("Unique order IDs:",
      orders_clean["order_id"].nunique())

# ------------------------------------------------------------
# 5. ORDER ID MISSINGNESS
# ------------------------------------------------------------

print("\nOrder ID audit:")
print("Missing order IDs:",
      orders_clean["order_id"].isna().sum())

# ------------------------------------------------------------
# 6. CUSTOMER ID AUDIT
# ------------------------------------------------------------

print("\nCustomer ID audit:")
print("Missing customer IDs:",
      orders_clean["customer_id"].isna().sum())

print("Unique customer IDs:",
      orders_clean["customer_id"].nunique())

# ------------------------------------------------------------
# 7. ORDER STATUS
# ------------------------------------------------------------

print("\nOrder-status values:")
print(orders_clean["order_status"].value_counts(dropna=False))

# ------------------------------------------------------------
# 8. CHANNEL
# ------------------------------------------------------------

print("\nOrder-channel values:")
print(orders_clean["channel"].value_counts(dropna=False))

# ------------------------------------------------------------
# 9. PAYMENT METHOD
# ------------------------------------------------------------

print("\nPayment-method values:")
print(orders_clean["payment_method"].value_counts(dropna=False))

# ------------------------------------------------------------
# 10. SHIPPING COST
# ------------------------------------------------------------

print("\nShipping-cost audit:")
print("Missing shipping cost:",
      orders_clean["shipping_cost"].isna().sum())

print("Negative shipping cost:",
      (orders_clean["shipping_cost"] < 0).sum())

# ------------------------------------------------------------
# 11. TOTAL AMOUNT
# ------------------------------------------------------------

print("\nTotal-amount audit:")
print("Missing total amount:",
      orders_clean["total_amount"].isna().sum())

print("Negative total amount:",
      (orders_clean["total_amount"] < 0).sum())

# ------------------------------------------------------------
# 12. DISCOUNT
# ------------------------------------------------------------

print("\nDiscount audit:")
print("Missing discount:",
      orders_clean["discount_pct"].isna().sum())

print("Discount values:")
print(
    orders_clean["discount_pct"]
    .value_counts(dropna=False)
    .sort_index()
)

# ------------------------------------------------------------
# 13. ORDER DATE
# ------------------------------------------------------------

print("\nOrder-date audit:")
print("Current dtype:",
      orders_clean["order_date"].dtype)

print("Missing order dates:",
      orders_clean["order_date"].isna().sum())

# Test date conversion without modifying the dataframe yet.
order_dates_test = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce"
)

print("Unparseable order dates:",
      order_dates_test.isna().sum())

print("Earliest parsed order date:",
      order_dates_test.min())

print("Latest parsed order date:",
      order_dates_test.max())

print("\nInitial orders audit complete.")
print("=" * 70)

ORDERS - INITIAL DATA-QUALITY AUDIT
Shape: (28297, 9)

Data types:
order_id           object
customer_id        object
order_date         object
channel            object
order_status       object
payment_method     object
shipping_cost     float64
discount_pct        int64
total_amount      float64
dtype: object

Missing values by column:
order_id            0
customer_id         0
order_date        424
channel             0
order_status        0
payment_method      0
shipping_cost       0
discount_pct        0
total_amount        0
dtype: int64

Duplicate audit:
Exact duplicate rows: 25
Duplicate order IDs: 25
Unique order IDs: 28272

Order ID audit:
Missing order IDs: 0

Customer ID audit:
Missing customer IDs: 0
Unique customer IDs: 5683

Order-status values:
order_status
Completed    24435
Returned      1168
Cancelled      935
Pending        557
COMPLETED      539
completed      530
RETURNED        33
returned        30
CANCELLED       27
pending         18
cancelled       14
PEND

In [ ]:
# ============================================================
# CELL 34: INVESTIGATE DUPLICATE ORDERS
# ============================================================

print("DUPLICATE ORDER INVESTIGATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. IDENTIFY DUPLICATE ORDER IDs
# ------------------------------------------------------------

duplicate_order_ids = (
    orders_clean.loc[
        orders_clean["order_id"].duplicated(keep=False),
        "order_id"
    ]
    .drop_duplicates()
    .tolist()
)

print("Number of duplicated order IDs:",
      len(duplicate_order_ids))

print("\nDuplicated order IDs:")
print(duplicate_order_ids)

# ------------------------------------------------------------
# 2. DISPLAY THE DUPLICATE RECORDS
# ------------------------------------------------------------

duplicate_orders = orders_clean[
    orders_clean["order_id"].isin(duplicate_order_ids)
].sort_values("order_id")

print("\nDuplicate order records:")
print(
    duplicate_orders.to_string(index=False)
)

# ------------------------------------------------------------
# 3. CHECK WHETHER EACH DUPLICATE ID HAS IDENTICAL ROWS
# ------------------------------------------------------------

print("\nChecking whether duplicate order records are identical:")

duplicate_group_check = (
    duplicate_orders
    .groupby("order_id")
    .size()
)

print("\nRows per duplicated order ID:")
print(duplicate_group_check.value_counts().sort_index())

# ------------------------------------------------------------
# 4. CHECK NUMBER OF UNIQUE RECORDS WITHIN EACH ID
# ------------------------------------------------------------

unique_records_per_id = (
    duplicate_orders
    .groupby("order_id")
    .apply(lambda group: len(group.drop_duplicates()))
)

print("\nUnique records within each duplicated order ID:")
print(unique_records_per_id.value_counts().sort_index())

# ------------------------------------------------------------
# 5. CHECK WHETHER ANY DUPLICATE ID HAS DIFFERENT VALUES
# ------------------------------------------------------------

different_value_duplicates = []

for order_id in duplicate_order_ids:

    group = orders_clean[
        orders_clean["order_id"] == order_id
    ]

    if len(group.drop_duplicates()) > 1:
        different_value_duplicates.append(order_id)

print("\nDuplicate IDs containing different records:")
print(different_value_duplicates)

print("\nDuplicate investigation complete.")
print("=" * 70)

DUPLICATE ORDER INVESTIGATION
Number of duplicated order IDs: 25

Duplicated order IDs:
['ORD102482', 'ORD120778', 'ORD118529', 'ORD126857', 'ORD122218', 'ORD125341', 'ORD127030', 'ORD103787', 'ORD104809', 'ORD127481', 'ORD120063', 'ORD118032', 'ORD119968', 'ORD123411', 'ORD119263', 'ORD116074', 'ORD126024', 'ORD120765', 'ORD120909', 'ORD104413', 'ORD103359', 'ORD103914', 'ORD109684', 'ORD107269', 'ORD109574']

Duplicate order records:
 order_id customer_id order_date    channel order_status payment_method  shipping_cost  discount_pct  total_amount
ORD102482  CUST005630 2025-03-17 Mobile App    Completed    Credit Card           0.00             0       2852.97
ORD102482  CUST005630 2025-03-17 Mobile App    Completed    Credit Card           0.00             0       2852.97
ORD103359  CUST005402 2025-05-29   In-Store    Completed    Credit Card           0.00             0        420.05
ORD103359  CUST005402 2025-05-29   In-Store    Completed    Credit Card           0.00             0

/tmp/ipykernel_3757/1109868374.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: len(group.drop_duplicates()))


In [ ]:
# ============================================================
# CELL 35: INVESTIGATE ORDER DATE AND SHIPPING COST ISSUES
# ============================================================

print("ORDER DATE AND SHIPPING COST INVESTIGATION")
print("=" * 70)


# ------------------------------------------------------------
# PART A: ORDER DATE INVESTIGATION
# ------------------------------------------------------------

print("\nA. ORDER DATE")
print("-" * 70)

# Count missing order dates
missing_order_dates = orders_clean["order_date"].isna().sum()

print("Missing order dates:", missing_order_dates)

# Convert dates temporarily for investigation
order_dates_test = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce"
)

print(
    "Unparseable order dates:",
    order_dates_test.isna().sum() - missing_order_dates
)

# Display the earliest and latest valid dates
print(
    "Earliest valid order date:",
    order_dates_test.min()
)

print(
    "Latest valid order date:",
    order_dates_test.max()
)

# Display examples of records with missing dates
print("\nSample records with missing order dates:")

print(
    orders_clean[
        orders_clean["order_date"].isna()
    ][
        [
            "order_id",
            "customer_id",
            "channel",
            "order_status",
            "payment_method",
            "shipping_cost",
            "discount_pct",
            "total_amount"
        ]
    ]
    .head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# PART B: NEGATIVE SHIPPING COST INVESTIGATION
# ------------------------------------------------------------

print("\n\nB. NEGATIVE SHIPPING COST")
print("-" * 70)

negative_shipping = orders_clean[
    orders_clean["shipping_cost"] < 0
]

print(
    "Number of orders with negative shipping cost:",
    len(negative_shipping)
)

print("\nNegative shipping-cost records:")

print(
    negative_shipping[
        [
            "order_id",
            "customer_id",
            "order_date",
            "channel",
            "order_status",
            "shipping_cost",
            "discount_pct",
            "total_amount"
        ]
    ]
    .sort_values("shipping_cost")
    .to_string(index=False)
)


# ------------------------------------------------------------
# PART C: CHECK WHETHER NEGATIVE VALUES FOLLOW A PATTERN
# ------------------------------------------------------------

print("\n\nC. NEGATIVE SHIPPING-COST PATTERN")
print("-" * 70)

if len(negative_shipping) > 0:

    print(
        "Negative shipping-cost values:"
    )

    print(
        negative_shipping["shipping_cost"]
        .value_counts()
        .sort_index()
    )

    print(
        "\nMinimum shipping cost:",
        negative_shipping["shipping_cost"].min()
    )

    print(
        "Maximum shipping cost:",
        negative_shipping["shipping_cost"].max()
    )


print("\nOrder date and shipping-cost investigation complete.")
print("=" * 70)

ORDER DATE AND SHIPPING COST INVESTIGATION

A. ORDER DATE
----------------------------------------------------------------------
Missing order dates: 424
Unparseable order dates: 0
Earliest valid order date: 2020-09-15 00:00:00
Latest valid order date: 2026-07-03 00:00:00

Sample records with missing order dates:
 order_id customer_id    channel order_status payment_method  shipping_cost  discount_pct  total_amount
ORD119703  CUST003480     Online    Completed         PayPal          12.99             0       1483.08
ORD102895  CUST003791   In-Store    Completed      Gift Card           0.00             0       1836.57
ORD101053  CUST003641 Mobile App    Completed      Gift Card           5.99             0       1326.77
ORD119231  CUST005663   In-Store     Returned   Store Credit           4.99             0       2387.41
ORD126573  CUST003678 Mobile App    Cancelled     Debit Card           0.00             0       1225.95
ORD115857  CUST005615     Online    Cancelled    Credit Card 

In [ ]:
# ============================================================
# CELL 36: INVESTIGATE ORDER STATUS AND CUSTOMER INTEGRITY
# ============================================================

print("ORDER STATUS AND CUSTOMER INTEGRITY INVESTIGATION")
print("=" * 70)


# ------------------------------------------------------------
# PART A: ORDER STATUS VARIANTS
# ------------------------------------------------------------

print("\nA. ORDER STATUS VARIANTS")
print("-" * 70)

print("Raw order-status values and frequencies:")

status_counts = (
    orders_clean["order_status"]
    .value_counts(dropna=False)
)

print(status_counts)


# Normalize only for investigation
status_normalized = (
    orders_clean["order_status"]
    .astype("string")
    .str.strip()
    .str.title()
)

print("\nStatus values after standardizing case:")

print(
    status_normalized
    .value_counts(dropna=False)
)


print("\nUnique standardized status values:")

print(
    sorted(
        status_normalized
        .dropna()
        .unique()
        .tolist()
    )
)


# ------------------------------------------------------------
# PART B: CHECK FOR MISSING OR UNEXPECTED STATUS VALUES
# ------------------------------------------------------------

print("\n\nB. STATUS VALIDITY")
print("-" * 70)

expected_statuses = {
    "Completed",
    "Returned",
    "Cancelled",
    "Pending"
}

actual_statuses = set(
    status_normalized
    .dropna()
    .unique()
)

unexpected_statuses = actual_statuses - expected_statuses

print("Expected statuses:")
print(expected_statuses)

print("\nUnexpected statuses:")
print(unexpected_statuses)

print(
    "\nMissing order-status values:",
    orders_clean["order_status"].isna().sum()
)


# ------------------------------------------------------------
# PART C: CUSTOMER REFERENTIAL INTEGRITY
# ------------------------------------------------------------

print("\n\nC. CUSTOMER REFERENTIAL INTEGRITY")
print("-" * 70)

# Get all valid customer IDs
valid_customer_ids = set(
    customers_clean["customer_id"]
    .dropna()
)

# Find orders whose customer does not exist
orphan_orders = orders_clean[
    ~orders_clean["customer_id"].isin(valid_customer_ids)
].copy()

print(
    "Number of orders linked to non-existent customers:",
    len(orphan_orders)
)

print(
    "Number of unique orphan customer IDs:",
    orphan_orders["customer_id"].nunique()
)

print("\nOrphan customer IDs:")

print(
    orphan_orders["customer_id"]
    .drop_duplicates()
    .tolist()
)


# ------------------------------------------------------------
# PART D: INSPECT ORPHAN ORDERS
# ------------------------------------------------------------

print("\n\nD. ORPHAN ORDER RECORDS")
print("-" * 70)

if len(orphan_orders) > 0:

    print(
        orphan_orders[
            [
                "order_id",
                "customer_id",
                "order_date",
                "channel",
                "order_status",
                "payment_method",
                "shipping_cost",
                "discount_pct",
                "total_amount"
            ]
        ]
        .sort_values("customer_id")
        .to_string(index=False)
    )


print("\nOrder status and customer integrity investigation complete.")
print("=" * 70)

ORDER STATUS AND CUSTOMER INTEGRITY INVESTIGATION

A. ORDER STATUS VARIANTS
----------------------------------------------------------------------
Raw order-status values and frequencies:
order_status
Completed    24435
Returned      1168
Cancelled      935
Pending        557
COMPLETED      539
completed      530
RETURNED        33
returned        30
CANCELLED       27
pending         18
cancelled       14
PENDING         11
Name: count, dtype: int64

Status values after standardizing case:
order_status
Completed    25504
Returned      1231
Cancelled      976
Pending        586
Name: count, dtype: Int64

Unique standardized status values:
['Cancelled', 'Completed', 'Pending', 'Returned']


B. STATUS VALIDITY
----------------------------------------------------------------------
Expected statuses:
{'Completed', 'Pending', 'Cancelled', 'Returned'}

Unexpected statuses:
set()

Missing order-status values: 0


C. CUSTOMER REFERENTIAL INTEGRITY
----------------------------------------------

In [ ]:
# ============================================================
# CELL 37: INVESTIGATE ORDER FINANCIAL FIELDS
# ============================================================

print("ORDER FINANCIAL FIELD INVESTIGATION")
print("=" * 70)


# ------------------------------------------------------------
# PART A: DISCOUNT PERCENTAGE
# ------------------------------------------------------------

print("\nA. DISCOUNT PERCENTAGE")
print("-" * 70)

print("Discount percentage data type:")
print(orders_clean["discount_pct"].dtype)

print("\nMissing discount values:")
print(orders_clean["discount_pct"].isna().sum())

print("\nUnique discount percentages:")
print(
    sorted(
        orders_clean["discount_pct"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nDiscount frequency:")
print(
    orders_clean["discount_pct"]
    .value_counts(dropna=False)
    .sort_index()
)


# Check whether any discount is outside the expected range
invalid_discount = orders_clean[
    (orders_clean["discount_pct"] < 0) |
    (orders_clean["discount_pct"] > 100)
]

print(
    "\nNumber of discounts outside 0–100%:",
    len(invalid_discount)
)


# ------------------------------------------------------------
# PART B: TOTAL AMOUNT
# ------------------------------------------------------------

print("\n\nB. TOTAL AMOUNT")
print("-" * 70)

print("Total amount data type:")
print(orders_clean["total_amount"].dtype)

print("\nMissing total amounts:")
print(orders_clean["total_amount"].isna().sum())

print(
    "\nNegative total amounts:",
    (orders_clean["total_amount"] < 0).sum()
)

print(
    "Zero total amounts:",
    (orders_clean["total_amount"] == 0).sum()
)

print("\nTotal amount descriptive statistics:")

print(
    orders_clean["total_amount"]
    .describe()
)


# ------------------------------------------------------------
# PART C: EXTREME TOTAL AMOUNTS
# ------------------------------------------------------------

print("\n\nC. EXTREME TOTAL AMOUNTS")
print("-" * 70)

q1 = orders_clean["total_amount"].quantile(0.25)
q3 = orders_clean["total_amount"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower IQR bound:", lower_bound)
print("Upper IQR bound:", upper_bound)

iqr_outliers = orders_clean[
    (orders_clean["total_amount"] < lower_bound) |
    (orders_clean["total_amount"] > upper_bound)
]

print(
    "\nNumber of IQR outliers:",
    len(iqr_outliers)
)

print("\nLargest total amounts:")

print(
    orders_clean[
        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
            "discount_pct",
            "total_amount"
        ]
    ]
    .sort_values("total_amount", ascending=False)
    .head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# PART D: CHECK SHIPPING COST RANGE
# ------------------------------------------------------------

print("\n\nD. SHIPPING COST RANGE")
print("-" * 70)

print(
    "Minimum shipping cost:",
    orders_clean["shipping_cost"].min()
)

print(
    "Maximum shipping cost:",
    orders_clean["shipping_cost"].max()
)

print(
    "Median shipping cost:",
    orders_clean["shipping_cost"].median()
)

print("\nShipping-cost frequency:")

print(
    orders_clean["shipping_cost"]
    .value_counts()
    .sort_index()
)


print("\nOrder financial field investigation complete.")
print("=" * 70)

ORDER FINANCIAL FIELD INVESTIGATION

A. DISCOUNT PERCENTAGE
----------------------------------------------------------------------
Discount percentage data type:
int64

Missing discount values:
0

Unique discount percentages:
[0, 5, 10, 15, 20, 25]

Discount frequency:
discount_pct
0     19943
5      2721
10     2223
15     1700
20     1092
25      618
Name: count, dtype: int64

Number of discounts outside 0–100%: 0


B. TOTAL AMOUNT
----------------------------------------------------------------------
Total amount data type:
float64

Missing total amounts:
0

Negative total amounts: 0
Zero total amounts: 0

Total amount descriptive statistics:
count    28297.000000
mean      1541.876389
std       1174.373886
min          7.070000
25%        671.230000
50%       1265.840000
75%       2125.780000
max      10536.770000
Name: total_amount, dtype: float64


C. EXTREME TOTAL AMOUNTS
----------------------------------------------------------------------
Q1: 671.23
Q3: 2125.78
IQR: 1454.5500

In [ ]:
# ============================================================
# CELL 38: QUANTIFY OVERLAPPING ORDER-QUALITY ISSUES
# ============================================================

print("ORDER-QUALITY ISSUE OVERLAP ANALYSIS")
print("=" * 70)


# ------------------------------------------------------------
# CREATE ISSUE FLAGS
# ------------------------------------------------------------

# Duplicate order IDs
duplicate_order_flag = orders_clean["order_id"].duplicated(
    keep=False
)

# Missing order dates
missing_date_flag = orders_clean["order_date"].isna()

# Negative shipping costs
negative_shipping_flag = (
    orders_clean["shipping_cost"] < 0
)

# Orphan customer IDs
valid_customer_ids = set(
    customers_clean["customer_id"].dropna()
)

orphan_customer_flag = ~orders_clean[
    "customer_id"
].isin(valid_customer_ids)


# ------------------------------------------------------------
# INDIVIDUAL ISSUE COUNTS
# ------------------------------------------------------------

print("\nINDIVIDUAL ISSUE COUNTS")
print("-" * 70)

print(
    "Rows involved in duplicate order IDs:",
    duplicate_order_flag.sum()
)

print(
    "Rows with missing order dates:",
    missing_date_flag.sum()
)

print(
    "Rows with negative shipping costs:",
    negative_shipping_flag.sum()
)

print(
    "Rows linked to orphan customers:",
    orphan_customer_flag.sum()
)


# ------------------------------------------------------------
# OVERLAP BETWEEN ISSUES
# ------------------------------------------------------------

print("\n\nISSUE OVERLAPS")
print("-" * 70)

print(
    "Duplicate + missing date:",
    (
        duplicate_order_flag
        & missing_date_flag
    ).sum()
)

print(
    "Duplicate + negative shipping:",
    (
        duplicate_order_flag
        & negative_shipping_flag
    ).sum()
)

print(
    "Duplicate + orphan customer:",
    (
        duplicate_order_flag
        & orphan_customer_flag
    ).sum()
)

print(
    "Missing date + negative shipping:",
    (
        missing_date_flag
        & negative_shipping_flag
    ).sum()
)

print(
    "Missing date + orphan customer:",
    (
        missing_date_flag
        & orphan_customer_flag
    ).sum()
)

print(
    "Negative shipping + orphan customer:",
    (
        negative_shipping_flag
        & orphan_customer_flag
    ).sum()
)


# ------------------------------------------------------------
# RECORDS WITH MULTIPLE ISSUES
# ------------------------------------------------------------

issue_count = (
    duplicate_order_flag.astype(int)
    + missing_date_flag.astype(int)
    + negative_shipping_flag.astype(int)
    + orphan_customer_flag.astype(int)
)

print("\n\nNUMBER OF ISSUES PER ROW")
print("-" * 70)

print(
    issue_count.value_counts()
    .sort_index()
)


print("\nRows containing two or more issues:")

multi_issue_rows = orders_clean[
    issue_count >= 2
]

print(
    multi_issue_rows[
        [
            "order_id",
            "customer_id",
            "order_date",
            "shipping_cost"
        ]
    ]
    .sort_values("order_id")
    .to_string(index=False)
)


print("\nOrder-quality overlap analysis complete.")
print("=" * 70)

ORDER-QUALITY ISSUE OVERLAP ANALYSIS

INDIVIDUAL ISSUE COUNTS
----------------------------------------------------------------------
Rows involved in duplicate order IDs: 50
Rows with missing order dates: 424
Rows with negative shipping costs: 8
Rows linked to orphan customers: 20


ISSUE OVERLAPS
----------------------------------------------------------------------
Duplicate + missing date: 0
Duplicate + negative shipping: 0
Duplicate + orphan customer: 0
Missing date + negative shipping: 0
Missing date + orphan customer: 1
Negative shipping + orphan customer: 0


NUMBER OF ISSUES PER ROW
----------------------------------------------------------------------
0    27796
1      500
2        1
Name: count, dtype: int64

Rows containing two or more issues:
 order_id customer_id order_date  shipping_cost
ORD110459  CUST900016        NaN           5.99

Order-quality overlap analysis complete.


In [ ]:
# ============================================================
# CELL 39: CLEAN AND TRANSFORM ORDERS
# ============================================================

print("ORDERS CLEANING AND TRANSFORMATION")
print("=" * 70)


# ------------------------------------------------------------
# SAVE ORIGINAL ROW COUNT
# ------------------------------------------------------------

orders_before = len(orders_clean)

print("Orders before cleaning:", orders_before)


# ------------------------------------------------------------
# STEP 1: REMOVE EXACT DUPLICATE ORDER RECORDS
# ------------------------------------------------------------

# Each duplicated order_id appeared exactly twice and the
# records were identical.
# Keeping the first copy removes only the redundant record.

duplicate_rows_before = orders_clean["order_id"].duplicated(
    keep="first"
).sum()

orders_clean = (
    orders_clean
    .drop_duplicates(subset="order_id", keep="first")
    .copy()
)

print(
    "\nDuplicate order rows removed:",
    duplicate_rows_before
)

print(
    "Rows after duplicate removal:",
    len(orders_clean)
)


# ------------------------------------------------------------
# STEP 2: STANDARDIZE ORDER STATUS
# ------------------------------------------------------------

# Convert values such as:
# COMPLETED / completed / Completed
# into:
# Completed

orders_clean["order_status"] = (
    orders_clean["order_status"]
    .astype("string")
    .str.strip()
    .str.title()
)

print("\nStandardized order statuses:")

print(
    orders_clean["order_status"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# STEP 3: CONVERT ORDER DATE TO DATETIME
# ------------------------------------------------------------

# Valid dates become datetime values.
# Missing dates remain NaT rather than being invented.

orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce"
)

print("\nOrder date dtype:")
print(orders_clean["order_date"].dtype)

print(
    "Missing order dates retained:",
    orders_clean["order_date"].isna().sum()
)


# ------------------------------------------------------------
# STEP 4: CORRECT NEGATIVE SHIPPING COSTS
# ------------------------------------------------------------

negative_shipping_before = (
    orders_clean["shipping_cost"] < 0
).sum()

# Negative shipping costs are treated as sign errors.
# Absolute value preserves the original magnitude.

orders_clean["shipping_cost"] = (
    orders_clean["shipping_cost"].abs()
)

print(
    "\nNegative shipping costs corrected:",
    negative_shipping_before
)

print(
    "Remaining negative shipping costs:",
    (
        orders_clean["shipping_cost"] < 0
    ).sum()
)


# ------------------------------------------------------------
# STEP 5: REMOVE ORPHAN CUSTOMER ORDERS
# ------------------------------------------------------------

valid_customer_ids = set(
    customers_clean["customer_id"]
    .dropna()
)

orphan_order_rows = (
    ~orders_clean["customer_id"]
    .isin(valid_customer_ids)
).sum()

orders_clean = orders_clean[
    orders_clean["customer_id"].isin(valid_customer_ids)
].copy()

print(
    "\nOrphan customer orders removed:",
    orphan_order_rows
)

print(
    "Rows after orphan removal:",
    len(orders_clean)
)


# ------------------------------------------------------------
# STEP 6: VERIFY TOTAL AMOUNT OUTLIERS WERE RETAINED
# ------------------------------------------------------------

q1 = orders_clean["total_amount"].quantile(0.25)
q3 = orders_clean["total_amount"].quantile(0.75)
iqr = q3 - q1

upper_bound = q3 + (1.5 * iqr)

total_amount_outliers = (
    orders_clean["total_amount"] > upper_bound
).sum()

print(
    "\nHigh-value IQR outliers retained:",
    total_amount_outliers
)


# ------------------------------------------------------------
# STEP 7: FINAL CLEANING SUMMARY
# ------------------------------------------------------------

orders_after = len(orders_clean)

rows_removed = orders_before - orders_after

print("\n\nFINAL ORDERS CLEANING SUMMARY")
print("-" * 70)

print("Rows before cleaning:", orders_before)
print("Rows after cleaning:", orders_after)
print("Total rows removed:", rows_removed)

print(
    "\nExpected cleaned row count:",
    28252
)

print(
    "Actual cleaned row count:",
    orders_after
)

if orders_after == 28252:
    print("\nPASS: Cleaned order count matches expectation.")
else:
    print("\nCHECK REQUIRED: Cleaned count differs from expectation.")


print("\nOrders cleaning and transformation complete.")
print("=" * 70)

ORDERS CLEANING AND TRANSFORMATION
Orders before cleaning: 28297

Duplicate order rows removed: 25
Rows after duplicate removal: 28272

Standardized order statuses:
order_status
Completed    25481
Returned      1230
Cancelled      976
Pending        585
Name: count, dtype: Int64

Order date dtype:
datetime64[ns]
Missing order dates retained: 424

Negative shipping costs corrected: 8
Remaining negative shipping costs: 0

Orphan customer orders removed: 20
Rows after orphan removal: 28252

High-value IQR outliers retained: 861


FINAL ORDERS CLEANING SUMMARY
----------------------------------------------------------------------
Rows before cleaning: 28297
Rows after cleaning: 28252
Total rows removed: 45

Expected cleaned row count: 28252
Actual cleaned row count: 28252

PASS: Cleaned order count matches expectation.

Orders cleaning and transformation complete.


In [ ]:
# ============================================================
# CELL 40: FINAL VALIDATION OF CLEANED ORDERS
# ============================================================

print("FINAL ORDERS VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. BASIC STRUCTURE
# ------------------------------------------------------------

print("\n1. BASIC STRUCTURE")
print("-" * 70)

print("Rows:", len(orders_clean))
print("Columns:", len(orders_clean.columns))

print("\nColumns:")
print(orders_clean.columns.tolist())

print("\nData types:")
print(orders_clean.dtypes)


# ------------------------------------------------------------
# 2. PRIMARY KEY VALIDATION
# ------------------------------------------------------------

print("\n\n2. PRIMARY KEY VALIDATION")
print("-" * 70)

duplicate_order_ids = orders_clean["order_id"].duplicated().sum()
missing_order_ids = orders_clean["order_id"].isna().sum()

print("Duplicate order IDs:", duplicate_order_ids)
print("Missing order IDs:", missing_order_ids)


# ------------------------------------------------------------
# 3. CUSTOMER FOREIGN-KEY VALIDATION
# ------------------------------------------------------------

print("\n\n3. CUSTOMER FOREIGN-KEY VALIDATION")
print("-" * 70)

valid_customer_ids = set(
    customers_clean["customer_id"].dropna()
)

orphan_orders = orders_clean[
    ~orders_clean["customer_id"].isin(valid_customer_ids)
]

print(
    "Orders linked to non-existent customers:",
    len(orphan_orders)
)


# ------------------------------------------------------------
# 4. ORDER DATE VALIDATION
# ------------------------------------------------------------

print("\n\n4. ORDER DATE VALIDATION")
print("-" * 70)

print(
    "Order date data type:",
    orders_clean["order_date"].dtype
)

print(
    "Missing order dates:",
    orders_clean["order_date"].isna().sum()
)

print(
    "Earliest valid order date:",
    orders_clean["order_date"].min()
)

print(
    "Latest valid order date:",
    orders_clean["order_date"].max()
)


# ------------------------------------------------------------
# 5. ORDER STATUS VALIDATION
# ------------------------------------------------------------

print("\n\n5. ORDER STATUS VALIDATION")
print("-" * 70)

expected_statuses = {
    "Completed",
    "Returned",
    "Cancelled",
    "Pending"
}

actual_statuses = set(
    orders_clean["order_status"]
    .dropna()
    .unique()
)

unexpected_statuses = actual_statuses - expected_statuses

print("Unique statuses:")
print(sorted(actual_statuses))

print("Unexpected statuses:")
print(unexpected_statuses)

print(
    "Missing order statuses:",
    orders_clean["order_status"].isna().sum()
)


# ------------------------------------------------------------
# 6. SHIPPING COST VALIDATION
# ------------------------------------------------------------

print("\n\n6. SHIPPING COST VALIDATION")
print("-" * 70)

print(
    "Missing shipping costs:",
    orders_clean["shipping_cost"].isna().sum()
)

print(
    "Negative shipping costs:",
    (orders_clean["shipping_cost"] < 0).sum()
)

print(
    "Minimum shipping cost:",
    orders_clean["shipping_cost"].min()
)

print(
    "Maximum shipping cost:",
    orders_clean["shipping_cost"].max()
)


# ------------------------------------------------------------
# 7. DISCOUNT VALIDATION
# ------------------------------------------------------------

print("\n\n7. DISCOUNT VALIDATION")
print("-" * 70)

invalid_discounts = orders_clean[
    (orders_clean["discount_pct"] < 0) |
    (orders_clean["discount_pct"] > 100)
]

print(
    "Missing discounts:",
    orders_clean["discount_pct"].isna().sum()
)

print(
    "Invalid discounts:",
    len(invalid_discounts)
)

print(
    "Unique discount values:",
    sorted(
        orders_clean["discount_pct"]
        .dropna()
        .unique()
        .tolist()
    )
)


# ------------------------------------------------------------
# 8. TOTAL AMOUNT VALIDATION
# ------------------------------------------------------------

print("\n\n8. TOTAL AMOUNT VALIDATION")
print("-" * 70)

print(
    "Missing total amounts:",
    orders_clean["total_amount"].isna().sum()
)

print(
    "Negative total amounts:",
    (orders_clean["total_amount"] < 0).sum()
)

print(
    "Zero total amounts:",
    (orders_clean["total_amount"] == 0).sum()
)


# ------------------------------------------------------------
# 9. EXACT DUPLICATE ROW VALIDATION
# ------------------------------------------------------------

print("\n\n9. EXACT DUPLICATE ROW VALIDATION")
print("-" * 70)

exact_duplicate_rows = orders_clean.duplicated().sum()

print(
    "Exact duplicate rows:",
    exact_duplicate_rows
)


# ------------------------------------------------------------
# 10. FINAL PASS/FAIL CHECKS
# ------------------------------------------------------------

print("\n\n10. FINAL PASS/FAIL CHECKS")
print("-" * 70)

checks = {
    "Correct final row count": len(orders_clean) == 28252,
    "No duplicate order IDs": duplicate_order_ids == 0,
    "No missing order IDs": missing_order_ids == 0,
    "No orphan customers": len(orphan_orders) == 0,
    "Order date is datetime": str(
        orders_clean["order_date"].dtype
    ) == "datetime64[ns]",
    "Valid order statuses": unexpected_statuses == set(),
    "No missing order status": (
        orders_clean["order_status"].isna().sum() == 0
    ),
    "No negative shipping costs": (
        (orders_clean["shipping_cost"] < 0).sum() == 0
    ),
    "Valid discounts": len(invalid_discounts) == 0,
    "No negative total amounts": (
        (orders_clean["total_amount"] < 0).sum() == 0
    ),
    "No exact duplicate rows": exact_duplicate_rows == 0
}

for check_name, result in checks.items():

    print(
        f"{check_name}:",
        "PASS" if result else "FAIL"
    )


overall_pass = all(checks.values())

print("\n" + "=" * 70)

if overall_pass:
    print("OVERALL ORDERS VALIDATION: PASS")
else:
    print("OVERALL ORDERS VALIDATION: CHECK REQUIRED")

print("=" * 70)

FINAL ORDERS VALIDATION

1. BASIC STRUCTURE
----------------------------------------------------------------------
Rows: 28252
Columns: 9

Columns:
['order_id', 'customer_id', 'order_date', 'channel', 'order_status', 'payment_method', 'shipping_cost', 'discount_pct', 'total_amount']

Data types:
order_id                  object
customer_id               object
order_date        datetime64[ns]
channel                   object
order_status      string[python]
payment_method            object
shipping_cost            float64
discount_pct               int64
total_amount             float64
dtype: object


2. PRIMARY KEY VALIDATION
----------------------------------------------------------------------
Duplicate order IDs: 0
Missing order IDs: 0


3. CUSTOMER FOREIGN-KEY VALIDATION
----------------------------------------------------------------------
Orders linked to non-existent customers: 0


4. ORDER DATE VALIDATION
----------------------------------------------------------------------


In [ ]:
# ============================================================
# CELL 41: ORDER ITEMS INITIAL AUDIT
# ============================================================

print("ORDER ITEMS INITIAL AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# 1. BASIC STRUCTURE
# ------------------------------------------------------------

print("\n1. BASIC STRUCTURE")
print("-" * 70)

print("Rows:", len(order_items_clean))
print("Columns:", len(order_items_clean.columns))

print("\nColumns:")
print(order_items_clean.columns.tolist())

print("\nData types:")
print(order_items_clean.dtypes)


# ------------------------------------------------------------
# 2. MISSING VALUES
# ------------------------------------------------------------

print("\n\n2. MISSING VALUES")
print("-" * 70)

missing_summary = (
    order_items_clean
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing_summary)


# ------------------------------------------------------------
# 3. PRIMARY KEY / DUPLICATE AUDIT
# ------------------------------------------------------------

print("\n\n3. DUPLICATE AUDIT")
print("-" * 70)

print(
    "Unique order_item_id:",
    order_items_clean["order_item_id"].nunique()
)

print(
    "Duplicate order_item_id records:",
    order_items_clean["order_item_id"].duplicated(
        keep=False
    ).sum()
)

print(
    "Exact duplicate rows:",
    order_items_clean.duplicated().sum()
)


# ------------------------------------------------------------
# 4. ORDER AND PRODUCT ID COVERAGE
# ------------------------------------------------------------

print("\n\n4. ORDER AND PRODUCT ID COVERAGE")
print("-" * 70)

print(
    "Unique order IDs referenced:",
    order_items_clean["order_id"].nunique()
)

print(
    "Unique product IDs referenced:",
    order_items_clean["product_id"].nunique()
)

print(
    "Missing order IDs:",
    order_items_clean["order_id"].isna().sum()
)

print(
    "Missing product IDs:",
    order_items_clean["product_id"].isna().sum()
)


# ------------------------------------------------------------
# 5. QUANTITY AUDIT
# ------------------------------------------------------------

print("\n\n5. QUANTITY AUDIT")
print("-" * 70)

print("Quantity data type:")
print(order_items_clean["quantity"].dtype)

print(
    "Missing quantities:",
    order_items_clean["quantity"].isna().sum()
)

print(
    "Zero quantities:",
    (order_items_clean["quantity"] == 0).sum()
)

print(
    "Negative quantities:",
    (order_items_clean["quantity"] < 0).sum()
)

print("\nQuantity descriptive statistics:")

print(
    order_items_clean["quantity"].describe()
)


# ------------------------------------------------------------
# 6. UNIT PRICE AT PURCHASE AUDIT
# ------------------------------------------------------------

print("\n\n6. UNIT PRICE AT PURCHASE AUDIT")
print("-" * 70)

print("Unit price data type:")
print(
    order_items_clean[
        "unit_price_at_purchase"
    ].dtype
)

print(
    "Missing purchase prices:",
    order_items_clean[
        "unit_price_at_purchase"
    ].isna().sum()
)

print(
    "Zero purchase prices:",
    (
        order_items_clean[
            "unit_price_at_purchase"
        ] == 0
    ).sum()
)

print(
    "Negative purchase prices:",
    (
        order_items_clean[
            "unit_price_at_purchase"
        ] < 0
    ).sum()
)

print("\nPurchase-price descriptive statistics:")

print(
    order_items_clean[
        "unit_price_at_purchase"
    ].describe()
)


print("\nOrder Items initial audit complete.")
print("=" * 70)

ORDER ITEMS INITIAL AUDIT

1. BASIC STRUCTURE
----------------------------------------------------------------------
Rows: 73505
Columns: 5

Columns:
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price_at_purchase']

Data types:
order_item_id              object
order_id                   object
product_id                 object
quantity                    int64
unit_price_at_purchase    float64
dtype: object


2. MISSING VALUES
----------------------------------------------------------------------
unit_price_at_purchase    1469
order_item_id                0
order_id                     0
product_id                   0
quantity                     0
dtype: int64


3. DUPLICATE AUDIT
----------------------------------------------------------------------
Unique order_item_id: 73465
Duplicate order_item_id records: 80
Exact duplicate rows: 40


4. ORDER AND PRODUCT ID COVERAGE
----------------------------------------------------------------------
Unique order IDs referenc

In [ ]:
# ============================================================
# CELL 42: INVESTIGATE DUPLICATE ORDER ITEM IDs
# ============================================================

print("DUPLICATE ORDER ITEM ID INVESTIGATION")
print("=" * 70)

# Find order_item_ids that appear more than once
duplicate_item_ids = (
    order_items_clean["order_item_id"]
    .value_counts()
)

duplicate_item_ids = duplicate_item_ids[
    duplicate_item_ids > 1
]

print("\n1. DUPLICATED ORDER ITEM IDs")
print("-" * 70)
print("Number of duplicated order_item_ids:",
      len(duplicate_item_ids))

print("\nFrequency of duplicated IDs:")
print(duplicate_item_ids.value_counts().sort_index())


# Display the duplicated records
duplicate_records = order_items_clean[
    order_items_clean["order_item_id"].isin(duplicate_item_ids.index)
].sort_values("order_item_id")

print("\n\n2. DUPLICATED RECORDS")
print("-" * 70)
print(duplicate_records.head(20))


# Check whether each duplicated pair is identical across all columns
comparison_columns = [
    "order_id",
    "product_id",
    "quantity",
    "unit_price_at_purchase"
]

duplicate_groups = (
    duplicate_records
    .groupby("order_item_id")[comparison_columns]
    .nunique(dropna=False)
)

print("\n\n3. CHECK WHETHER DUPLICATES ARE EXACT COPIES")
print("-" * 70)

# Every value should have only one unique value within each duplicated ID
non_identical_duplicates = duplicate_groups[
    (duplicate_groups > 1).any(axis=1)
]

print("Duplicated IDs with conflicting values:",
      len(non_identical_duplicates))

if len(non_identical_duplicates) == 0:
    print("RESULT: All duplicated order_item_ids are exact copies.")
else:
    print("RESULT: Some duplicated order_item_ids contain conflicting values.")
    print("\nConflicting duplicated IDs:")
    print(non_identical_duplicates)

print("\nDuplicate investigation complete.")
print("=" * 70)

DUPLICATE ORDER ITEM ID INVESTIGATION

1. DUPLICATED ORDER ITEM IDs
----------------------------------------------------------------------
Number of duplicated order_item_ids: 40

Frequency of duplicated IDs:
count
2    40
Name: count, dtype: int64


2. DUPLICATED RECORDS
----------------------------------------------------------------------
      order_item_id   order_id product_id  quantity  unit_price_at_purchase
41701      OI501045  ORD105556   PROD1058         2                  503.85
46542      OI501045  ORD105556   PROD1058         2                  503.85
17274      OI502291  ORD100604   PROD1049         2                   20.33
51376      OI502291  ORD100604   PROD1049         2                   20.33
40523      OI502812  ORD102369   PROD1213         2                  457.96
16457      OI502812  ORD102369   PROD1213         2                  457.96
71916      OI503791  ORD122023   PROD1071         2                  338.19
72663      OI503791  ORD122023   PROD1071       

In [ ]:
# ============================================================
# CELL 43: INVESTIGATE NEGATIVE ORDER ITEM QUANTITIES
# ============================================================

print("NEGATIVE ORDER ITEM QUANTITY INVESTIGATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Identify all records with negative quantities
# ------------------------------------------------------------

negative_quantity_items = order_items_clean[
    order_items_clean["quantity"] < 0
].copy()

print("\n1. NEGATIVE QUANTITY RECORDS")
print("-" * 70)
print("Number of records:", len(negative_quantity_items))

print("\nNegative quantity values:")
print(
    negative_quantity_items["quantity"]
    .value_counts()
    .sort_index()
)

print("\nNegative quantity records:")
print(
    negative_quantity_items[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price_at_purchase"
        ]
    ].sort_values("quantity")
)


# ------------------------------------------------------------
# 2. Compare negative quantity records with order status
# ------------------------------------------------------------

negative_with_status = negative_quantity_items.merge(
    orders_clean[
        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
            "channel",
            "total_amount"
        ]
    ],
    on="order_id",
    how="left"
)

print("\n\n2. ORDER STATUS OF NEGATIVE-QUANTITY ITEMS")
print("-" * 70)

print(
    negative_with_status["order_status"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 3. Show the detailed records with order status
# ------------------------------------------------------------

print("\nDetailed records with order status:")
print(
    negative_with_status[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price_at_purchase",
            "order_status",
            "order_date",
            "total_amount"
        ]
    ].sort_values(["order_status", "order_id"])
)


# ------------------------------------------------------------
# 4. Check whether negative quantities are associated
#    consistently with returned/cancelled orders
# ------------------------------------------------------------

print("\n\n3. BUSINESS INTERPRETATION CHECK")
print("-" * 70)

status_summary = (
    negative_with_status
    .groupby("order_status", dropna=False)
    .agg(
        negative_item_count=("order_item_id", "count"),
        total_negative_quantity=("quantity", "sum")
    )
    .reset_index()
)

print(status_summary)

print("\nNegative quantity investigation complete.")
print("=" * 70)

NEGATIVE ORDER ITEM QUANTITY INVESTIGATION

1. NEGATIVE QUANTITY RECORDS
----------------------------------------------------------------------
Number of records: 18

Negative quantity values:
quantity
-4    3
-3    4
-2    2
-1    9
Name: count, dtype: int64

Negative quantity records:
      order_item_id   order_id product_id  quantity  unit_price_at_purchase
4209       OI510461  ORD127669   PROD1119        -4                  391.60
56806      OI564623  ORD125447   PROD1143        -4                  387.92
55211      OI549969  ORD109797   PROD1023        -4                  258.71
5687       OI552598  ORD116292   PROD1114        -3                  782.62
61567      OI534983  ORD104772   PROD1137        -3                  422.01
54553      OI501248  ORD112306   PROD1147        -3                  108.58
22410      OI532150  ORD126741   PROD1101        -3                  177.69
48666      OI534040  ORD100650   PROD1142        -2                  310.13
62640      OI530475  ORD1141

In [ ]:
# ============================================================
# CELL 44: CHECK FINANCIAL CONSISTENCY OF NEGATIVE QUANTITIES
# ============================================================

print("NEGATIVE QUANTITY FINANCIAL CONSISTENCY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Calculate the financial contribution of each item
# ------------------------------------------------------------
# Normally:
# item_value = quantity × unit_price_at_purchase
#
# A negative quantity therefore produces a negative item value.

negative_financial_check = negative_with_status.copy()

negative_financial_check["calculated_item_value"] = (
    negative_financial_check["quantity"]
    * negative_financial_check["unit_price_at_purchase"]
)

print("\n1. NEGATIVE ITEM VALUES")
print("-" * 70)

print(
    negative_financial_check[
        [
            "order_item_id",
            "order_id",
            "quantity",
            "unit_price_at_purchase",
            "calculated_item_value",
            "order_status",
            "total_amount"
        ]
    ].sort_values("calculated_item_value")
)


# ------------------------------------------------------------
# 2. Compare negative item value with the order total
# ------------------------------------------------------------

print("\n\n2. FINANCIAL DIRECTION CHECK")
print("-" * 70)

negative_item_value_count = (
    negative_financial_check["calculated_item_value"] < 0
).sum()

positive_order_total_count = (
    negative_financial_check["total_amount"] > 0
).sum()

print("Negative item values:",
      negative_item_value_count)

print("Positive order totals:",
      positive_order_total_count)

print(
    "\nNegative item values combined:",
    round(
        negative_financial_check["calculated_item_value"].sum(),
        2
    )
)

print(
    "Positive order totals combined:",
    round(
        negative_financial_check["total_amount"].sum(),
        2
    )
)


# ------------------------------------------------------------
# 3. Check the one Returned order separately
# ------------------------------------------------------------

print("\n\n3. RETURNED ORDER CHECK")
print("-" * 70)

returned_negative = negative_financial_check[
    negative_financial_check["order_status"] == "Returned"
]

print(returned_negative[
    [
        "order_item_id",
        "order_id",
        "quantity",
        "unit_price_at_purchase",
        "calculated_item_value",
        "order_status",
        "total_amount"
    ]
])


# ------------------------------------------------------------
# 4. Check whether negative quantities occur alongside
#    other positive order items
# ------------------------------------------------------------

print("\n\n4. OTHER ITEMS IN THE AFFECTED ORDERS")
print("-" * 70)

affected_order_ids = negative_financial_check["order_id"].unique()

affected_order_items = order_items_clean[
    order_items_clean["order_id"].isin(affected_order_ids)
].merge(
    orders_clean[
        [
            "order_id",
            "order_status",
            "total_amount"
        ]
    ],
    on="order_id",
    how="left"
)

print(
    affected_order_items[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price_at_purchase",
            "order_status",
            "total_amount"
        ]
    ].sort_values(["order_id", "quantity"])
)

print("\nFinancial consistency investigation complete.")
print("=" * 70)

NEGATIVE QUANTITY FINANCIAL CONSISTENCY CHECK

1. NEGATIVE ITEM VALUES
----------------------------------------------------------------------
   order_item_id   order_id  quantity  unit_price_at_purchase  \
1       OI552598  ORD116292        -3                  782.62   
0       OI510461  ORD127669        -4                  391.60   
13      OI564623  ORD125447        -4                  387.92   
14      OI534983  ORD104772        -3                  422.01   
12      OI549969  ORD109797        -4                  258.71   
15      OI517949  ORD124656        -1                  978.01   
6       OI500334  ORD127627        -1                  725.68   
2       OI547959  ORD104939        -1                  636.24   
10      OI534040  ORD100650        -2                  310.13   
5       OI532150  ORD126741        -3                  177.69   
7       OI503467  ORD119668        -1                  484.17   
8       OI521929  ORD121945        -1                  359.69   
9       OI510

In [ ]:
# ============================================================
# CELL 45: INVESTIGATE MISSING PURCHASE PRICES
# ============================================================

print("MISSING PURCHASE PRICE INVESTIGATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Identify records with missing purchase prices
# ------------------------------------------------------------

missing_price_items = order_items_clean[
    order_items_clean["unit_price_at_purchase"].isna()
].copy()

print("\n1. MISSING PURCHASE PRICE RECORDS")
print("-" * 70)

print("Number of records:",
      len(missing_price_items))

print(
    "Percentage of order-item records:",
    round(
        len(missing_price_items) / len(order_items_clean) * 100,
        2
    ),
    "%"
)


# ------------------------------------------------------------
# 2. Check whether missing prices are associated with
#    particular products
# ------------------------------------------------------------

print("\n\n2. MISSING PRICES BY PRODUCT")
print("-" * 70)

missing_by_product = (
    missing_price_items
    .groupby("product_id")
    .size()
    .sort_values(ascending=False)
)

print(missing_by_product.head(20))

print(
    "\nUnique products affected:",
    missing_price_items["product_id"].nunique()
)


# ------------------------------------------------------------
# 3. Bring in the current product price
# ------------------------------------------------------------
# This is for investigation only.
# We are NOT assuming that current product price equals
# historical purchase price.

missing_with_product_price = missing_price_items.merge(
    products_clean[
        [
            "product_id",
            "product_name",
            "category",
            "unit_price"
        ]
    ],
    on="product_id",
    how="left"
)

print("\n\n3. CURRENT PRODUCT PRICE COMPARISON")
print("-" * 70)

print(
    missing_with_product_price[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price_at_purchase",
            "unit_price",
            "product_name",
            "category"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 4. Check whether every missing purchase-price record has
#    a corresponding product price
# ------------------------------------------------------------

print("\n\n4. PRODUCT PRICE AVAILABILITY")
print("-" * 70)

print(
    "Missing current product prices:",
    missing_with_product_price["unit_price"].isna().sum()
)

print(
    "Available current product prices:",
    missing_with_product_price["unit_price"].notna().sum()
)


# ------------------------------------------------------------
# 5. Check order status of missing-price records
# ------------------------------------------------------------

missing_with_status = missing_with_product_price.merge(
    orders_clean[
        [
            "order_id",
            "order_status",
            "order_date",
            "channel",
            "total_amount"
        ]
    ],
    on="order_id",
    how="left"
)

print("\n\n5. ORDER STATUS OF MISSING-PRICE RECORDS")
print("-" * 70)

print(
    missing_with_status["order_status"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 6. Check whether missing prices are concentrated in
#    completed, returned, cancelled or pending orders
# ------------------------------------------------------------

print("\n\n6. MISSING PRICES BY ORDER STATUS")
print("-" * 70)

status_price_summary = (
    missing_with_status
    .groupby("order_status", dropna=False)
    .agg(
        missing_price_items=("order_item_id", "count"),
        total_quantity=("quantity", "sum")
    )
    .reset_index()
)

print(status_price_summary)

print("\nMissing purchase price investigation complete.")
print("=" * 70)

MISSING PURCHASE PRICE INVESTIGATION

1. MISSING PURCHASE PRICE RECORDS
----------------------------------------------------------------------
Number of records: 1469
Percentage of order-item records: 2.0 %


2. MISSING PRICES BY PRODUCT
----------------------------------------------------------------------
product_id
PROD1199    14
PROD1063    13
PROD1079    13
PROD1110    13
PROD1061    12
PROD1036    12
PROD1034    12
PROD1200    12
PROD1179    12
PROD1078    12
PROD1031    11
PROD1133    11
PROD1131    11
PROD1107    11
PROD1148    11
PROD1207    11
PROD1171    11
PROD1120    10
PROD1194    10
PROD1081    10
dtype: int64

Unique products affected: 219


3. CURRENT PRODUCT PRICE COMPARISON
----------------------------------------------------------------------
   order_item_id   order_id product_id  quantity  unit_price_at_purchase  \
0       OI569419  ORD102651   PROD1005         1                     NaN   
1       OI531728  ORD127958   PROD1151         2                     NaN   

In [ ]:
# ============================================================
# CELL 46: COMPARE PURCHASE PRICE WITH CURRENT PRODUCT PRICE
# ============================================================

print("HISTORICAL PURCHASE PRICE VS CURRENT PRODUCT PRICE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Merge known purchase prices with product prices
# ------------------------------------------------------------
# We deliberately exclude missing purchase prices here.
# The purpose is to learn whether current product price is
# a reasonable proxy for historical purchase price.

known_price_items = order_items_clean[
    order_items_clean["unit_price_at_purchase"].notna()
].copy()

known_price_comparison = known_price_items.merge(
    products_clean[
        [
            "product_id",
            "unit_price"
        ]
    ],
    on="product_id",
    how="left"
)

# ------------------------------------------------------------
# 2. Calculate price difference
# ------------------------------------------------------------

known_price_comparison["price_difference"] = (
    known_price_comparison["unit_price_at_purchase"]
    - known_price_comparison["unit_price"]
)

known_price_comparison["absolute_price_difference"] = (
    known_price_comparison["price_difference"].abs()
)

known_price_comparison["percentage_difference"] = np.where(
    known_price_comparison["unit_price"] != 0,
    (
        known_price_comparison["price_difference"]
        / known_price_comparison["unit_price"]
    ) * 100,
    np.nan
)

print("\n1. PRICE COMPARISON SUMMARY")
print("-" * 70)

print(
    known_price_comparison[
        [
            "unit_price_at_purchase",
            "unit_price",
            "price_difference",
            "percentage_difference"
        ]
    ].describe()
)


# ------------------------------------------------------------
# 3. How often are the prices exactly equal?
# ------------------------------------------------------------

exact_match = (
    known_price_comparison["unit_price_at_purchase"]
    == known_price_comparison["unit_price"]
)

print("\n\n2. EXACT PRICE MATCH")
print("-" * 70)

print("Exact matches:", exact_match.sum())
print("Non-matches:", (~exact_match).sum())

print(
    "Exact match percentage:",
    round(exact_match.mean() * 100, 2),
    "%"
)


# ------------------------------------------------------------
# 4. Check how close the prices are
# ------------------------------------------------------------

print("\n\n3. PRICE DIFFERENCE BANDS")
print("-" * 70)

price_difference_bands = pd.cut(
    known_price_comparison["absolute_price_difference"],
    bins=[-np.inf, 0.01, 1, 10, 50, 100, np.inf],
    labels=[
        "Exactly equal",
        "Within $1",
        "$1–$10",
        "$10–$50",
        "$50–$100",
        "Over $100"
    ],
    include_lowest=True
)

print(
    price_difference_bands
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 5. Show examples where historical and current prices differ
# ------------------------------------------------------------

print("\n\n4. EXAMPLES OF PRICE DIFFERENCES")
print("-" * 70)

different_prices = known_price_comparison[
    ~exact_match
].copy()

print(
    different_prices[
        [
            "product_id",
            "unit_price_at_purchase",
            "unit_price",
            "price_difference",
            "percentage_difference"
        ]
    ]
    .sort_values("absolute_price_difference", ascending=False)
    .head(20)
)

print("\nPrice comparison investigation complete.")
print("=" * 70)

HISTORICAL PURCHASE PRICE VS CURRENT PRODUCT PRICE

1. PRICE COMPARISON SUMMARY
----------------------------------------------------------------------
       unit_price_at_purchase  unit_price  price_difference  \
count            72036.000000     72036.0           72036.0   
mean               394.540724  404.629133        -10.088409   
std                261.980754   268.55324          9.679314   
min                  8.790000        9.25            -51.33   
25%                174.677500      180.28            -15.08   
50%                356.610000       368.3             -6.98   
75%                621.000000      644.08             -2.41   
max               1027.760000     1027.96               0.0   

       percentage_difference  
count           72036.000000  
mean               -2.497184  
std                 1.443278  
min                -5.017921  
25%                -3.740544  
50%                -2.495407  
75%                -1.252501  
max                 0.000000  




KeyError: 'absolute_price_difference'

In [ ]:
# ============================================================
# CELL 47: DISPLAY PRICE DIFFERENCE EXAMPLES CORRECTLY
# ============================================================

print("CORRECTED PRICE DIFFERENCE EXAMPLES")
print("=" * 70)

# Select records where historical purchase price differs
# from the current product price.
different_prices = known_price_comparison[
    known_price_comparison["unit_price_at_purchase"]
    != known_price_comparison["unit_price"]
].copy()

print("\n1. NUMBER OF NON-MATCHING PRICE RECORDS")
print("-" * 70)
print("Non-matching records:", len(different_prices))

# Display the records with the largest absolute differences.
# We keep absolute_price_difference in the dataframe so
# that sorting works correctly.
print("\n\n2. LARGEST PRICE DIFFERENCES")
print("-" * 70)

print(
    different_prices[
        [
            "product_id",
            "unit_price_at_purchase",
            "unit_price",
            "price_difference",
            "absolute_price_difference",
            "percentage_difference"
        ]
    ]
    .sort_values(
        "absolute_price_difference",
        ascending=False
    )
    .head(20)
)

# ------------------------------------------------------------
# 3. Confirm the direction of the differences
# ------------------------------------------------------------

print("\n\n3. DIRECTION OF PRICE DIFFERENCES")
print("-" * 70)

print(
    "Historical purchase price LOWER than current price:",
    (
        different_prices["price_difference"] < 0
    ).sum()
)

print(
    "Historical purchase price HIGHER than current price:",
    (
        different_prices["price_difference"] > 0
    ).sum()
)

print(
    "Historical purchase price EXACTLY equal to current price:",
    (
        different_prices["price_difference"] == 0
    ).sum()
)

print("\nCorrected price comparison complete.")
print("=" * 70)

CORRECTED PRICE DIFFERENCE EXAMPLES

1. NUMBER OF NON-MATCHING PRICE RECORDS
----------------------------------------------------------------------
Non-matching records: 71986


2. LARGEST PRICE DIFFERENCES
----------------------------------------------------------------------
      product_id  unit_price_at_purchase  unit_price  price_difference  \
44000   PROD1120                  976.63     1027.96            -51.33   
60346   PROD1120                  976.65     1027.96            -51.31   
55854   PROD1120                  976.66     1027.96             -51.3   
62587   PROD1120                  976.90     1027.96            -51.06   
695     PROD1183                  971.32     1022.36            -51.04   
28400   PROD1183                  971.64     1022.36            -50.72   
57916   PROD1120                  977.27     1027.96            -50.69   
55439   PROD1183                  971.78     1022.36            -50.58   
27283   PROD1120                  977.47     1027.96    

In [ ]:
# ============================================================
# CELL 48: ORDER-ITEM REFERENTIAL INTEGRITY
# ============================================================
# Purpose:
# Check whether every order_item belongs to an order that
# still exists in our cleaned orders table.
#
# We already removed 20 orphan orders from orders_clean.
# Therefore, some order_items may now point to orders that
# no longer exist.
# ============================================================

print("ORDER-ITEM REFERENTIAL INTEGRITY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Identify order IDs that exist in order_items but NOT
#    in our cleaned orders table.
# ------------------------------------------------------------

valid_order_ids = set(orders_clean["order_id"])

orphan_order_items = order_items_clean[
    ~order_items_clean["order_id"].isin(valid_order_ids)
].copy()

print("\n1. ORDER ITEMS LINKED TO REMOVED/ORPHAN ORDERS")
print("-" * 70)

print(
    "Order-item records linked to invalid orders:",
    len(orphan_order_items)
)

print(
    "Unique invalid order IDs:",
    orphan_order_items["order_id"].nunique()
)

# ------------------------------------------------------------
# 2. Show the affected order IDs and number of order-items.
# ------------------------------------------------------------

print("\n\n2. AFFECTED ORDER IDs")
print("-" * 70)

if len(orphan_order_items) > 0:

    print(
        orphan_order_items
        .groupby("order_id")
        .size()
        .sort_values(ascending=False)
        .to_string()
    )

else:

    print("No orphan order-items found.")

# ------------------------------------------------------------
# 3. Check whether any order-items reference a product
#    that does not exist in products_clean.
# ------------------------------------------------------------

valid_product_ids = set(products_clean["product_id"])

orphan_product_items = order_items_clean[
    ~order_items_clean["product_id"].isin(valid_product_ids)
].copy()

print("\n\n3. ORDER ITEMS LINKED TO INVALID PRODUCTS")
print("-" * 70)

print(
    "Order-item records linked to invalid products:",
    len(orphan_product_items)
)

print(
    "Unique invalid product IDs:",
    orphan_product_items["product_id"].nunique()
)

# ------------------------------------------------------------
# 4. Final interpretation
# ------------------------------------------------------------

print("\n\n4. REFERENTIAL INTEGRITY INTERPRETATION")
print("-" * 70)

if len(orphan_order_items) == 0:
    print("✓ All order-items reference valid orders.")

else:
    print(
        "⚠ Some order-items reference orders that were removed "
        "during order cleaning."
    )

if len(orphan_product_items) == 0:
    print("✓ All order-items reference valid products.")

else:
    print(
        "⚠ Some order-items reference products that do not "
        "exist in products_clean."
    )

print("\nReferential integrity check complete.")
print("=" * 70)

ORDER-ITEM REFERENTIAL INTEGRITY CHECK

1. ORDER ITEMS LINKED TO REMOVED/ORPHAN ORDERS
----------------------------------------------------------------------
Order-item records linked to invalid orders: 60
Unique invalid order IDs: 20


2. AFFECTED ORDER IDs
----------------------------------------------------------------------
order_id
ORD113397    6
ORD109519    5
ORD112476    5
ORD105419    4
ORD124916    4
ORD126941    4
ORD127135    3
ORD104431    3
ORD116460    3
ORD115601    3
ORD109263    3
ORD107614    3
ORD110459    3
ORD106194    2
ORD102970    2
ORD104582    2
ORD115976    2
ORD101422    1
ORD111134    1
ORD109372    1


3. ORDER ITEMS LINKED TO INVALID PRODUCTS
----------------------------------------------------------------------
Order-item records linked to invalid products: 0
Unique invalid product IDs: 0


4. REFERENTIAL INTEGRITY INTERPRETATION
----------------------------------------------------------------------
⚠ Some order-items reference orders that were removed 

In [ ]:
# ============================================================
# CELL 49: CLEAN ORDER ITEMS DATA
# ============================================================
# Purpose:
# Apply the evidence-based cleaning decisions identified
# during the Order Items investigation.
#
# Cleaning decisions:
# 1. Remove exact duplicate rows.
# 2. Correct negative quantities using absolute values.
# 3. Remove order-items linked to orphan orders.
# 4. Preserve missing historical purchase prices.
# 5. Keep all valid product references.
# ============================================================

print("ORDER ITEMS CLEANING TRANSFORMATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Start from the raw working copy
# ------------------------------------------------------------

order_items_clean = order_items.copy()

initial_rows = len(order_items_clean)

print("\nInitial rows:", initial_rows)

# ------------------------------------------------------------
# 2. Remove exact duplicate rows
# ------------------------------------------------------------
# Earlier investigation showed 40 exact duplicate rows.
# We keep one copy and remove the redundant copies.

duplicate_rows_before = order_items_clean.duplicated().sum()

order_items_clean = order_items_clean.drop_duplicates().copy()

duplicate_rows_removed = (
    initial_rows - len(order_items_clean)
)

print("\n1. EXACT DUPLICATE REMOVAL")
print("-" * 70)
print("Duplicate rows detected:", duplicate_rows_before)
print("Duplicate rows removed:", duplicate_rows_removed)
print("Rows remaining:", len(order_items_clean))

# ------------------------------------------------------------
# 3. Correct negative quantities
# ------------------------------------------------------------
# Investigation showed 18 negative quantities.
#
# They produced negative line-item values even though the
# associated order totals were positive.
#
# Therefore, the negative signs are treated as data-entry/
# data-quality errors rather than legitimate returns.

negative_quantity_before = (
    order_items_clean["quantity"] < 0
).sum()

order_items_clean["quantity"] = (
    order_items_clean["quantity"].abs()
)

print("\n\n2. NEGATIVE QUANTITY CORRECTION")
print("-" * 70)
print(
    "Negative quantities before correction:",
    negative_quantity_before
)

print(
    "Negative quantities after correction:",
    (
        order_items_clean["quantity"] < 0
    ).sum()
)

# ------------------------------------------------------------
# 4. Remove order-items linked to orphan orders
# ------------------------------------------------------------
# Only retain order-items whose order_id exists in
# orders_clean.

valid_order_ids = set(orders_clean["order_id"])

orphan_order_item_mask = (
    ~order_items_clean["order_id"].isin(valid_order_ids)
)

orphan_order_items_removed = (
    orphan_order_item_mask.sum()
)

order_items_clean = (
    order_items_clean[
        ~orphan_order_item_mask
    ]
    .copy()
)

print("\n\n3. ORPHAN ORDER-ITEM REMOVAL")
print("-" * 70)
print(
    "Order-items linked to removed orders:",
    orphan_order_items_removed
)

print(
    "Rows remaining:",
    len(order_items_clean)
)

# ------------------------------------------------------------
# 5. Preserve missing historical purchase prices
# ------------------------------------------------------------
# IMPORTANT:
# We deliberately do NOT fill missing
# unit_price_at_purchase values.
#
# Current product prices were shown to be systematically
# higher than historical purchase prices.
#
# Therefore, imputing current prices could introduce bias.

missing_purchase_prices = (
    order_items_clean["unit_price_at_purchase"]
    .isna()
    .sum()
)

print("\n\n4. MISSING HISTORICAL PURCHASE PRICES")
print("-" * 70)

print(
    "Missing unit_price_at_purchase values preserved:",
    missing_purchase_prices
)

# ------------------------------------------------------------
# 6. Confirm all product IDs remain valid
# ------------------------------------------------------------

valid_product_ids = set(products_clean["product_id"])

invalid_product_links = (
    ~order_items_clean["product_id"]
    .isin(valid_product_ids)
).sum()

print("\n\n5. PRODUCT REFERENTIAL INTEGRITY")
print("-" * 70)

print(
    "Invalid product references:",
    invalid_product_links
)

# ------------------------------------------------------------
# 7. Final transformation summary
# ------------------------------------------------------------

print("\n\n6. FINAL TRANSFORMATION SUMMARY")
print("-" * 70)

print("Initial rows:", initial_rows)
print("Exact duplicate rows removed:", duplicate_rows_removed)
print("Orphan order-item rows removed:", orphan_order_items_removed)
print("Final rows:", len(order_items_clean))

print("\nOrder Items cleaning transformation complete.")
print("=" * 70)

ORDER ITEMS CLEANING TRANSFORMATION

Initial rows: 73505

1. EXACT DUPLICATE REMOVAL
----------------------------------------------------------------------
Duplicate rows detected: 40
Duplicate rows removed: 40
Rows remaining: 73465


2. NEGATIVE QUANTITY CORRECTION
----------------------------------------------------------------------
Negative quantities before correction: 18
Negative quantities after correction: 0


3. ORPHAN ORDER-ITEM REMOVAL
----------------------------------------------------------------------
Order-items linked to removed orders: 60
Rows remaining: 73405


4. MISSING HISTORICAL PURCHASE PRICES
----------------------------------------------------------------------
Missing unit_price_at_purchase values preserved: 1467


5. PRODUCT REFERENTIAL INTEGRITY
----------------------------------------------------------------------
Invalid product references: 0


6. FINAL TRANSFORMATION SUMMARY
----------------------------------------------------------------------
Initial r

In [ ]:
# ============================================================
# CELL 50: SUPPORT TICKETS INITIAL AUDIT
# ============================================================
# Purpose:
# Re-examine the main data-quality issues in the support
# tickets dataset before making cleaning decisions.
#
# We will investigate:
# 1. Missing satisfaction scores
# 2. Invalid satisfaction scores
# 3. Negative resolution times
# 4. Ticket/customer ID integrity
# 5. Duplicate records
# ============================================================

print("SUPPORT TICKETS INITIAL AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. BASIC STRUCTURE
# ------------------------------------------------------------

print("\n1. DATASET STRUCTURE")
print("-" * 70)

print("Rows:", len(support_tickets))
print("Columns:", len(support_tickets.columns))

print("\nColumns:")
print(support_tickets.columns.tolist())

print("\nData types:")
print(support_tickets.dtypes)

# ------------------------------------------------------------
# 2. MISSING VALUES
# ------------------------------------------------------------

print("\n\n2. MISSING VALUES")
print("-" * 70)

missing_support = (
    support_tickets
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing_support[missing_support > 0])

# ------------------------------------------------------------
# 3. SATISFACTION SCORE DISTRIBUTION
# ------------------------------------------------------------
# Valid satisfaction scores should normally be between 1 and 5.
# We inspect the actual values before deciding what to do.

print("\n\n3. SATISFACTION SCORE DISTRIBUTION")
print("-" * 70)

print(
    support_tickets["satisfaction_score"]
    .value_counts(dropna=False)
    .sort_index()
)

# ------------------------------------------------------------
# 4. INVALID SATISFACTION SCORES
# ------------------------------------------------------------

invalid_satisfaction = support_tickets[
    support_tickets["satisfaction_score"].notna()
    &
    (
        (support_tickets["satisfaction_score"] < 1)
        |
        (support_tickets["satisfaction_score"] > 5)
    )
].copy()

print("\n\n4. INVALID SATISFACTION SCORES")
print("-" * 70)

print(
    "Invalid satisfaction records:",
    len(invalid_satisfaction)
)

if len(invalid_satisfaction) > 0:
    print("\nInvalid values:")
    print(
        invalid_satisfaction[
            "satisfaction_score"
        ]
        .value_counts()
        .sort_index()
    )

# ------------------------------------------------------------
# 5. NEGATIVE RESOLUTION TIMES
# ------------------------------------------------------------

negative_resolution = support_tickets[
    support_tickets["resolution_time_hours"] < 0
].copy()

print("\n\n5. NEGATIVE RESOLUTION TIMES")
print("-" * 70)

print(
    "Negative resolution-time records:",
    len(negative_resolution)
)

if len(negative_resolution) > 0:

    print("\nNegative values:")
    print(
        negative_resolution[
            "resolution_time_hours"
        ]
        .value_counts()
        .sort_index()
    )

    print("\nAffected records:")
    print(
        negative_resolution[
            [
                "ticket_id",
                "customer_id",
                "ticket_date",
                "issue_category",
                "resolution_time_hours",
                "satisfaction_score"
            ]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 6. ID INTEGRITY
# ------------------------------------------------------------

print("\n\n6. ID INTEGRITY")
print("-" * 70)

print(
    "Missing ticket IDs:",
    support_tickets["ticket_id"].isna().sum()
)

print(
    "Duplicate ticket IDs:",
    support_tickets["ticket_id"].duplicated().sum()
)

print(
    "Missing customer IDs:",
    support_tickets["customer_id"].isna().sum()
)

# ------------------------------------------------------------
# 7. CUSTOMER REFERENTIAL INTEGRITY
# ------------------------------------------------------------

valid_customer_ids = set(customers_clean["customer_id"])

orphan_tickets = support_tickets[
    ~support_tickets["customer_id"].isin(valid_customer_ids)
].copy()

print("\n\n7. CUSTOMER REFERENTIAL INTEGRITY")
print("-" * 70)

print(
    "Tickets linked to invalid customers:",
    len(orphan_tickets)
)

print(
    "Unique invalid customer IDs:",
    orphan_tickets["customer_id"].nunique()
)

# ------------------------------------------------------------
# 8. EXACT DUPLICATE ROWS
# ------------------------------------------------------------

print("\n\n8. EXACT DUPLICATE ROWS")
print("-" * 70)

print(
    "Exact duplicate rows:",
    support_tickets.duplicated().sum()
)

print("\nSupport Tickets initial audit complete.")
print("=" * 70)

SUPPORT TICKETS INITIAL AUDIT

1. DATASET STRUCTURE
----------------------------------------------------------------------
Rows: 5200
Columns: 8

Columns:
['ticket_id', 'customer_id', 'ticket_date', 'issue_category', 'channel', 'resolution_time_hours', 'satisfaction_score', 'ticket_text']

Data types:
ticket_id                 object
customer_id               object
ticket_date               object
issue_category            object
channel                   object
resolution_time_hours    float64
satisfaction_score       float64
ticket_text               object
dtype: object


2. MISSING VALUES
----------------------------------------------------------------------
satisfaction_score    592
dtype: int64


3. SATISFACTION SCORE DISTRIBUTION
----------------------------------------------------------------------
satisfaction_score
1.0     271
2.0     343
3.0     756
4.0    1475
5.0    1763
NaN     592
Name: count, dtype: int64


4. INVALID SATISFACTION SCORES
-------------------------------

In [ ]:
# ============================================================
# CELL 51: INVESTIGATE NEGATIVE RESOLUTION TIMES
# ============================================================
# Purpose:
# Understand whether the six negative resolution times are
# genuine data errors and determine the safest treatment.
#
# We will compare the affected tickets with:
# - Ticket dates
# - Issue categories
# - Satisfaction scores
# - Other ticket fields
#
# We will also inspect the overall distribution of valid
# resolution times to understand the expected range.
# ============================================================

print("NEGATIVE RESOLUTION TIME INVESTIGATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. OVERALL RESOLUTION-TIME SUMMARY
# ------------------------------------------------------------

valid_resolution = support_tickets[
    support_tickets["resolution_time_hours"] >= 0
]["resolution_time_hours"]

print("\n1. VALID RESOLUTION-TIME SUMMARY")
print("-" * 70)

print(
    valid_resolution.describe()
)

# ------------------------------------------------------------
# 2. AFFECTED RECORDS
# ------------------------------------------------------------

negative_resolution = support_tickets[
    support_tickets["resolution_time_hours"] < 0
].copy()

print("\n\n2. NEGATIVE RESOLUTION-TIME RECORDS")
print("-" * 70)

print(
    negative_resolution[
        [
            "ticket_id",
            "customer_id",
            "ticket_date",
            "issue_category",
            "channel",
            "resolution_time_hours",
            "satisfaction_score"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 3. ISSUE-CATEGORY DISTRIBUTION
# ------------------------------------------------------------

print("\n\n3. ISSUE CATEGORIES OF AFFECTED TICKETS")
print("-" * 70)

print(
    negative_resolution["issue_category"]
    .value_counts()
)

# ------------------------------------------------------------
# 4. CHANNEL DISTRIBUTION
# ------------------------------------------------------------

print("\n\n4. CHANNEL DISTRIBUTION OF AFFECTED TICKETS")
print("-" * 70)

print(
    negative_resolution["channel"]
    .value_counts()
)

# ------------------------------------------------------------
# 5. SATISFACTION SCORES
# ------------------------------------------------------------

print("\n\n5. SATISFACTION SCORES OF AFFECTED TICKETS")
print("-" * 70)

print(
    negative_resolution["satisfaction_score"]
    .value_counts(dropna=False)
    .sort_index()
)

# ------------------------------------------------------------
# 6. CHECK HOW EXTREME THE NEGATIVE VALUES ARE
# ------------------------------------------------------------

print("\n\n6. NEGATIVE VALUE MAGNITUDE")
print("-" * 70)

print(
    negative_resolution[
        "resolution_time_hours"
    ].describe()
)

# ------------------------------------------------------------
# 7. DECISION SUPPORT
# ------------------------------------------------------------

print("\n\n7. DECISION SUPPORT")
print("-" * 70)

print(
    "Number of negative resolution times:",
    len(negative_resolution)
)

print(
    "Number of valid resolution times:",
    len(valid_resolution)
)

print(
    "Minimum valid resolution time:",
    valid_resolution.min()
)

print(
    "Maximum valid resolution time:",
    valid_resolution.max()
)

print("\nInvestigation complete.")
print("=" * 70)

NEGATIVE RESOLUTION TIME INVESTIGATION

1. VALID RESOLUTION-TIME SUMMARY
----------------------------------------------------------------------
count    5194.000000
mean       18.478244
std        25.012761
min         0.000000
25%         5.200000
50%        12.200000
75%        24.700000
max       782.000000
Name: resolution_time_hours, dtype: float64


2. NEGATIVE RESOLUTION-TIME RECORDS
----------------------------------------------------------------------
ticket_id customer_id ticket_date issue_category      channel  resolution_time_hours  satisfaction_score
TIX101080  CUST000769  2024-10-20  Refund Status Social Media                   -0.9                 3.0
TIX101728  CUST003749  2026-05-18  Refund Status Social Media                  -30.2                 NaN
TIX101756  CUST004987  2024-10-05   Damaged Item        Phone                   -0.8                 4.0
TIX102429  CUST004997  2026-06-09 Account Access        Phone                  -21.6                 5.0
TIX104511 

In [ ]:
# ============================================================
# CELL 52: INVESTIGATE MISSING SATISFACTION SCORES
# ============================================================
# Purpose:
# Understand the pattern of the 592 missing satisfaction scores.
#
# We will compare tickets with missing satisfaction scores
# against tickets with recorded scores using:
# - Issue category
# - Channel
# - Resolution time
#
# We do NOT want to invent customer satisfaction scores.
# The investigation helps us justify whether the missing
# values should remain missing.
# ============================================================

print("MISSING SATISFACTION SCORE INVESTIGATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. BASIC MISSINGNESS COUNTS
# ------------------------------------------------------------

missing_satisfaction = (
    support_tickets["satisfaction_score"].isna()
)

print("\n1. SATISFACTION SCORE MISSINGNESS")
print("-" * 70)

print(
    "Missing satisfaction scores:",
    missing_satisfaction.sum()
)

print(
    "Recorded satisfaction scores:",
    support_tickets["satisfaction_score"].notna().sum()
)

print(
    "Missing percentage:",
    round(
        missing_satisfaction.mean() * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# 2. ISSUE CATEGORY COMPARISON
# ------------------------------------------------------------

print("\n\n2. MISSING SATISFACTION BY ISSUE CATEGORY")
print("-" * 70)

issue_comparison = (
    support_tickets
    .assign(
        satisfaction_missing=missing_satisfaction
    )
    .groupby("issue_category")["satisfaction_missing"]
    .agg(
        tickets="count",
        missing_count="sum",
        missing_rate="mean"
    )
)

issue_comparison["missing_rate"] = (
    issue_comparison["missing_rate"] * 100
)

print(
    issue_comparison
    .sort_values(
        "missing_rate",
        ascending=False
    )
    .round(2)
)

# ------------------------------------------------------------
# 3. CHANNEL COMPARISON
# ------------------------------------------------------------

print("\n\n3. MISSING SATISFACTION BY CHANNEL")
print("-" * 70)

channel_comparison = (
    support_tickets
    .assign(
        satisfaction_missing=missing_satisfaction
    )
    .groupby("channel")["satisfaction_missing"]
    .agg(
        tickets="count",
        missing_count="sum",
        missing_rate="mean"
    )
)

channel_comparison["missing_rate"] = (
    channel_comparison["missing_rate"] * 100
)

print(
    channel_comparison
    .sort_values(
        "missing_rate",
        ascending=False
    )
    .round(2)
)

# ------------------------------------------------------------
# 4. RESOLUTION TIME COMPARISON
# ------------------------------------------------------------
# Compare resolution time for tickets where satisfaction
# was recorded versus tickets where it was missing.

print("\n\n4. RESOLUTION TIME BY SATISFACTION-MISSING STATUS")
print("-" * 70)

resolution_comparison = (
    support_tickets
    .assign(
        satisfaction_status=np.where(
            missing_satisfaction,
            "Missing",
            "Recorded"
        )
    )
    .groupby("satisfaction_status")[
        "resolution_time_hours"
    ]
    .describe()
)

print(
    resolution_comparison.round(2)
)

# ------------------------------------------------------------
# 5. CUSTOMER-LEVEL MISSINGNESS
# ------------------------------------------------------------
# Check whether missing satisfaction scores are concentrated
# among a small number of customers.

print("\n\n5. CUSTOMER-LEVEL MISSINGNESS")
print("-" * 70)

customer_missingness = (
    support_tickets[
        missing_satisfaction
    ]
    .groupby("customer_id")
    .size()
    .sort_values(ascending=False)
)

print(
    "Customers with at least one missing satisfaction score:",
    len(customer_missingness)
)

print("\nTop customers by number of missing scores:")
print(
    customer_missingness.head(10)
)

# ------------------------------------------------------------
# 6. DECISION SUPPORT
# ------------------------------------------------------------

print("\n\n6. DECISION SUPPORT")
print("-" * 70)

print(
    "Missing satisfaction scores:",
    missing_satisfaction.sum()
)

print(
    "Percentage missing:",
    round(
        missing_satisfaction.mean() * 100,
        2
    ),
    "%"
)

print(
    "\nRecommendation: investigate the pattern above before "
    "deciding how missing satisfaction scores should be handled."
)

print("\nMissing satisfaction investigation complete.")
print("=" * 70)

MISSING SATISFACTION SCORE INVESTIGATION

1. SATISFACTION SCORE MISSINGNESS
----------------------------------------------------------------------
Missing satisfaction scores: 592
Recorded satisfaction scores: 4608
Missing percentage: 11.38 %


2. MISSING SATISFACTION BY ISSUE CATEGORY
----------------------------------------------------------------------
                    tickets  missing_count  missing_rate
issue_category                                          
Return Request          641             92         14.35
Wrong Item Shipped      601             72         11.98
Damaged Item            663             78         11.76
Billing Issue           633             73         11.53
Delivery Delay          615             68         11.06
Refund Status           671             73         10.88
Product Question        715             74         10.35
Account Access          661             62          9.38


3. MISSING SATISFACTION BY CHANNEL
-----------------------------------

In [ ]:
# ============================================================
# CELL 53: SUPPORT TICKETS CATEGORICAL AND DATE AUDIT
# ============================================================
# Purpose:
# Check categorical consistency and date quality before
# creating the final support_tickets_clean table.
#
# We will inspect:
# 1. Issue categories
# 2. Support channels
# 3. Ticket dates
# 4. Date parsing problems
# 5. Future ticket dates
# ============================================================

print("SUPPORT TICKETS CATEGORICAL AND DATE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. ISSUE CATEGORY VALUES
# ------------------------------------------------------------

print("\n1. ISSUE CATEGORY VALUES")
print("-" * 70)

print(
    support_tickets["issue_category"]
    .value_counts(dropna=False)
)

# ------------------------------------------------------------
# 2. CHANNEL VALUES
# ------------------------------------------------------------

print("\n\n2. CHANNEL VALUES")
print("-" * 70)

print(
    support_tickets["channel"]
    .value_counts(dropna=False)
)

# ------------------------------------------------------------
# 3. TICKET DATE RAW SUMMARY
# ------------------------------------------------------------

print("\n\n3. TICKET DATE RAW SUMMARY")
print("-" * 70)

print(
    "Missing ticket dates:",
    support_tickets["ticket_date"].isna().sum()
)

print(
    "Unique raw ticket-date values:",
    support_tickets["ticket_date"].nunique()
)

# ------------------------------------------------------------
# 4. PARSE TICKET DATES
# ------------------------------------------------------------
# We create a temporary parsed series for investigation.
# The original dataset is not changed yet.

parsed_ticket_dates = pd.to_datetime(
    support_tickets["ticket_date"],
    errors="coerce"
)

print("\n\n4. PARSED TICKET DATE SUMMARY")
print("-" * 70)

print(
    "Successfully parsed dates:",
    parsed_ticket_dates.notna().sum()
)

print(
    "Missing/unparsed dates:",
    parsed_ticket_dates.isna().sum()
)

if parsed_ticket_dates.notna().sum() > 0:

    print(
        "Earliest ticket date:",
        parsed_ticket_dates.min()
    )

    print(
        "Latest ticket date:",
        parsed_ticket_dates.max()
    )

# ------------------------------------------------------------
# 5. FUTURE DATE CHECK
# ------------------------------------------------------------
# Use today's date only as a validation boundary.
# Ticket dates should not be in the future relative to the
# current date.

today = pd.Timestamp.today().normalize()

future_ticket_dates = (
    parsed_ticket_dates > today
)

print("\n\n5. FUTURE TICKET DATE CHECK")
print("-" * 70)

print(
    "Current validation date:",
    today.date()
)

print(
    "Future ticket dates:",
    future_ticket_dates.sum()
)

if future_ticket_dates.sum() > 0:

    print("\nFuture ticket dates:")
    print(
        support_tickets.loc[
            future_ticket_dates,
            ["ticket_id", "customer_id", "ticket_date"]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 6. SUMMARY
# ------------------------------------------------------------

print("\n\n6. AUDIT SUMMARY")
print("-" * 70)

print(
    "Issue categories:",
    support_tickets["issue_category"].nunique()
)

print(
    "Support channels:",
    support_tickets["channel"].nunique()
)

print(
    "Missing/unparsed ticket dates:",
    parsed_ticket_dates.isna().sum()
)

print(
    "Future ticket dates:",
    future_ticket_dates.sum()
)

print("\nSupport Tickets categorical/date audit complete.")
print("=" * 70)

SUPPORT TICKETS CATEGORICAL AND DATE AUDIT

1. ISSUE CATEGORY VALUES
----------------------------------------------------------------------
issue_category
Product Question      715
Refund Status         671
Damaged Item          663
Account Access        661
Return Request        641
Billing Issue         633
Delivery Delay        615
Wrong Item Shipped    601
Name: count, dtype: int64


2. CHANNEL VALUES
----------------------------------------------------------------------
channel
Phone           1343
Social Media    1304
Live Chat       1301
Email           1252
Name: count, dtype: int64


3. TICKET DATE RAW SUMMARY
----------------------------------------------------------------------
Missing ticket dates: 0
Unique raw ticket-date values: 666


4. PARSED TICKET DATE SUMMARY
----------------------------------------------------------------------
Successfully parsed dates: 5200
Missing/unparsed dates: 0
Earliest ticket date: 2024-09-03 00:00:00
Latest ticket date: 2026-06-30 00:00:00


In [ ]:
# ============================================================
# CELL 54: CLEAN SUPPORT TICKETS DATA
# ============================================================
# Purpose:
# Create the final cleaned support_tickets_clean dataset
# using the decisions supported by our investigations.
#
# Cleaning decisions:
# 1. Convert ticket_date to datetime.
# 2. Replace negative resolution times with NaN.
# 3. Preserve missing satisfaction scores.
# 4. Keep valid satisfaction scores unchanged.
# 5. Preserve all valid ticket records.
# 6. Keep customer relationships intact.
# ============================================================

print("SUPPORT TICKETS CLEANING TRANSFORMATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Create a working copy
# ------------------------------------------------------------

support_tickets_clean = support_tickets.copy()

initial_rows = len(support_tickets_clean)

print("\nInitial rows:", initial_rows)

# ------------------------------------------------------------
# 2. Convert ticket_date to datetime
# ------------------------------------------------------------

support_tickets_clean["ticket_date"] = pd.to_datetime(
    support_tickets_clean["ticket_date"],
    errors="coerce"
)

print("\n1. TICKET DATE CONVERSION")
print("-" * 70)

print(
    "Missing dates after conversion:",
    support_tickets_clean["ticket_date"].isna().sum()
)

print(
    "Date data type:",
    support_tickets_clean["ticket_date"].dtype
)

# ------------------------------------------------------------
# 3. Correct invalid negative resolution times
# ------------------------------------------------------------
# Negative resolution times have no logical business meaning.
# We retain the ticket but mark the invalid measurement as
# missing rather than using abs() or deleting the record.

negative_resolution_mask = (
    support_tickets_clean["resolution_time_hours"] < 0
)

negative_resolution_count = (
    negative_resolution_mask.sum()
)

support_tickets_clean.loc[
    negative_resolution_mask,
    "resolution_time_hours"
] = np.nan

print("\n\n2. NEGATIVE RESOLUTION TIME CORRECTION")
print("-" * 70)

print(
    "Negative resolution times corrected:",
    negative_resolution_count
)

print(
    "Negative resolution times remaining:",
    (
        support_tickets_clean[
            "resolution_time_hours"
        ] < 0
    ).sum()
)

# ------------------------------------------------------------
# 4. Preserve missing satisfaction scores
# ------------------------------------------------------------
# We deliberately do not impute satisfaction scores because
# satisfaction is a subjective customer-reported measure.

missing_satisfaction_count = (
    support_tickets_clean[
        "satisfaction_score"
    ].isna().sum()
)

print("\n\n3. SATISFACTION SCORE HANDLING")
print("-" * 70)

print(
    "Missing satisfaction scores preserved:",
    missing_satisfaction_count
)

print(
    "Invalid satisfaction scores:",
    (
        support_tickets_clean[
            "satisfaction_score"
        ].notna()
        &
        (
            (support_tickets_clean["satisfaction_score"] < 1)
            |
            (support_tickets_clean["satisfaction_score"] > 5)
        )
    ).sum()
)

# ------------------------------------------------------------
# 5. Confirm valid ticket IDs
# ------------------------------------------------------------

print("\n\n4. TICKET ID VALIDATION")
print("-" * 70)

print(
    "Missing ticket IDs:",
    support_tickets_clean["ticket_id"].isna().sum()
)

print(
    "Duplicate ticket IDs:",
    support_tickets_clean["ticket_id"].duplicated().sum()
)

# ------------------------------------------------------------
# 6. Confirm customer referential integrity
# ------------------------------------------------------------

valid_customer_ids = set(customers_clean["customer_id"])

invalid_customer_links = (
    ~support_tickets_clean["customer_id"]
    .isin(valid_customer_ids)
).sum()

print("\n\n5. CUSTOMER REFERENTIAL INTEGRITY")
print("-" * 70)

print(
    "Invalid customer references:",
    invalid_customer_links
)

# ------------------------------------------------------------
# 7. Final transformation summary
# ------------------------------------------------------------

print("\n\n6. FINAL TRANSFORMATION SUMMARY")
print("-" * 70)

print("Initial rows:", initial_rows)
print("Final rows:", len(support_tickets_clean))
print(
    "Rows removed:",
    initial_rows - len(support_tickets_clean)
)

print(
    "Missing resolution times:",
    support_tickets_clean[
        "resolution_time_hours"
    ].isna().sum()
)

print(
    "Missing satisfaction scores:",
    support_tickets_clean[
        "satisfaction_score"
    ].isna().sum()
)

print("\nSupport Tickets cleaning transformation complete.")
print("=" * 70)

SUPPORT TICKETS CLEANING TRANSFORMATION

Initial rows: 5200

1. TICKET DATE CONVERSION
----------------------------------------------------------------------
Missing dates after conversion: 0
Date data type: datetime64[ns]


2. NEGATIVE RESOLUTION TIME CORRECTION
----------------------------------------------------------------------
Negative resolution times corrected: 6
Negative resolution times remaining: 0


3. SATISFACTION SCORE HANDLING
----------------------------------------------------------------------
Missing satisfaction scores preserved: 592
Invalid satisfaction scores: 0


4. TICKET ID VALIDATION
----------------------------------------------------------------------
Missing ticket IDs: 0
Duplicate ticket IDs: 0


5. CUSTOMER REFERENTIAL INTEGRITY
----------------------------------------------------------------------
Invalid customer references: 0


6. FINAL TRANSFORMATION SUMMARY
----------------------------------------------------------------------
Initial rows: 5200
Fina

In [ ]:
# ============================================================
# CELL 55: CROSS-DATASET INTEGRITY AUDIT
# ============================================================
# Purpose:
# Verify that all five cleaned datasets work together
# correctly after the individual cleaning steps.
#
# We will check:
# 1. Customer -> Orders
# 2. Orders -> Order Items
# 3. Products -> Order Items
# 4. Customer -> Support Tickets
# 5. Primary-key uniqueness
# 6. Basic row-count reconciliation
# ============================================================

print("CROSS-DATASET INTEGRITY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. CUSTOMERS -> ORDERS
# ------------------------------------------------------------
# Every order should belong to a valid customer.

valid_customer_ids = set(
    customers_clean["customer_id"]
)

orphan_orders = orders_clean[
    ~orders_clean["customer_id"].isin(
        valid_customer_ids
    )
]

print("\n1. CUSTOMERS -> ORDERS")
print("-" * 70)

print(
    "Orders with invalid customer references:",
    len(orphan_orders)
)

# ------------------------------------------------------------
# 2. ORDERS -> ORDER ITEMS
# ------------------------------------------------------------
# Every order-item should belong to an order that exists
# in orders_clean.

valid_order_ids = set(
    orders_clean["order_id"]
)

orphan_order_items = order_items_clean[
    ~order_items_clean["order_id"].isin(
        valid_order_ids
    )
]

print("\n\n2. ORDERS -> ORDER ITEMS")
print("-" * 70)

print(
    "Order-items with invalid order references:",
    len(orphan_order_items)
)

# ------------------------------------------------------------
# 3. PRODUCTS -> ORDER ITEMS
# ------------------------------------------------------------
# Every order-item should reference a valid product.

valid_product_ids = set(
    products_clean["product_id"]
)

orphan_product_items = order_items_clean[
    ~order_items_clean["product_id"].isin(
        valid_product_ids
    )
]

print("\n\n3. PRODUCTS -> ORDER ITEMS")
print("-" * 70)

print(
    "Order-items with invalid product references:",
    len(orphan_product_items)
)

# ------------------------------------------------------------
# 4. CUSTOMERS -> SUPPORT TICKETS
# ------------------------------------------------------------
# Every support ticket should belong to a valid customer.

orphan_tickets = support_tickets_clean[
    ~support_tickets_clean["customer_id"].isin(
        valid_customer_ids
    )
]

print("\n\n4. CUSTOMERS -> SUPPORT TICKETS")
print("-" * 70)

print(
    "Support tickets with invalid customer references:",
    len(orphan_tickets)
)

# ------------------------------------------------------------
# 5. PRIMARY KEY UNIQUENESS
# ------------------------------------------------------------

print("\n\n5. PRIMARY KEY UNIQUENESS")
print("-" * 70)

print(
    "Duplicate customer IDs:",
    customers_clean["customer_id"].duplicated().sum()
)

print(
    "Duplicate product IDs:",
    products_clean["product_id"].duplicated().sum()
)

print(
    "Duplicate order IDs:",
    orders_clean["order_id"].duplicated().sum()
)

print(
    "Duplicate order-item IDs:",
    order_items_clean["order_item_id"].duplicated().sum()
)

print(
    "Duplicate ticket IDs:",
    support_tickets_clean["ticket_id"].duplicated().sum()
)

# ------------------------------------------------------------
# 6. ROW COUNT RECONCILIATION
# ------------------------------------------------------------
# These counts allow us to verify the transformations against
# our earlier cleaning decisions.

print("\n\n6. FINAL CLEANED ROW COUNTS")
print("-" * 70)

print(
    "Customers:",
    len(customers_clean)
)

print(
    "Products:",
    len(products_clean)
)

print(
    "Orders:",
    len(orders_clean)
)

print(
    "Order Items:",
    len(order_items_clean)
)

print(
    "Support Tickets:",
    len(support_tickets_clean)
)

# ------------------------------------------------------------
# 7. OVERALL INTEGRITY RESULT
# ------------------------------------------------------------

all_reference_checks_pass = (
    len(orphan_orders) == 0
    and
    len(orphan_order_items) == 0
    and
    len(orphan_product_items) == 0
    and
    len(orphan_tickets) == 0
)

all_pk_checks_pass = (
    customers_clean["customer_id"].duplicated().sum() == 0
    and
    products_clean["product_id"].duplicated().sum() == 0
    and
    orders_clean["order_id"].duplicated().sum() == 0
    and
    order_items_clean["order_item_id"].duplicated().sum() == 0
    and
    support_tickets_clean["ticket_id"].duplicated().sum() == 0
)

print("\n\n7. OVERALL INTEGRITY RESULT")
print("-" * 70)

if all_reference_checks_pass:
    print("✓ All foreign-key relationships are valid.")
else:
    print("⚠ Foreign-key integrity issues remain.")

if all_pk_checks_pass:
    print("✓ All primary keys are unique.")
else:
    print("⚠ Primary-key uniqueness issues remain.")

if all_reference_checks_pass and all_pk_checks_pass:
    print("\nOVERALL RESULT: PASS")
else:
    print("\nOVERALL RESULT: REVIEW REQUIRED")

print("\nCross-dataset integrity audit complete.")
print("=" * 70)

CROSS-DATASET INTEGRITY AUDIT

1. CUSTOMERS -> ORDERS
----------------------------------------------------------------------
Orders with invalid customer references: 0


2. ORDERS -> ORDER ITEMS
----------------------------------------------------------------------
Order-items with invalid order references: 0


3. PRODUCTS -> ORDER ITEMS
----------------------------------------------------------------------
Order-items with invalid product references: 0


4. CUSTOMERS -> SUPPORT TICKETS
----------------------------------------------------------------------
Support tickets with invalid customer references: 0


5. PRIMARY KEY UNIQUENESS
----------------------------------------------------------------------
Duplicate customer IDs: 0
Duplicate product IDs: 0
Duplicate order IDs: 0
Duplicate order-item IDs: 0
Duplicate ticket IDs: 0


6. FINAL CLEANED ROW COUNTS
----------------------------------------------------------------------
Customers: 6090
Products: 220
Orders: 28252
Order Items: 73

In [ ]:
# ============================================================
# CELL 56: OUTLIER AUDIT USING THE IQR METHOD
# ============================================================
# Purpose:
# Identify unusually high or low numerical values in the
# cleaned datasets.
#
# IMPORTANT:
# An outlier is NOT automatically a data error.
# We identify outliers first, then investigate whether they
# should be corrected, retained, or removed.
# ============================================================

print("OUTLIER AUDIT USING THE IQR METHOD")
print("=" * 70)


def iqr_outlier_summary(df, column, dataset_name):
    """
    Calculate IQR-based outlier statistics for a numeric column.

    IQR = Q3 - Q1
    Lower Bound = Q1 - 1.5 × IQR
    Upper Bound = Q3 + 1.5 × IQR

    Values outside these bounds are flagged as potential outliers.
    """

    series = df[column].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = series[
        (series < lower_bound) |
        (series > upper_bound)
    ]

    print(f"\n{dataset_name} → {column}")
    print("-" * 70)

    print(f"Valid observations: {len(series):,}")
    print(f"Q1: {q1:,.2f}")
    print(f"Q3: {q3:,.2f}")
    print(f"IQR: {iqr:,.2f}")
    print(f"Lower bound: {lower_bound:,.2f}")
    print(f"Upper bound: {upper_bound:,.2f}")
    print(f"Potential outliers: {len(outliers):,}")

    if len(series) > 0:
        print(
            f"Outlier percentage: "
            f"{(len(outliers) / len(series)) * 100:.2f}%"
        )


# ------------------------------------------------------------
# 1. ORDER TOTAL AMOUNT
# ------------------------------------------------------------

iqr_outlier_summary(
    orders_clean,
    "total_amount",
    "Orders"
)


# ------------------------------------------------------------
# 2. SHIPPING COST
# ------------------------------------------------------------

iqr_outlier_summary(
    orders_clean,
    "shipping_cost",
    "Orders"
)


# ------------------------------------------------------------
# 3. ORDER ITEM QUANTITY
# ------------------------------------------------------------

iqr_outlier_summary(
    order_items_clean,
    "quantity",
    "Order Items"
)


# ------------------------------------------------------------
# 4. HISTORICAL PURCHASE PRICE
# ------------------------------------------------------------

iqr_outlier_summary(
    order_items_clean,
    "unit_price_at_purchase",
    "Order Items"
)


# ------------------------------------------------------------
# 5. SUPPORT TICKET RESOLUTION TIME
# ------------------------------------------------------------

iqr_outlier_summary(
    support_tickets_clean,
    "resolution_time_hours",
    "Support Tickets"
)


print("\n" + "=" * 70)
print("OUTLIER AUDIT COMPLETE")
print("=" * 70)

OUTLIER AUDIT USING THE IQR METHOD

Orders → total_amount
----------------------------------------------------------------------
Valid observations: 28,252
Q1: 671.53
Q3: 2,125.78
IQR: 1,454.25
Lower bound: -1,509.83
Upper bound: 4,307.15
Potential outliers: 861
Outlier percentage: 3.05%

Orders → shipping_cost
----------------------------------------------------------------------
Valid observations: 28,252
Q1: 0.00
Q3: 7.99
IQR: 7.99
Lower bound: -11.98
Upper bound: 19.98
Potential outliers: 0
Outlier percentage: 0.00%

Order Items → quantity
----------------------------------------------------------------------
Valid observations: 73,405
Q1: 1.00
Q3: 2.00
IQR: 1.00
Lower bound: -0.50
Upper bound: 3.50
Potential outliers: 3,749
Outlier percentage: 5.11%

Order Items → unit_price_at_purchase
----------------------------------------------------------------------
Valid observations: 71,938
Q1: 174.68
Q3: 621.18
IQR: 446.50
Lower bound: -495.06
Upper bound: 1,290.93
Potential outliers: 0


In [ ]:
# ============================================================
# CELL 57: FINAL DATA TYPE & FORMAT VALIDATION
# ============================================================
# Purpose:
# Confirm that the cleaned datasets use appropriate data types
# before they are saved for EDA and machine learning.
#
# Correct data types are important because they affect:
# - calculations
# - grouping
# - plotting
# - statistical analysis
# - machine-learning preprocessing
# ============================================================

print("FINAL DATA TYPE & FORMAT VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. DISPLAY DATA TYPES
# ------------------------------------------------------------

print("\n1. CUSTOMERS DATA TYPES")
print("-" * 70)
print(customers_clean.dtypes)


print("\n\n2. PRODUCTS DATA TYPES")
print("-" * 70)
print(products_clean.dtypes)


print("\n\n3. ORDERS DATA TYPES")
print("-" * 70)
print(orders_clean.dtypes)


print("\n\n4. ORDER ITEMS DATA TYPES")
print("-" * 70)
print(order_items_clean.dtypes)


print("\n\n5. SUPPORT TICKETS DATA TYPES")
print("-" * 70)
print(support_tickets_clean.dtypes)


# ------------------------------------------------------------
# 2. CHECK IMPORTANT NUMERIC COLUMNS
# ------------------------------------------------------------

numeric_columns = {
    "orders": [
        "shipping_cost",
        "discount_pct",
        "total_amount"
    ],
    "order_items": [
        "quantity",
        "unit_price_at_purchase"
    ],
    "support_tickets": [
        "resolution_time_hours",
        "satisfaction_score"
    ]
}

print("\n\n6. NUMERIC COLUMN VALIDATION")
print("-" * 70)

for dataset_name, columns in numeric_columns.items():

    print(f"\n{dataset_name.upper()}")

    if dataset_name == "orders":
        df = orders_clean
    elif dataset_name == "order_items":
        df = order_items_clean
    else:
        df = support_tickets_clean

    for column in columns:

        print(
            f"{column}: "
            f"{df[column].dtype}"
        )


# ------------------------------------------------------------
# 3. CHECK DATE COLUMNS
# ------------------------------------------------------------

date_columns = {
    "customers": ["date_of_birth", "signup_date"],
    "products": ["launch_date"],
    "orders": ["order_date"],
    "support_tickets": ["ticket_date"]
}

print("\n\n7. DATE COLUMN VALIDATION")
print("-" * 70)

for dataset_name, columns in date_columns.items():

    print(f"\n{dataset_name.upper()}")

    if dataset_name == "customers":
        df = customers_clean
    elif dataset_name == "products":
        df = products_clean
    elif dataset_name == "orders":
        df = orders_clean
    else:
        df = support_tickets_clean

    for column in columns:

        is_datetime = pd.api.types.is_datetime64_any_dtype(
            df[column]
        )

        print(
            f"{column}: "
            f"{df[column].dtype} | "
            f"Datetime: {is_datetime}"
        )


# ------------------------------------------------------------
# 4. CHECK KEY COLUMNS FOR MISSING VALUES
# ------------------------------------------------------------

key_columns = {
    "customers": [
        "customer_id",
        "first_name",
        "last_name",
        "signup_date"
    ],
    "products": [
        "product_id",
        "product_name",
        "category"
    ],
    "orders": [
        "order_id",
        "customer_id"
    ],
    "order_items": [
        "order_item_id",
        "order_id",
        "product_id"
    ],
    "support_tickets": [
        "ticket_id",
        "customer_id"
    ]
}

print("\n\n8. KEY COLUMN MISSING-VALUE VALIDATION")
print("-" * 70)

for dataset_name, columns in key_columns.items():

    print(f"\n{dataset_name.upper()}")

    if dataset_name == "customers":
        df = customers_clean
    elif dataset_name == "products":
        df = products_clean
    elif dataset_name == "orders":
        df = orders_clean
    elif dataset_name == "order_items":
        df = order_items_clean
    else:
        df = support_tickets_clean

    for column in columns:

        missing = df[column].isna().sum()

        print(
            f"{column}: "
            f"{missing} missing"
        )


print("\n" + "=" * 70)
print("DATA TYPE & FORMAT VALIDATION COMPLETE")
print("=" * 70)

FINAL DATA TYPE & FORMAT VALIDATION

1. CUSTOMERS DATA TYPES
----------------------------------------------------------------------
customer_id                  object
first_name                   object
last_name                    object
email                string[python]
phone                string[python]
gender                       object
date_of_birth        datetime64[ns]
signup_date          datetime64[ns]
state                        object
city                         object
membership_tier      string[python]
preferred_channel            object
marketing_opt_in             object
referral_source              object
dtype: object


2. PRODUCTS DATA TYPES
----------------------------------------------------------------------
product_id              object
product_name            object
category                object
sub_category            object
brand                   object
unit_cost              float64
unit_price             Float64
launch_date     datetime64[ns]
is_act

In [ ]:
# ============================================================
# CELL 58: CLEANING IMPACT AUDIT — CORRECTED
# ============================================================
# Purpose:
# Compare the original raw datasets with the final cleaned
# datasets and quantify the impact of our cleaning decisions.
#
# We will measure:
# 1. Raw vs cleaned row counts
# 2. Rows removed
# 3. Exact duplicates
# 4. Missing cells
# 5. Important cleaning corrections
# ============================================================

print("CLEANING IMPACT AUDIT")
print("=" * 80)


# ------------------------------------------------------------
# 1. RAW VS CLEANED ROW COUNTS
# ------------------------------------------------------------
# NOTE:
# The raw DataFrames are the variables created when the raw
# CSV files were originally loaded in this notebook.
# ------------------------------------------------------------

datasets_comparison = [
    ("Customers", customers, customers_clean),
    ("Products", products, products_clean),
    ("Orders", orders, orders_clean),
    ("Order Items", order_items, order_items_clean),
    ("Support Tickets", support_tickets, support_tickets_clean)
]


print("\n1. RAW VS CLEANED ROW COUNTS")
print("-" * 80)

for name, raw_df, clean_df in datasets_comparison:

    raw_rows = len(raw_df)
    clean_rows = len(clean_df)

    removed = raw_rows - clean_rows

    retained_pct = (
        clean_rows / raw_rows * 100
        if raw_rows > 0
        else 0
    )

    print(f"\n{name}")
    print(f"Raw rows:       {raw_rows:,}")
    print(f"Clean rows:     {clean_rows:,}")
    print(f"Rows removed:   {removed:,}")
    print(f"Rows retained:  {retained_pct:.2f}%")


# ------------------------------------------------------------
# 2. EXACT DUPLICATE COMPARISON
# ------------------------------------------------------------

print("\n\n2. EXACT DUPLICATE COMPARISON")
print("-" * 80)

for name, raw_df, clean_df in datasets_comparison:

    raw_duplicates = raw_df.duplicated().sum()
    clean_duplicates = clean_df.duplicated().sum()

    print(
        f"{name}: "
        f"raw duplicates = {raw_duplicates:,} | "
        f"clean duplicates = {clean_duplicates:,}"
    )


# ------------------------------------------------------------
# 3. MISSING-VALUE COMPARISON
# ------------------------------------------------------------

print("\n\n3. MISSING-VALUE COMPARISON")
print("-" * 80)

for name, raw_df, clean_df in datasets_comparison:

    raw_missing = raw_df.isna().sum().sum()
    clean_missing = clean_df.isna().sum().sum()

    print(f"\n{name}")
    print(f"Total missing cells - raw:   {raw_missing:,}")
    print(f"Total missing cells - clean: {clean_missing:,}")
    print(
        f"Change in missing cells:     "
        f"{clean_missing - raw_missing:+,}"
    )


# ------------------------------------------------------------
# 4. IMPORTANT DATASET-SPECIFIC CLEANING EFFECTS
# ------------------------------------------------------------

print("\n\n4. IMPORTANT DATASET-SPECIFIC CLEANING EFFECTS")
print("-" * 80)


# ------------------------------------------------------------
# CUSTOMERS
# ------------------------------------------------------------

raw_dob = pd.to_datetime(
    customers["date_of_birth"],
    errors="coerce"
)

raw_future_dob = (
    raw_dob > pd.Timestamp.today()
).sum()

clean_missing_dob = (
    customers_clean["date_of_birth"].isna().sum()
)

print("\nCUSTOMERS")
print(f"Future DOB records identified: {raw_future_dob}")
print(
    f"Missing/invalid DOB after cleaning: "
    f"{clean_missing_dob}"
)


# ------------------------------------------------------------
# PRODUCTS
# ------------------------------------------------------------

raw_product_price_missing = (
    products["unit_price"].isna().sum()
)

clean_product_price_missing = (
    products_clean["unit_price"].isna().sum()
)

print("\nPRODUCTS")
print(
    f"Missing unit_price - raw:   "
    f"{raw_product_price_missing:,}"
)
print(
    f"Missing unit_price - clean: "
    f"{clean_product_price_missing:,}"
)

print(
    "Note: currency-formatted price values were "
    "converted to numeric."
)


# ------------------------------------------------------------
# ORDERS
# ------------------------------------------------------------

raw_negative_shipping = (
    pd.to_numeric(
        orders["shipping_cost"],
        errors="coerce"
    ) < 0
).sum()

clean_negative_shipping = (
    orders_clean["shipping_cost"] < 0
).sum()

raw_duplicate_orders = (
    orders["order_id"].duplicated().sum()
)

clean_duplicate_orders = (
    orders_clean["order_id"].duplicated().sum()
)

print("\nORDERS")
print(
    f"Negative shipping - raw:   "
    f"{raw_negative_shipping}"
)
print(
    f"Negative shipping - clean: "
    f"{clean_negative_shipping}"
)
print(
    f"Duplicate order IDs - raw:   "
    f"{raw_duplicate_orders}"
)
print(
    f"Duplicate order IDs - clean: "
    f"{clean_duplicate_orders}"
)


# ------------------------------------------------------------
# ORDER ITEMS
# ------------------------------------------------------------

raw_negative_quantity = (
    pd.to_numeric(
        order_items["quantity"],
        errors="coerce"
    ) < 0
).sum()

clean_negative_quantity = (
    order_items_clean["quantity"] < 0
).sum()

raw_duplicate_item_ids = (
    order_items["order_item_id"].duplicated().sum()
)

clean_duplicate_item_ids = (
    order_items_clean["order_item_id"].duplicated().sum()
)

print("\nORDER ITEMS")
print(
    f"Negative quantities - raw:   "
    f"{raw_negative_quantity}"
)
print(
    f"Negative quantities - clean: "
    f"{clean_negative_quantity}"
)
print(
    f"Duplicate item IDs - raw:    "
    f"{raw_duplicate_item_ids}"
)
print(
    f"Duplicate item IDs - clean:  "
    f"{clean_duplicate_item_ids}"
)


# ------------------------------------------------------------
# SUPPORT TICKETS
# ------------------------------------------------------------

raw_negative_resolution = (
    pd.to_numeric(
        support_tickets["resolution_time_hours"],
        errors="coerce"
    ) < 0
).sum()

clean_negative_resolution = (
    support_tickets_clean["resolution_time_hours"] < 0
).sum()

print("\nSUPPORT TICKETS")
print(
    f"Negative resolution times - raw:   "
    f"{raw_negative_resolution}"
)
print(
    f"Negative resolution times - clean: "
    f"{clean_negative_resolution}"
)


# ------------------------------------------------------------
# 5. FINAL CLEANING SUMMARY
# ------------------------------------------------------------

print("\n\n5. FINAL CLEANING SUMMARY")
print("-" * 80)

print("Customers:       ", len(customers_clean))
print("Products:        ", len(products_clean))
print("Orders:          ", len(orders_clean))
print("Order Items:     ", len(order_items_clean))
print("Support Tickets: ", len(support_tickets_clean))

print("\nCleaning impact audit complete.")
print("=" * 80)

CLEANING IMPACT AUDIT

1. RAW VS CLEANED ROW COUNTS
--------------------------------------------------------------------------------

Customers
Raw rows:       6,090
Clean rows:     6,090
Rows removed:   0
Rows retained:  100.00%

Products
Raw rows:       220
Clean rows:     220
Rows removed:   0
Rows retained:  100.00%

Orders
Raw rows:       28,297
Clean rows:     28,252
Rows removed:   45
Rows retained:  99.84%

Order Items
Raw rows:       73,505
Clean rows:     73,405
Rows removed:   100
Rows retained:  99.86%

Support Tickets
Raw rows:       5,200
Clean rows:     5,200
Rows removed:   0
Rows retained:  100.00%


2. EXACT DUPLICATE COMPARISON
--------------------------------------------------------------------------------
Customers: raw duplicates = 0 | clean duplicates = 0
Products: raw duplicates = 0 | clean duplicates = 0
Orders: raw duplicates = 25 | clean duplicates = 0
Order Items: raw duplicates = 40 | clean duplicates = 0
Support Tickets: raw duplicates = 0 | clean duplicat

In [ ]:
# ============================================================
# CELL 59: FINAL CLEANING DECISION SUMMARY
# ============================================================
# Purpose:
# Create a concise record of the major data-cleaning actions,
# the evidence behind them, and their final status.
#
# This table will be useful later for:
# - Phase 6 documentation
# - Technical report
# - README
# - Presentation
# - Viva/Q&A preparation
# ============================================================

cleaning_summary = pd.DataFrame([
    {
        "Dataset": "Customers",
        "Issue": "Invalid/future and placeholder dates",
        "Action": "Converted invalid DOB values to missing",
        "Result": "15 invalid DOB values handled",
        "Status": "Resolved"
    },
    {
        "Dataset": "Customers",
        "Issue": "Categorical inconsistencies",
        "Action": "Standardized membership tiers and states",
        "Result": "Standardized categorical values",
        "Status": "Resolved"
    },
    {
        "Dataset": "Customers",
        "Issue": "Duplicate phone/name combinations",
        "Action": "Investigated but did not merge automatically",
        "Result": "Entity-resolution uncertainty documented",
        "Status": "Retained with limitation"
    },
    {
        "Dataset": "Products",
        "Issue": "Currency-formatted unit prices",
        "Action": "Converted price values to numeric",
        "Result": "Numeric price field created",
        "Status": "Resolved"
    },
    {
        "Dataset": "Products",
        "Issue": "Missing sub-category",
        "Action": "Preserved missing values",
        "Result": "11 missing values retained",
        "Status": "Retained"
    },
    {
        "Dataset": "Products",
        "Issue": "Cost greater than selling price",
        "Action": "Investigated and retained",
        "Result": "6 systematic cases retained pending authoritative source",
        "Status": "Retained with limitation"
    },
    {
        "Dataset": "Orders",
        "Issue": "Duplicate order records",
        "Action": "Removed exact duplicate rows",
        "Result": "25 duplicate order IDs removed",
        "Status": "Resolved"
    },
    {
        "Dataset": "Orders",
        "Issue": "Negative shipping costs",
        "Action": "Converted negative values to absolute values",
        "Result": "8 negative values corrected",
        "Status": "Resolved"
    },
    {
        "Dataset": "Orders",
        "Issue": "Orphan customer references",
        "Action": "Removed orders referencing nonexistent customers",
        "Result": "20 orphan orders removed",
        "Status": "Resolved"
    },
    {
        "Dataset": "Orders",
        "Issue": "Missing order dates",
        "Action": "Preserved missing dates",
        "Result": "423 missing order dates remain",
        "Status": "Retained"
    },
    {
        "Dataset": "Order Items",
        "Issue": "Duplicate order-item records",
        "Action": "Removed exact duplicate rows",
        "Result": "40 duplicate IDs removed",
        "Status": "Resolved"
    },
    {
        "Dataset": "Order Items",
        "Issue": "Negative quantities",
        "Action": "Converted negative quantities to absolute values",
        "Result": "18 negative quantities corrected",
        "Status": "Resolved"
    },
    {
        "Dataset": "Order Items",
        "Issue": "Orphan order references",
        "Action": "Removed records linked to removed orders",
        "Result": "60 orphan order-item records removed",
        "Status": "Resolved"
    },
    {
        "Dataset": "Order Items",
        "Issue": "Missing historical purchase prices",
        "Action": "Preserved missing values; did not impute from current price",
        "Result": "1,467 missing historical prices remain",
        "Status": "Retained with limitation"
    },
    {
        "Dataset": "Support Tickets",
        "Issue": "Negative resolution times",
        "Action": "Converted invalid negative values to missing",
        "Result": "6 invalid values handled",
        "Status": "Resolved"
    },
    {
        "Dataset": "Support Tickets",
        "Issue": "Missing satisfaction scores",
        "Action": "Preserved missing subjective scores",
        "Result": "592 missing satisfaction scores remain",
        "Status": "Retained"
    },
    {
        "Dataset": "All Datasets",
        "Issue": "Referential integrity",
        "Action": "Validated all cross-dataset relationships",
        "Result": "0 orphan references remain",
        "Status": "PASS"
    },
    {
        "Dataset": "All Datasets",
        "Issue": "Primary-key uniqueness",
        "Action": "Validated all primary identifiers",
        "Result": "0 duplicate primary keys remain",
        "Status": "PASS"
    },
    {
        "Dataset": "All Datasets",
        "Issue": "Statistical outliers",
        "Action": "Assessed using IQR; legitimate extremes retained",
        "Result": "Outliers identified and not automatically deleted",
        "Status": "Reviewed"
    }
])


print("FINAL CLEANING DECISION SUMMARY")
print("=" * 100)

display(cleaning_summary)

print("\nTotal documented cleaning decisions:", len(cleaning_summary))

print("\nStatus counts:")
print(
    cleaning_summary["Status"]
    .value_counts()
)

print("\n" + "=" * 100)
print("FINAL CLEANING DECISION SUMMARY COMPLETE")
print("=" * 100)

FINAL CLEANING DECISION SUMMARY


,Dataset,Issue,Action,Result,Status
0,Customers,Invalid/future and placeholder dates,Converted invalid DOB values to missing,15 invalid DOB values handled,Resolved
1,Customers,Categorical inconsistencies,Standardized membership tiers and states,Standardized categorical values,Resolved
2,Customers,Duplicate phone/name combinations,Investigated but did not merge automatically,Entity-resolution uncertainty documented,Retained with limitation
3,Products,Currency-formatted unit prices,Converted price values to numeric,Numeric price field created,Resolved
4,Products,Missing sub-category,Preserved missing values,11 missing values retained,Retained
5,Products,Cost greater than selling price,Investigated and retained,6 systematic cases retained pending authoritat...,Retained with limitation
6,Orders,Duplicate order records,Removed exact duplicate rows,25 duplicate order IDs removed,Resolved
7,Orders,Negative shipping costs,Converted negative values to absolute values,8 negative values corrected,Resolved
8,Orders,Orphan customer references,Removed orders referencing nonexistent customers,20 orphan orders removed,Resolved
9,Orders,Missing order dates,Preserved missing dates,423 missing order dates remain,Retained



Total documented cleaning decisions: 19

Status counts:
Status
Resolved                    10
Retained with limitation     3
Retained                     3
PASS                         2
Reviewed                     1
Name: count, dtype: int64

FINAL CLEANING DECISION SUMMARY COMPLETE


In [ ]:
# ============================================================
# CELL 60: CREATE PROCESSED DATA DIRECTORY — CORRECTED
# ============================================================
# Purpose:
# Create the folder where the final cleaned datasets will be
# stored for the remaining phases of the project.
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# Convert the existing PROJECT_PATH string into a Path object
# ------------------------------------------------------------

PROJECT_PATH = Path(PROJECT_PATH)


# ------------------------------------------------------------
# Define the processed-data directory
# ------------------------------------------------------------

PROCESSED_PATH = PROJECT_PATH / "data" / "processed"


# ------------------------------------------------------------
# Create the directory if it does not already exist
# ------------------------------------------------------------

PROCESSED_PATH.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Confirm the directory
# ------------------------------------------------------------

print("PROCESSED DATA DIRECTORY")
print("=" * 70)

print("Project path:")
print(PROJECT_PATH)

print("\nProcessed-data path:")
print(PROCESSED_PATH)

print("\nDirectory exists:", PROCESSED_PATH.exists())
print("Is directory:", PROCESSED_PATH.is_dir())

print("\nProcessed-data directory is ready.")
print("=" * 70)

PROCESSED DATA DIRECTORY
Project path:
/content/drive/MyDrive/PORA Capstone

Processed-data path:
/content/drive/MyDrive/PORA Capstone/data/processed

Directory exists: True
Is directory: True

Processed-data directory is ready.


In [ ]:
# ============================================================
# CELL 61: SAVE FINAL CLEANED DATASETS
# ============================================================
# Purpose:
# Save the five final cleaned datasets as CSV files in the
# project's data/processed directory.
#
# These files will become the main input datasets for:
# - EDA
# - Visualization
# - RFM analysis
# - Churn analysis
# - Machine learning
# ============================================================


# ------------------------------------------------------------
# 1. Define output file paths
# ------------------------------------------------------------

output_files = {
    "customers_clean": PROCESSED_PATH / "customers_clean.csv",
    "products_clean": PROCESSED_PATH / "products_clean.csv",
    "orders_clean": PROCESSED_PATH / "orders_clean.csv",
    "order_items_clean": PROCESSED_PATH / "order_items_clean.csv",
    "support_tickets_clean": PROCESSED_PATH / "support_tickets_clean.csv"
}


# ------------------------------------------------------------
# 2. Save each cleaned DataFrame
# ------------------------------------------------------------

customers_clean.to_csv(
    output_files["customers_clean"],
    index=False
)

products_clean.to_csv(
    output_files["products_clean"],
    index=False
)

orders_clean.to_csv(
    output_files["orders_clean"],
    index=False
)

order_items_clean.to_csv(
    output_files["order_items_clean"],
    index=False
)

support_tickets_clean.to_csv(
    output_files["support_tickets_clean"],
    index=False
)


# ------------------------------------------------------------
# 3. Verify that the files were created
# ------------------------------------------------------------

print("FINAL CLEANED DATASETS SAVED")
print("=" * 80)

for name, path in output_files.items():

    print(
        f"{name}: "
        f"{'✓ SAVED' if path.exists() else '✗ NOT FOUND'}"
    )
    print(f"  Path: {path}")
    print(f"  Size: {path.stat().st_size:,} bytes")


# ------------------------------------------------------------
# 4. Confirm row counts
# ------------------------------------------------------------

print("\nFINAL SAVED ROW COUNTS")
print("-" * 80)

for name, path in output_files.items():

    saved_df = pd.read_csv(path)

    print(
        f"{name}: "
        f"{len(saved_df):,} rows × "
        f"{len(saved_df.columns)} columns"
    )


print("\n" + "=" * 80)
print("ALL CLEANED DATASETS SAVED AND VERIFIED")
print("=" * 80)

FINAL CLEANED DATASETS SAVED
customers_clean: ✓ SAVED
  Path: /content/drive/MyDrive/PORA Capstone/data/processed/customers_clean.csv
  Size: 868,591 bytes
products_clean: ✓ SAVED
  Path: /content/drive/MyDrive/PORA Capstone/data/processed/products_clean.csv
  Size: 21,166 bytes
orders_clean: ✓ SAVED
  Path: /content/drive/MyDrive/PORA Capstone/data/processed/orders_clean.csv
  Size: 2,142,275 bytes
order_items_clean: ✓ SAVED
  Path: /content/drive/MyDrive/PORA Capstone/data/processed/order_items_clean.csv
  Size: 2,689,100 bytes
support_tickets_clean: ✓ SAVED
  Path: /content/drive/MyDrive/PORA Capstone/data/processed/support_tickets_clean.csv
  Size: 659,125 bytes

FINAL SAVED ROW COUNTS
--------------------------------------------------------------------------------
customers_clean: 6,090 rows × 14 columns
products_clean: 220 rows × 9 columns
orders_clean: 28,252 rows × 9 columns
order_items_clean: 73,405 rows × 5 columns
support_tickets_clean: 5,200 rows × 8 columns

ALL CLEANED DA

In [ ]:
# ============================================================
# CELL 62: FINAL PROCESSED-FILE VALIDATION
# ============================================================
# Purpose:
# Reload the saved processed CSV files and confirm that:
# 1. All five files can be read successfully.
# 2. Expected row counts are preserved.
# 3. Expected column counts are preserved.
# 4. Key identifiers contain no missing values.
# 5. Relationships between the reloaded datasets remain valid.
#
# This is the final quality gate before moving to EDA.
# ============================================================


print("FINAL PROCESSED-FILE VALIDATION")
print("=" * 80)


# ------------------------------------------------------------
# 1. RELOAD THE SAVED PROCESSED FILES
# ------------------------------------------------------------

customers_processed = pd.read_csv(
    output_files["customers_clean"]
)

products_processed = pd.read_csv(
    output_files["products_clean"]
)

orders_processed = pd.read_csv(
    output_files["orders_clean"]
)

order_items_processed = pd.read_csv(
    output_files["order_items_clean"]
)

support_tickets_processed = pd.read_csv(
    output_files["support_tickets_clean"]
)


print("\n1. FILE RELOAD STATUS")
print("-" * 80)

print(
    "Customers:",
    customers_processed.shape
)

print(
    "Products:",
    products_processed.shape
)

print(
    "Orders:",
    orders_processed.shape
)

print(
    "Order Items:",
    order_items_processed.shape
)

print(
    "Support Tickets:",
    support_tickets_processed.shape
)


# ------------------------------------------------------------
# 2. EXPECTED ROW/ COLUMN COUNTS
# ------------------------------------------------------------

expected_shapes = {
    "Customers": (6090, 14),
    "Products": (220, 9),
    "Orders": (28252, 9),
    "Order Items": (73405, 5),
    "Support Tickets": (5200, 8)
}

actual_shapes = {
    "Customers": customers_processed.shape,
    "Products": products_processed.shape,
    "Orders": orders_processed.shape,
    "Order Items": order_items_processed.shape,
    "Support Tickets": support_tickets_processed.shape
}


print("\n\n2. SHAPE VALIDATION")
print("-" * 80)

shape_checks = []

for name in expected_shapes:

    passed = (
        actual_shapes[name]
        == expected_shapes[name]
    )

    shape_checks.append(passed)

    print(
        f"{name}: "
        f"{'PASS' if passed else 'FAIL'} | "
        f"Expected {expected_shapes[name]} | "
        f"Actual {actual_shapes[name]}"
    )


# ------------------------------------------------------------
# 3. KEY COLUMN MISSING-VALUE CHECK
# ------------------------------------------------------------

print("\n\n3. KEY IDENTIFIER VALIDATION")
print("-" * 80)

key_checks = {
    "Customers customer_id":
        customers_processed["customer_id"].isna().sum(),

    "Products product_id":
        products_processed["product_id"].isna().sum(),

    "Orders order_id":
        orders_processed["order_id"].isna().sum(),

    "Orders customer_id":
        orders_processed["customer_id"].isna().sum(),

    "Order Items order_item_id":
        order_items_processed["order_item_id"].isna().sum(),

    "Order Items order_id":
        order_items_processed["order_id"].isna().sum(),

    "Order Items product_id":
        order_items_processed["product_id"].isna().sum(),

    "Support Tickets ticket_id":
        support_tickets_processed["ticket_id"].isna().sum(),

    "Support Tickets customer_id":
        support_tickets_processed["customer_id"].isna().sum()
}

key_missing_checks = []

for name, missing_count in key_checks.items():

    passed = missing_count == 0
    key_missing_checks.append(passed)

    print(
        f"{name}: "
        f"{missing_count} missing | "
        f"{'PASS' if passed else 'FAIL'}"
    )


# ------------------------------------------------------------
# 4. PRIMARY-KEY UNIQUENESS
# ------------------------------------------------------------

print("\n\n4. PRIMARY-KEY UNIQUENESS")
print("-" * 80)

pk_checks = {
    "Customers":
        customers_processed["customer_id"].duplicated().sum(),

    "Products":
        products_processed["product_id"].duplicated().sum(),

    "Orders":
        orders_processed["order_id"].duplicated().sum(),

    "Order Items":
        order_items_processed["order_item_id"].duplicated().sum(),

    "Support Tickets":
        support_tickets_processed["ticket_id"].duplicated().sum()
}

pk_unique_checks = []

for name, duplicate_count in pk_checks.items():

    passed = duplicate_count == 0
    pk_unique_checks.append(passed)

    print(
        f"{name}: "
        f"{duplicate_count} duplicate IDs | "
        f"{'PASS' if passed else 'FAIL'}"
    )


# ------------------------------------------------------------
# 5. FOREIGN-KEY RELATIONSHIP VALIDATION
# ------------------------------------------------------------

print("\n\n5. FOREIGN-KEY RELATIONSHIP VALIDATION")
print("-" * 80)

customer_ids = set(
    customers_processed["customer_id"]
)

product_ids = set(
    products_processed["product_id"]
)

order_ids = set(
    orders_processed["order_id"]
)


orphan_orders = orders_processed[
    ~orders_processed["customer_id"].isin(customer_ids)
]

orphan_order_items_orders = order_items_processed[
    ~order_items_processed["order_id"].isin(order_ids)
]

orphan_order_items_products = order_items_processed[
    ~order_items_processed["product_id"].isin(product_ids)
]

orphan_tickets = support_tickets_processed[
    ~support_tickets_processed["customer_id"].isin(customer_ids)
]


fk_checks = [
    len(orphan_orders) == 0,
    len(orphan_order_items_orders) == 0,
    len(orphan_order_items_products) == 0,
    len(orphan_tickets) == 0
]


print(
    "Orders → Customers:",
    len(orphan_orders),
    "orphans |",
    "PASS" if len(orphan_orders) == 0 else "FAIL"
)

print(
    "Order Items → Orders:",
    len(orphan_order_items_orders),
    "orphans |",
    "PASS" if len(orphan_order_items_orders) == 0 else "FAIL"
)

print(
    "Order Items → Products:",
    len(orphan_order_items_products),
    "orphans |",
    "PASS" if len(orphan_order_items_products) == 0 else "FAIL"
)

print(
    "Support Tickets → Customers:",
    len(orphan_tickets),
    "orphans |",
    "PASS" if len(orphan_tickets) == 0 else "FAIL"
)


# ------------------------------------------------------------
# 6. OVERALL FINAL VALIDATION
# ------------------------------------------------------------

overall_pass = (
    all(shape_checks)
    and all(key_missing_checks)
    and all(pk_unique_checks)
    and all(fk_checks)
)


print("\n\n6. OVERALL FINAL VALIDATION")
print("-" * 80)

if overall_pass:
    print("✓ All processed files passed validation.")
    print("✓ Expected shapes preserved.")
    print("✓ Key identifiers contain no missing values.")
    print("✓ Primary keys remain unique.")
    print("✓ Foreign-key relationships remain valid.")
    print("\nOVERALL RESULT: PASS")
else:
    print("⚠ One or more processed-file checks failed.")
    print("\nOVERALL RESULT: REVIEW REQUIRED")


print("\n" + "=" * 80)
print("FINAL PROCESSED-FILE VALIDATION COMPLETE")
print("=" * 80)

FINAL PROCESSED-FILE VALIDATION

1. FILE RELOAD STATUS
--------------------------------------------------------------------------------
Customers: (6090, 14)
Products: (220, 9)
Orders: (28252, 9)
Order Items: (73405, 5)
Support Tickets: (5200, 8)


2. SHAPE VALIDATION
--------------------------------------------------------------------------------
Customers: PASS | Expected (6090, 14) | Actual (6090, 14)
Products: PASS | Expected (220, 9) | Actual (220, 9)
Orders: PASS | Expected (28252, 9) | Actual (28252, 9)
Order Items: PASS | Expected (73405, 5) | Actual (73405, 5)
Support Tickets: PASS | Expected (5200, 8) | Actual (5200, 8)


3. KEY IDENTIFIER VALIDATION
--------------------------------------------------------------------------------
Customers customer_id: 0 missing | PASS
Products product_id: 0 missing | PASS
Orders order_id: 0 missing | PASS
Orders customer_id: 0 missing | PASS
Order Items order_item_id: 0 missing | PASS
Order Items order_id: 0 missing | PASS
Order Items produc